In [1]:
!pip install -q stim pymatching "qiskit>=1.3" qiskit-ibm-runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.9/625.9 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.6/412.6 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.7/120.7 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.3/113.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.2/234.2 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.0/1

In [2]:
import stim, pymatching, numpy as np, math, random, time, statistics, importlib
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
fp = importlib.import_module("qiskit_ibm_runtime.fake_provider")

# ---------- per-gate / per-coupler / per-qubit noise from a chip's calibration ----------
class ChipNoise:
    def __init__(self, backend):
        self.b, t = backend, backend.target
        self.dt = backend.dt
        self.qp = t.qubit_properties or [None] * t.num_qubits
        def err(name, qargs):
            if name not in t.operation_names:
                return None
            p = t[name].get(qargs)
            return p.error if p is not None else None
        self._err = err
        self.cz_name = "cz" if "cz" in t.operation_names else None
        if self.cz_name is None:
            raise RuntimeError(f"{backend.name} has no CZ gate — use a Heron chip (Marrakesh, Kingston, Fez, Torino, Aachen)")
    def t1(self, q):
        p = self.qp[q]
        return p.t1 if p is not None and p.t1 else 100e-6
    def t2(self, q):
        p = self.qp[q]
        return min(p.t2 if p is not None and p.t2 else 100e-6, 2 * self.t1(q))
    def ro(self, q):
        e = self._err("measure", (q,))
        return min(e, 0.45) if e is not None else 0.03
    def sx(self, q):
        e = self._err("sx", (q,))
        return min(e, 0.1) if e is not None else 3e-4
    def cz(self, a, b):
        e = self._err(self.cz_name, (a, b))
        if e is None:
            e = self._err(self.cz_name, (b, a))
        return min(e, 0.5) if e is not None else 5e-3
    def idle(self, q, dur):
        a = 1 - math.exp(-dur / self.t1(q))
        b = 1 - math.exp(-dur / self.t2(q))
        return a / 4, a / 4, max(b / 2 - a / 4, 0.0)

# ---------- step 1: Stim code circuit -> Qiskit circuit (+ detector/observable bookkeeping) ----------
def stim_to_qiskit(sc):
    flat = sc.flattened()
    qc = QuantumCircuit(sc.num_qubits, sc.num_measurements)
    m, dets, obs = 0, [], {}
    for inst in flat:
        name, tg = inst.name, inst.targets_copy()
        if name == "R":
            for t in tg: qc.reset(t.value)
        elif name == "RX":
            for t in tg: qc.reset(t.value); qc.h(t.value)
        elif name == "H":
            for t in tg: qc.h(t.value)
        elif name == "CX":
            for a, b in zip(tg[0::2], tg[1::2]): qc.cx(a.value, b.value)
        elif name in ("M", "MR", "MX"):
            for t in tg:
                if name == "MX": qc.h(t.value)
                qc.measure(t.value, m)
                m += 1
                if name == "MR": qc.reset(t.value)
        elif name == "DETECTOR":
            dets.append([m + t.value for t in tg])            # rec[-k] -> absolute measurement index
        elif name == "OBSERVABLE_INCLUDE":
            k = int(inst.gate_args_copy()[0])
            obs.setdefault(k, []).extend(m + t.value for t in tg)
        elif name in ("TICK", "QUBIT_COORDS", "SHIFT_COORDS"):
            pass
        else:
            raise ValueError(f"unexpected Stim instruction {name}")
    return qc, dets, obs

# ---------- step 3: compiled (ISA) circuit -> Stim with calibrated noise ----------
RZ = {0: None, 1: "S", 2: "Z", 3: "S_DAG"}
def isa_to_stim(isa, noise, dets, obs, noisy=True):
    out, pos, nmeas = stim.Circuit(), {}, 0
    for inst in isa.data:
        op, name = inst.operation, inst.operation.name
        qs = [isa.find_bit(q).index for q in inst.qubits]
        if name in ("barrier", "id"):
            continue
        if name == "delay":
            val = float(op.params[0])
            dur = val * noise.dt if getattr(op, "unit", "dt") == "dt" else val
            if noisy and dur > 0:
                out.append("PAULI_CHANNEL_1", qs, list(noise.idle(qs[0], dur)))
        elif name == "rz":
            ang = float(op.params[0]) / (math.pi / 2)
            k = round(ang)
            if abs(ang - k) > 1e-6:
                raise ValueError(f"non-Clifford rz angle {op.params[0]}")
            g = RZ[k % 4]
            if g:
                out.append(g, qs)
        elif name in ("sx", "sxdg", "x"):
            out.append({"sx": "SQRT_X", "sxdg": "SQRT_X_DAG", "x": "X"}[name], qs)
            if noisy: out.append("DEPOLARIZE1", qs, noise.sx(qs[0]))
        elif name == "cz":
            out.append("CZ", qs)
            if noisy: out.append("DEPOLARIZE2", qs, noise.cz(*qs))
        elif name == "measure":
            if noisy: out.append("X_ERROR", qs, noise.ro(qs[0]))
            out.append("M", qs)
            pos[isa.find_bit(inst.clbits[0]).index] = nmeas
            nmeas += 1
        elif name == "reset":
            out.append("R", qs)
            if noisy: out.append("X_ERROR", qs, noise.ro(qs[0]))
        else:
            raise ValueError(f"unsupported gate in compiled circuit: {name}")
    for det in dets:
        out.append("DETECTOR", [stim.target_rec(pos[c] - nmeas) for c in det])
    for k, lst in obs.items():
        out.append("OBSERVABLE_INCLUDE", [stim.target_rec(pos[c] - nmeas) for c in lst], k)
    return out

# ---------- the full pipeline for one layout ----------
def compile_code(backend, d, basis, seed, initial_layout=None):
    sc = stim.Circuit.generated(f"surface_code:rotated_memory_{basis.lower()}", distance=d, rounds=d)
    qc, dets, obs = stim_to_qiskit(sc)
    kw = dict(optimization_level=1, target=backend.target, seed_transpiler=seed, scheduling_method="alap")
    if initial_layout is not None:
        kw["initial_layout"] = initial_layout
    isa = generate_preset_pass_manager(**kw).run(qc)
    return isa, dets, obs

def dem_of(circ):
    return circ.detector_error_model(decompose_errors=True, ignore_decomposition_failures=True)

def logical_error(circ, shots, seed=1):
    m = pymatching.Matching.from_detector_error_model(dem_of(circ))
    det, ob = circ.compile_detector_sampler(seed=seed).sample(shots, separate_observables=True)
    return float(np.mean(np.any(m.decode_batch(det) != ob, axis=1)))

def per_round(pL, rounds):
    return 1 - (1 - min(pL, 0.4999)) ** (1 / rounds)

def validate(isa, noise, dets, obs, d):
    clean = isa_to_stim(isa, noise, dets, obs, noisy=False)
    det, ob = clean.compile_detector_sampler(seed=3).sample(500, separate_observables=True)
    assert not det.any(), "noiseless circuit fires detectors — conversion is WRONG"
    assert not ob.any(), "noiseless circuit flips the logical observable — conversion is WRONG"
    noisy = isa_to_stim(isa, noise, dets, obs, noisy=True)
    dist = len(dem_of(noisy).shortest_graphlike_error(ignore_ungraphlike_errors=True))
    return noisy, dist

def used_qubits(isa):
    return sorted({isa.find_bit(q).index for inst in isa.data for q in inst.qubits
                   if inst.operation.name not in ("barrier", "delay")})

print("✅ Chip-realistic pipeline ready")

✅ Chip-realistic pipeline ready


In [6]:
# ---------- FIX: idle noise only while a qubit holds quantum information; Stim option for idle channel ----------
def isa_to_stim(isa, noise, dets, obs, noisy=True):
    out, pos, nmeas = stim.Circuit(), {}, 0
    active = set()                       # qubits currently holding quantum information
    for inst in isa.data:
        op, name = inst.operation, inst.operation.name
        qs = [isa.find_bit(q).index for q in inst.qubits]
        if name in ("barrier", "id"):
            continue
        if name == "delay":
            if noisy and qs[0] in active:
                val = float(op.params[0])
                dur = val * noise.dt if getattr(op, "unit", "dt") == "dt" else val
                if dur > 0:
                    out.append("PAULI_CHANNEL_1", qs, list(noise.idle(qs[0], dur)))
            continue
        if name == "rz":
            ang = float(op.params[0]) / (math.pi / 2)
            k = round(ang)
            if abs(ang - k) > 1e-6:
                raise ValueError(f"non-Clifford rz angle {op.params[0]}")
            g = RZ[k % 4]
            if g:
                out.append(g, qs)
            active.update(qs)
        elif name in ("sx", "sxdg", "x"):
            out.append({"sx": "SQRT_X", "sxdg": "SQRT_X_DAG", "x": "X"}[name], qs)
            if noisy: out.append("DEPOLARIZE1", qs, noise.sx(qs[0]))
            active.update(qs)
        elif name == "cz":
            out.append("CZ", qs)
            if noisy: out.append("DEPOLARIZE2", qs, noise.cz(*qs))
            active.update(qs)
        elif name == "measure":
            if noisy: out.append("X_ERROR", qs, noise.ro(qs[0]))
            out.append("M", qs)
            pos[isa.find_bit(inst.clbits[0]).index] = nmeas
            nmeas += 1
            active.difference_update(qs)       # state collapsed: idling now is harmless until the next gate
        elif name == "reset":
            out.append("R", qs)
            if noisy: out.append("X_ERROR", qs, noise.ro(qs[0]))
            active.difference_update(qs)       # fresh |0> cannot decay
        else:
            raise ValueError(f"unsupported gate in compiled circuit: {name}")
    for det in dets:
        out.append("DETECTOR", [stim.target_rec(pos[c] - nmeas) for c in det])
    for k, lst in obs.items():
        out.append("OBSERVABLE_INCLUDE", [stim.target_rec(pos[c] - nmeas) for c in lst], k)
    return out

def dem_of(circ):
    return circ.detector_error_model(decompose_errors=True, ignore_decomposition_failures=True,
                                     approximate_disjoint_errors=True)
print("✅ Fixed: idle noise only on active qubits; Stim idle-channel option enabled")

✅ Fixed: idle noise only on active qubits; Stim idle-channel option enabled


In [7]:
# ---------- PHASE B: layout tournament on the realistic chip model ----------
# candidates = many different routings (Qiskit's layout+routing with different seeds)
# selectors:   Qiskit default (seed 1) | lowest calibration error budget | best simulated logical error
N_CAND = {3: 24, 5: 10}
PILOT_SHOTS = {3: 20000, 5: 10000}
FINAL_SHOTS = {3: 200000, 5: 100000}

def error_budget(circ):
    # total probability mass of all noise processes — a quick, model-only score
    tot = 0.0
    for inst in circ.flattened():
        if inst.name in ("DEPOLARIZE1", "DEPOLARIZE2", "X_ERROR"):
            tot += inst.gate_args_copy()[0] * (len(inst.targets_copy()) // (2 if inst.name == "DEPOLARIZE2" else 1))
        elif inst.name == "PAULI_CHANNEL_1":
            tot += sum(inst.gate_args_copy()) * len(inst.targets_copy())
    return tot

TB = []
for chip in CHIPS_HH:
    backend = getattr(fp, chip)()
    noise = ChipNoise(backend)
    for basis in ("Z", "X"):
        for d in (3, 5):
            t0 = time.time()
            cands = []
            for s in range(1, N_CAND[d] + 1):
                isa, dets, obs = compile_code(backend, d, basis, seed=s)
                try:
                    noisy, dist = validate(isa, noise, dets, obs, d)
                except AssertionError as e:
                    print("  candidate rejected:", e)
                    continue
                cands.append({"seed": s, "circ": noisy, "dist": dist, "cz": isa.count_ops().get("cz", 0),
                              "budget": error_budget(noisy)})
            default = next(c for c in cands if c["seed"] == 1)
            by_budget = min(cands, key=lambda c: (-c["dist"], c["budget"]))
            for c in cands:                                   # simulation-in-the-loop pilot
                c["pilot"] = logical_error(c["circ"], PILOT_SHOTS[d], seed=11)
            by_sim = min(cands, key=lambda c: c["pilot"])
            fin = {k: per_round(logical_error(c["circ"], FINAL_SHOTS[d], seed=999), d)   # fresh shots
                   for k, c in (("default", default), ("budget", by_budget), ("sim", by_sim))}
            dists = sorted({c["dist"] for c in cands})
            TB.append((chip, basis, d, fin, default, by_budget, by_sim))
            print(f"{chip:14s} {basis} d={d}: {len(cands)} candidates, distances seen {dists} | "
                  f"default {fin['default']*100:5.2f}% (dist {default['dist']}, {default['cz']} CZ) | "
                  f"budget-pick {fin['budget']*100:5.2f}% (dist {by_budget['dist']}, {by_budget['cz']} CZ) | "
                  f"sim-pick {fin['sim']*100:5.2f}% (dist {by_sim['dist']}, {by_sim['cz']} CZ)  per round "
                  f"({time.time()-t0:.0f}s)")

print("\nGain over Qiskit's default layout (default error / picked error, per round, fresh shots):")
for chip, basis, d, fin, *_ in TB:
    print(f"  {chip:14s} {basis} d={d}: budget-pick ×{fin['default']/max(fin['budget'],1e-9):.2f} | "
          f"sim-pick ×{fin['default']/max(fin['sim'],1e-9):.2f}")

FakeMarrakesh  Z d=3: 24 candidates, distances seen [2] | default  7.28% (dist 2, 273 CZ) | budget-pick  4.65% (dist 2, 279 CZ) | sim-pick  4.54% (dist 2, 297 CZ)  per round (7s)
FakeMarrakesh  Z d=5: 10 candidates, distances seen [3] | default 12.70% (dist 3, 2197 CZ) | budget-pick 12.81% (dist 3, 2023 CZ) | sim-pick 12.69% (dist 3, 1990 CZ)  per round (38s)
FakeMarrakesh  X d=3: 24 candidates, distances seen [2] | default 10.08% (dist 2, 273 CZ) | budget-pick  8.07% (dist 2, 279 CZ) | sim-pick  5.59% (dist 2, 270 CZ)  per round (8s)
FakeMarrakesh  X d=5: 10 candidates, distances seen [3] | default 12.88% (dist 3, 2197 CZ) | budget-pick 12.71% (dist 3, 2023 CZ) | sim-pick 12.94% (dist 3, 2227 CZ)  per round (41s)
FakeKingston   Z d=3: 24 candidates, distances seen [2] | default  6.41% (dist 2, 273 CZ) | budget-pick  3.87% (dist 2, 297 CZ) | sim-pick  3.70% (dist 2, 279 CZ)  per round (7s)
FakeKingston   Z d=5: 10 candidates, distances seen [3] | default 10.82% (dist 3, 2197 CZ) | budg

In [8]:
# ---------- PHASE B: layout tournament on the realistic chip model ----------
# candidates = many different routings (Qiskit's layout+routing with different seeds)
# selectors:   Qiskit default (seed 1) | lowest calibration error budget | best simulated logical error
N_CAND = {3: 24, 5: 10}
PILOT_SHOTS = {3: 20000, 5: 10000}
FINAL_SHOTS = {3: 200000, 5: 100000}

def error_budget(circ):
    # total probability mass of all noise processes — a quick, model-only score
    tot = 0.0
    for inst in circ.flattened():
        if inst.name in ("DEPOLARIZE1", "DEPOLARIZE2", "X_ERROR"):
            tot += inst.gate_args_copy()[0] * (len(inst.targets_copy()) // (2 if inst.name == "DEPOLARIZE2" else 1))
        elif inst.name == "PAULI_CHANNEL_1":
            tot += sum(inst.gate_args_copy()) * len(inst.targets_copy())
    return tot

TB = []
for chip in CHIPS_HH:
    backend = getattr(fp, chip)()
    noise = ChipNoise(backend)
    for basis in ("Z", "X"):
        for d in (3, 5):
            t0 = time.time()
            cands = []
            for s in range(1, N_CAND[d] + 1):
                isa, dets, obs = compile_code(backend, d, basis, seed=s)
                try:
                    noisy, dist = validate(isa, noise, dets, obs, d)
                except AssertionError as e:
                    print("  candidate rejected:", e)
                    continue
                cands.append({"seed": s, "circ": noisy, "dist": dist, "cz": isa.count_ops().get("cz", 0),
                              "budget": error_budget(noisy)})
            default = next(c for c in cands if c["seed"] == 1)
            by_budget = min(cands, key=lambda c: (-c["dist"], c["budget"]))
            for c in cands:                                   # simulation-in-the-loop pilot
                c["pilot"] = logical_error(c["circ"], PILOT_SHOTS[d], seed=11)
            by_sim = min(cands, key=lambda c: c["pilot"])
            fin = {k: per_round(logical_error(c["circ"], FINAL_SHOTS[d], seed=999), d)   # fresh shots
                   for k, c in (("default", default), ("budget", by_budget), ("sim", by_sim))}
            dists = sorted({c["dist"] for c in cands})
            TB.append((chip, basis, d, fin, default, by_budget, by_sim))
            print(f"{chip:14s} {basis} d={d}: {len(cands)} candidates, distances seen {dists} | "
                  f"default {fin['default']*100:5.2f}% (dist {default['dist']}, {default['cz']} CZ) | "
                  f"budget-pick {fin['budget']*100:5.2f}% (dist {by_budget['dist']}, {by_budget['cz']} CZ) | "
                  f"sim-pick {fin['sim']*100:5.2f}% (dist {by_sim['dist']}, {by_sim['cz']} CZ)  per round "
                  f"({time.time()-t0:.0f}s)")

print("\nGain over Qiskit's default layout (default error / picked error, per round, fresh shots):")
for chip, basis, d, fin, *_ in TB:
    print(f"  {chip:14s} {basis} d={d}: budget-pick ×{fin['default']/max(fin['budget'],1e-9):.2f} | "
          f"sim-pick ×{fin['default']/max(fin['sim'],1e-9):.2f}")

FakeMarrakesh  Z d=3: 24 candidates, distances seen [2] | default  7.28% (dist 2, 273 CZ) | budget-pick  4.65% (dist 2, 279 CZ) | sim-pick  4.54% (dist 2, 297 CZ)  per round (8s)
FakeMarrakesh  Z d=5: 10 candidates, distances seen [3] | default 12.70% (dist 3, 2197 CZ) | budget-pick 12.81% (dist 3, 2023 CZ) | sim-pick 12.69% (dist 3, 1990 CZ)  per round (38s)
FakeMarrakesh  X d=3: 24 candidates, distances seen [2] | default 10.08% (dist 2, 273 CZ) | budget-pick  8.07% (dist 2, 279 CZ) | sim-pick  5.59% (dist 2, 270 CZ)  per round (8s)
FakeMarrakesh  X d=5: 10 candidates, distances seen [3] | default 12.88% (dist 3, 2197 CZ) | budget-pick 12.71% (dist 3, 2023 CZ) | sim-pick 12.94% (dist 3, 2227 CZ)  per round (41s)
FakeKingston   Z d=3: 24 candidates, distances seen [2] | default  6.41% (dist 2, 273 CZ) | budget-pick  3.87% (dist 2, 297 CZ) | sim-pick  3.70% (dist 2, 279 CZ)  per round (7s)
FakeKingston   Z d=5: 10 candidates, distances seen [3] | default 10.82% (dist 3, 2197 CZ) | budg

In [10]:
# ---------- NATIVE HEAVY-HEX CODE (IBM subsystem code, rebuilt from qiskit-qec's construction) ----------
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister

def hhc_structure(d):
    idx = lambda r, c: r * d + c
    xg = [[idx(i, j), idx(i + 1, j)] for i in range(d - 1) for j in range(d)]       # vertical X gauges
    zg, zcol = [], []
    for i in range(d + 1):
        for j in range(d - 1):
            if i % 2 == j % 2:
                if i == 0:   zg.append([idx(0, j), idx(0, j + 1)])
                elif i == d: zg.append([idx(d - 1, j), idx(d - 1, j + 1)])
                else:        zg.append([idx(i - 1, j), idx(i - 1, j + 1), idx(i, j), idx(i, j + 1)])
                zcol.append(j)
    xstab = []
    for i in range(d - 1):
        for j in range(d):
            g = i * d + j
            if i % 2 == j % 2: xstab.append([g] if j == d - 1 else [g, g + 1])
            elif j == 0:       xstab.append([g])
    n = d * d
    xanc = [n + k for k in range(len(xg))]
    total, zanc = len(xg), []
    for z in zg:
        if len(z) == 2:
            zanc.append([n + total, n + total + 1, n + total + 2]); total += 3
        else:
            left  = next(n + k for k, x in enumerate(xg) if z[0] in x and z[2] in x)
            right = next(n + k for k, x in enumerate(xg) if z[1] in x and z[3] in x)
            zanc.append([left, n + total, right]); total += 1
    return {"d": d, "n": n, "xg": xg, "zg": zg, "zcol": zcol, "xstab": xstab,
            "xanc": xanc, "zanc": zanc, "nq": n + total}

def build_hhc_memory(d, rounds, basis):
    S = hhc_structure(d)
    n, xg, zg = S["n"], S["xg"], S["zg"]
    schedule = ("xz" if basis == "Z" else "zx") * rounds
    nx, nz = len(xg), len(zg)
    ncl = sum(nx if r == "x" else 3 * nz for r in schedule) + n
    qc = QuantumCircuit(QuantumRegister(S["nq"], "q"), ClassicalRegister(ncl, "c"))
    cb = iter(range(ncl))
    for i in range(n): qc.reset(i)
    if basis == "X":
        for i in range(n): qc.h(i)
    log = []
    for r in schedule:
        if r == "x":
            for a in S["xanc"]: qc.reset(a); qc.h(a)
            for step in range(2):
                for k, g in enumerate(xg): qc.cx(S["xanc"][k], g[step])
            bits = []
            for a in S["xanc"]:
                qc.h(a); b = next(cb); qc.measure(a, b); bits.append(b)
            log.append(("x", bits))
        else:
            gb, lf, rf = [None] * nz, [None] * nz, [None] * nz
            for step in range(7):
                for j, z in enumerate(zg):
                    l, m, rr = S["zanc"][j]
                    bnd, top = len(z) == 2, z[0] < d
                    if step == 0:   qc.reset(m); qc.reset(rr); qc.h(rr)
                    elif step == 1: qc.cx(rr, m); qc.reset(l); qc.h(l)
                    elif step == 2:
                        qc.cx(l, m)
                        if not (bnd and top): qc.cx(z[1], rr)
                    elif step == 3:
                        if bnd and top: qc.cx(z[0], l); qc.cx(z[1], rr)
                        elif not bnd:   qc.cx(z[2], l); qc.cx(z[3], rr)
                    elif step == 4:
                        qc.cx(rr, m)
                        if not (bnd and top): qc.cx(z[0], l)
                    elif step == 5:
                        qc.cx(l, m); qc.h(rr); rf[j] = next(cb); qc.measure(rr, rf[j])
                    elif step == 6:
                        qc.h(l)
                        gb[j] = next(cb); qc.measure(m, gb[j])
                        lf[j] = next(cb); qc.measure(l, lf[j])
            log.append(("z", gb, lf, rf))
    if basis == "X":
        for i in range(n): qc.h(i)
    data_bits = []
    for i in range(n):
        b = next(cb); qc.measure(i, b); data_bits.append(b)

    dets, flag_dets = [], []
    zr = [e for e in log if e[0] == "z"]; xr = [e for e in log if e[0] == "x"]
    zs = lambda e, j: [e[1][g] for g in range(nz) if S["zcol"][g] == j]
    xs = lambda e, s: [e[1][g] for g in S["xstab"][s]]
    for j in range(d - 1):
        for t, e in enumerate(zr):
            if t == 0:
                if basis == "Z": dets.append(zs(e, j))
            else: dets.append(zs(e, j) + zs(zr[t - 1], j))
        if basis == "Z":
            dets.append([data_bits[r * d + j] for r in range(d)] +
                        [data_bits[r * d + j + 1] for r in range(d)] + zs(zr[-1], j))
    for s, gs in enumerate(S["xstab"]):
        for t, e in enumerate(xr):
            if t == 0:
                if basis == "X": dets.append(xs(e, s))
            else: dets.append(xs(e, s) + xs(xr[t - 1], s))
        if basis == "X":
            sup = sorted({q_ for g in gs for q_ in xg[g]})
            dets.append([data_bits[q_] for q_ in sup] + xs(xr[-1], s))
    for e in zr: flag_dets += [[b] for b in e[2] + e[3]]
    return qc, dets, flag_dets, {0: list(data_bits)}, S

def check_dets(isa, noise, dets, obs):
    clean = isa_to_stim(isa, noise, dets, obs, noisy=False)
    det, ob = clean.compile_detector_sampler(seed=3).sample(400, separate_observables=True)
    return det.any(axis=0), ob.any()

print("✅ Heavy-hex code builder ready")

✅ Heavy-hex code builder ready


In [11]:
HH_CHIPS, HH_ROUNDS, HH_STORE = ["FakeKingston", "FakeMarrakesh"], 3, {}
for chip in HH_CHIPS:
    backend = getattr(fp, chip)(); noise = ChipNoise(backend)
    for basis in ("Z", "X"):
        for d in (3, 5):
            t0 = time.time()
            qc, dets, flag_dets, obs, S = build_hhc_memory(d, HH_ROUNDS, basis)
            n_cx = qc.count_ops().get("cx", 0)
            isa = generate_preset_pass_manager(optimization_level=1, target=backend.target, seed_transpiler=1,
                                               scheduling_method="alap").run(qc)
            n_cz = isa.count_ops().get("cz", 0)
            fired, ob_fired = check_dets(isa, noise, dets, obs)
            assert not fired.any(), f"{chip} {basis} d={d}: stabilizer detectors fire without noise"
            assert not ob_fired,    f"{chip} {basis} d={d}: logical flips without noise"
            ffired, _ = check_dets(isa, noise, flag_dets, obs)
            good_flags = [f for f, bad in zip(flag_dets, ffired) if not bad]
            noisy = isa_to_stim(isa, noise, dets + good_flags, obs, noisy=True)
            dist = len(dem_of(noisy).shortest_graphlike_error(ignore_ungraphlike_errors=True))
            pL = logical_error(noisy, 100000 if d == 3 else 40000)
            HH_STORE[(chip, basis, d)] = (qc, dets, good_flags, obs, S, isa)
            swaps = "NO SWAPs ✓" if n_cz == n_cx else f"+{n_cz - n_cx} extra CZ (SWAPs?)"
            print(f"{chip:14s} {basis} d={d}: noiseless OK | {S['nq']} qubits | {swaps} | "
                  f"flags {len(good_flags)}/{len(flag_dets)} | distance {dist} (ideal {d}) | "
                  f"logical {pL*100:5.2f}% → {per_round(pL, HH_ROUNDS)*100:5.2f}% per round  ({time.time()-t0:.0f}s)")
print("\nIBM 2022 hardware reference (same code, d=3): ~4–6%/round for Z basis, ~9–11%/round for X basis.")

FakeKingston   Z d=3: noiseless OK | 23 qubits | NO SWAPs ✓ | flags 24/24 | distance 3 (ideal 3) | logical 18.37% →  6.54% per round  (0s)
FakeKingston   Z d=5: noiseless OK | 65 qubits | NO SWAPs ✓ | flags 72/72 | distance 5 (ideal 5) | logical 27.77% → 10.28% per round  (1s)
FakeKingston   X d=3: noiseless OK | 23 qubits | NO SWAPs ✓ | flags 24/24 | distance 3 (ideal 3) | logical 17.24% →  6.11% per round  (0s)
FakeKingston   X d=5: noiseless OK | 65 qubits | NO SWAPs ✓ | flags 72/72 | distance 5 (ideal 5) | logical 35.21% → 13.47% per round  (1s)
FakeMarrakesh  Z d=3: noiseless OK | 23 qubits | NO SWAPs ✓ | flags 24/24 | distance 3 (ideal 3) | logical 22.67% →  8.21% per round  (1s)
FakeMarrakesh  Z d=5: noiseless OK | 65 qubits | NO SWAPs ✓ | flags 72/72 | distance 5 (ideal 5) | logical 33.24% → 12.60% per round  (1s)
FakeMarrakesh  X d=3: noiseless OK | 23 qubits | NO SWAPs ✓ | flags 24/24 | distance 3 (ideal 3) | logical 28.05% → 10.39% per round  (0s)
FakeMarrakesh  X d=5: noise

In [12]:
class ScaledNoise(ChipNoise):
    def __init__(self, backend, f):
        super().__init__(backend); self.f = f
    def t2(self, q):          # "dd": perfect dynamical decoupling → T2 reaches its 2·T1 limit
        return 2 * self.t1(q) if self.f.get("dd") else super().t2(q)
    def cz(self, a, b): return super().cz(a, b) * self.f.get("cz", 1)
    def ro(self, q):    return super().ro(q) * self.f.get("ro", 1)
    def sx(self, q):    return super().sx(q) * self.f.get("sx", 1)
    def idle(self, q, dur): return tuple(x * self.f.get("idle", 1) for x in super().idle(q, dur))

SCEN = [("today's chip",            {}),
        ("perfect DD (T2 → 2·T1)",   {"dd": 1}),
        ("no idle noise",            {"idle": 0}),
        ("no CZ noise",              {"cz": 0}),
        ("no readout/reset noise",   {"ro": 0}),
        ("no 1-qubit gate noise",    {"sx": 0}),
        ("CZ error ×0.5",            {"cz": 0.5}),
        ("readout error ×0.5",       {"ro": 0.5}),
        ("idle error ×0.5",          {"idle": 0.5}),
        ("everything ×0.5",          {"cz": 0.5, "ro": 0.5, "idle": 0.5, "sx": 0.5})]
BUDGET = {}
for chip in HH_CHIPS:
    backend = getattr(fp, chip)()
    for basis in ("Z", "X"):
        print(f"\n{chip} — {basis} basis      d=3 /round   d=5 /round   Λ=d3/d5 (>1 = bigger code helps)")
        for name, f in SCEN:
            sn = ScaledNoise(backend, f); row = {}
            for d in (3, 5):
                qc, dets, good_flags, obs, S, isa = HH_STORE[(chip, basis, d)]
                circ = isa_to_stim(isa, sn, dets + good_flags, obs, noisy=True)
                row[d] = per_round(logical_error(circ, 40000, seed=5), HH_ROUNDS)
            lam = row[3] / max(row[5], 1e-9)
            BUDGET[(chip, basis, name)] = (row[3], row[5], lam)
            print(f"  {name:26s} {row[3]*100:8.2f}%   {row[5]*100:8.2f}%    {lam:5.2f} {'✅' if lam > 1 else ''}")


FakeKingston — Z basis      d=3 /round   d=5 /round   Λ=d3/d5 (>1 = bigger code helps)
  today's chip                   6.62%      10.43%     0.63 
  perfect DD (T2 → 2·T1)         2.56%       4.29%     0.60 
  no idle noise                  0.79%       1.49%     0.53 
  no CZ noise                    5.79%       9.14%     0.63 
  no readout/reset noise         5.53%       7.35%     0.75 
  no 1-qubit gate noise          6.59%      10.44%     0.63 
  CZ error ×0.5                  6.08%       9.73%     0.62 
  readout error ×0.5             5.58%       8.85%     0.63 
  idle error ×0.5                3.46%       5.66%     0.61 
  everything ×0.5                2.27%       3.19%     0.71 

FakeKingston — X basis      d=3 /round   d=5 /round   Λ=d3/d5 (>1 = bigger code helps)
  today's chip                   6.26%      13.31%     0.47 
  perfect DD (T2 → 2·T1)         2.64%      11.10%     0.24 
  no idle noise                  1.15%       8.71%     0.13 
  no CZ noise                  

In [13]:
import rustworkx as rx
N_SHUFFLE, PER_SHUFFLE, TOP_K, N_RAND = 30, 40, 8, 4
PILOT = {3: 20000, 5: 10000}; FINAL = {3: 200000, 5: 100000}

def code_graph(qc):
    uses = {}
    for inst in qc.data:
        if inst.operation.name == "cx":
            a, b = sorted(qc.find_bit(q).index for q in inst.qubits)
            uses[(a, b)] = uses.get((a, b), 0) + 1
    g = rx.PyGraph(); g.add_nodes_from(range(qc.num_qubits)); g.add_edges_from_no_data(list(uses))
    return g, uses

def role_counts(qc):
    meas, rst = np.zeros(qc.num_qubits), np.zeros(qc.num_qubits)
    for inst in qc.data:
        i = qc.find_bit(inst.qubits[0]).index
        if inst.operation.name == "measure": meas[i] += 1
        elif inst.operation.name == "reset": rst[i] += 1
    return meas, rst

def candidate_layouts(backend, cg, rng):
    nq = backend.num_qubits
    edges = {tuple(sorted(e)) for e in backend.coupling_map.get_edges()}
    seen, out = set(), []
    for _ in range(N_SHUFFLE):
        perm = rng.permutation(nq); inv = {int(p): i for i, p in enumerate(perm)}
        chipg = rx.PyGraph(); chipg.add_nodes_from(range(nq))
        chipg.add_edges_from_no_data([(inv[a], inv[b]) for a, b in edges])
        it = rx.vf2_mapping(chipg, cg, subgraph=True, induced=False, id_order=False, call_limit=5_000_000)
        for k, m in enumerate(it):
            if k >= PER_SHUFFLE: break
            lay = [0] * cg.num_nodes()
            for cn, vn in m.items(): lay[vn] = int(perm[cn])
            t = tuple(lay)
            if t not in seen: seen.add(t); out.append(list(t))
    return out

def model_score(lay, uses, meas, rst, noise, n_data, T):
    s = sum(k * noise.cz(lay[a], lay[b]) for (a, b), k in uses.items())
    for i, p in enumerate(lay):
        s += (meas[i] + rst[i]) * noise.ro(p)
        s += sum(noise.idle(p, T)) * (1.0 if i < n_data else 0.5)   # data qubits idle the longest
    return s

def compile_with(backend, qc, lay):
    return generate_preset_pass_manager(optimization_level=1, target=backend.target, initial_layout=lay,
                                        seed_transpiler=1, scheduling_method="alap").run(qc)

TOURN = {}
for chip in HH_CHIPS:
    backend = getattr(fp, chip)(); noise = ChipNoise(backend)
    for basis in ("Z", "X"):
        for d in (3, 5):
            t0 = time.time(); rng = np.random.default_rng(7)
            qc, dets, good_flags, obs, S, isa_def = HH_STORE[(chip, basis, d)]
            cg, uses = code_graph(qc); meas, rst = role_counts(qc)
            n_cx = qc.count_ops().get("cx", 0)
            T = 2 * HH_ROUNDS * 5e-6
            pool = candidate_layouts(backend, cg, rng)
            scored = sorted(pool, key=lambda L: model_score(L, uses, meas, rst, noise, S["n"], T))
            lay_def = list(isa_def.layout.initial_index_layout(filter_ancillas=True))
            picks = [("default", lay_def)] + [(f"model#{i+1}", L) for i, L in enumerate(scored[:TOP_K])]
            picks += [(f"random#{i+1}", pool[j]) for i, j in enumerate(rng.choice(len(pool), N_RAND, replace=False))]
            cands = []
            for name, L in picks:
                isa = isa_def if name == "default" else compile_with(backend, qc, L)
                if isa.count_ops().get("cz", 0) != n_cx: continue
                fired, obf = check_dets(isa, noise, dets, obs)
                if fired.any() or obf: continue
                circ = isa_to_stim(isa, noise, dets + good_flags, obs, noisy=True)
                cands.append({"name": name, "lay": L, "circ": circ,
                              "pilot": logical_error(circ, PILOT[d], seed=11)})
            winner = min([c for c in cands if not c["name"].startswith("random")], key=lambda c: c["pilot"])
            final = {}
            for tag, c in [("default", cands[0]), ("model#1", next(c for c in cands if c["name"] == "model#1")),
                           ("tournament", winner)] + [(c["name"], c) for c in cands if c["name"].startswith("random")]:
                final[tag] = per_round(logical_error(c["circ"], FINAL[d], seed=999), HH_ROUNDS)
            rnd = [v for k, v in final.items() if k.startswith("random")]
            TOURN[(chip, basis, d)] = {"final": final, "winner": winner["name"], "layout": winner["lay"], "pool": len(pool)}
            print(f"{chip:14s} {basis} d={d}: pool {len(pool)} | random median {np.median(rnd)*100:5.2f}% | "
                  f"Qiskit {final['default']*100:5.2f}% | model#1 {final['model#1']*100:5.2f}% | "
                  f"tournament ({winner['name']}) {final['tournament']*100:5.2f}% /round "
                  f"→ ×{final['default']/max(final['tournament'],1e-9):.2f} vs Qiskit  ({time.time()-t0:.0f}s)")

print("\nΛ = d3/d5 per selector (>1 means the bigger code is better):")
for chip in HH_CHIPS:
    for basis in ("Z", "X"):
        a, b = TOURN[(chip, basis, 3)]["final"], TOURN[(chip, basis, 5)]["final"]
        print(f"  {chip:14s} {basis}: Qiskit Λ={a['default']/b['default']:.2f} | tournament Λ={a['tournament']/b['tournament']:.2f}")

FakeKingston   Z d=3: pool 148 | random median  5.84% | Qiskit  6.56% | model#1  5.11% | tournament (model#8)  5.05% /round → ×1.30 vs Qiskit  (6s)
FakeKingston   Z d=5: pool 24 | random median 10.04% | Qiskit 10.48% | model#1  9.60% | tournament (model#4)  9.51% /round → ×1.10 vs Qiskit  (15s)
FakeKingston   X d=3: pool 148 | random median  7.67% | Qiskit  6.09% | model#1  4.67% | tournament (model#2)  4.66% /round → ×1.31 vs Qiskit  (7s)
FakeKingston   X d=5: pool 24 | random median 13.85% | Qiskit 13.48% | model#1 11.37% | tournament (model#1) 11.37% /round → ×1.19 vs Qiskit  (19s)
FakeMarrakesh  Z d=3: pool 148 | random median 12.96% | Qiskit  8.26% | model#1  7.00% | tournament (model#3)  6.63% /round → ×1.24 vs Qiskit  (9s)
FakeMarrakesh  Z d=5: pool 24 | random median 14.72% | Qiskit 12.60% | model#1 12.75% | tournament (default) 12.60% /round → ×1.00 vs Qiskit  (20s)
FakeMarrakesh  X d=3: pool 148 | random median 14.85% | Qiskit 10.39% | model#1  6.44% | tournament (model#1)  6

In [14]:
import json, os
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import ALAPScheduleAnalysis, PadDelay, PadDynamicalDecoupling
from qiskit.circuit.library import XGate

from google.colab import userdata  # IBM key comes from Colab Secrets, never from the notebook
API_KEY = userdata.get("IBM_QUANTUM_KEY")
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=API_KEY)
choices = [service.backend(n) for n in ("ibm_kingston", "ibm_marrakesh")]
for b in choices: print(f"{b.name}: {b.status().pending_jobs} jobs in queue")
hw = min(choices, key=lambda b: b.status().pending_jobs)
print("→ using", hw.name)
HW_D, HW_SHOTS = 3, 10000
noise = ChipNoise(hw); dd_bound_noise = ScaledNoise(hw, {"dd": 1})

def route(qc, lay=None):
    return generate_preset_pass_manager(optimization_level=1, target=hw.target, initial_layout=lay,
                                        seed_transpiler=1).run(qc)
def finish(routed, dd):
    pad = PadDynamicalDecoupling(target=hw.target, dd_sequence=[XGate(), XGate()]) if dd else PadDelay(target=hw.target)
    return PassManager([ALAPScheduleAnalysis(target=hw.target), pad]).run(routed)

HW, pubs = {"backend": hw.name, "shots": HW_SHOTS, "rounds": HH_ROUNDS, "arms": []}, []
for basis in ("Z", "X"):
    t0 = time.time()
    qc, dets, flag_dets, obs, S = build_hhc_memory(HW_D, HH_ROUNDS, basis)
    n_cx = qc.count_ops()["cx"]
    r_def = route(qc)
    lay_def = list(r_def.layout.initial_index_layout(filter_ancillas=True))
    base = finish(r_def, False)
    fired, obf = check_dets(base, noise, dets, obs)
    assert not fired.any() and not obf, "noiseless check failed"
    ff, _ = check_dets(base, noise, flag_dets, obs)
    alld = dets + [f for f, bad in zip(flag_dets, ff) if not bad]
    cg, uses = code_graph(qc); meas, rst = role_counts(qc)
    pool = candidate_layouts(hw, cg, np.random.default_rng(7))
    scored = sorted(pool, key=lambda L: model_score(L, uses, meas, rst, noise, S["n"], 2 * HH_ROUNDS * 5e-6))
    best = None
    for L in scored[:TOP_K]:
        if L == lay_def: continue
        try: r = route(qc, L)
        except Exception: continue
        if r.count_ops().get("cz", 0) != n_cx: continue
        p = logical_error(isa_to_stim(finish(r, False), noise, alld, obs), 20000, seed=11)
        if best is None or p < best[0]: best = (p, L, r)
    print(f"\n{basis} basis: {len(pool)} zero-SWAP placements on live {hw.name} ({time.time()-t0:.0f}s)")
    for lname, routed, L in (("qiskit", r_def, lay_def), ("tournament", best[2], best[1])):
        for dd in (False, True):
            isa = finish(routed, dd)
            fired, obf = check_dets(isa, noise, dets, obs)
            assert not fired.any() and not obf, f"noiseless check failed for {lname} dd={dd}"
            circ = isa_to_stim(isa, noise, alld, obs, noisy=True)
            pred = per_round(logical_error(circ, 100000, seed=999), HH_ROUNDS)
            bound = per_round(logical_error(isa_to_stim(isa, dd_bound_noise, alld, obs, noisy=True), 100000, seed=999), HH_ROUNDS)
            sim_det_rate = float(circ.compile_detector_sampler(seed=4).sample(20000).mean())
            HW["arms"].append({"basis": basis, "layout_name": lname, "dd": dd, "layout": [int(x) for x in L],
                               "dets": alld, "obs": obs[0], "pred": pred, "dd_bound": bound,
                               "sim_det_rate": sim_det_rate, "stim": str(circ)})
            pubs.append(isa)
            print(f"  {lname:10s} DD {'on ' if dd else 'off'}: model predicts {pred*100:5.2f}%/round "
                  f"(if DD were perfect: {bound*100:5.2f}%) | {isa.count_ops().get('x',0)} DD pulses")

print("\nPRE-REGISTERED (written before the job runs):")
print("  P1: tournament < Qiskit placement in ≥3 of 4 comparisons (2 bases × DD off/on)")
print("  P2: DD on < DD off in ≥3 of 4 comparisons (2 bases × 2 placements)")
print("  P3: best hardware arm, Z basis, at or below IBM 2022's 4–6% per round")
job = SamplerV2(mode=hw).run(pubs, shots=HW_SHOTS)
HW["job_id"] = job.job_id()
fname = f"hh_hw_{HW['job_id']}.json"
json.dump(HW, open(fname, "w"))
if os.path.isdir("/content/drive/MyDrive"):
    os.makedirs("/content/drive/MyDrive/qmem_hw", exist_ok=True)
    json.dump(HW, open(f"/content/drive/MyDrive/qmem_hw/{fname}", "w"))
print(f"\n🚀 submitted job {HW['job_id']} on {hw.name} — saved {fname}")

qiskit_runtime_service._discover_account:WARNING:2026-10-14 16:14:32,321: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-10-14 16:14:35,015: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-10-14 16:14:35,016: Using instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-10-14 16:14:35,365: Using instance: open-instance, plan: open


ibm_kingston: 22 jobs in queue
ibm_marrakesh: 10 jobs in queue
→ using ibm_marrakesh

Z basis: 148 zero-SWAP placements on live ibm_marrakesh (2s)
  qiskit     DD off: model predicts 15.55%/round (if DD were perfect:  6.32%) | 0 DD pulses
  qiskit     DD on : model predicts 15.70%/round (if DD were perfect:  6.64%) | 170 DD pulses
  tournament DD off: model predicts 11.67%/round (if DD were perfect:  8.49%) | 0 DD pulses
  tournament DD on : model predicts 13.93%/round (if DD were perfect: 11.07%) | 168 DD pulses

X basis: 148 zero-SWAP placements on live ibm_marrakesh (2s)
  qiskit     DD off: model predicts 12.49%/round (if DD were perfect: 10.37%) | 0 DD pulses
  qiskit     DD on : model predicts 12.89%/round (if DD were perfect: 10.58%) | 156 DD pulses
  tournament DD off: model predicts 12.28%/round (if DD were perfect: 10.24%) | 0 DD pulses
  tournament DD on : model predicts 12.98%/round (if DD were perfect: 10.66%) | 164 DD pulses

PRE-REGISTERED (written before the job runs):


/tmp/ipykernel_707/761915957.py:67: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  job = SamplerV2(mode=hw).run(pubs, shots=HW_SHOTS)



🚀 submitted job db7qkveil99s73foma1g on ibm_marrakesh — saved hh_hw_db7qkveil99s73foma1g.json


In [15]:
import json, glob, stim, pymatching, numpy as np
from qiskit_ibm_runtime import QiskitRuntimeService
JOB_ID = None                       # leave None to use the job from H5, or paste a job id here
from google.colab import userdata  # IBM key comes from Colab Secrets, never from the notebook
API_KEY = userdata.get("IBM_QUANTUM_KEY")
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=API_KEY)
if JOB_ID is None: JOB_ID = HW["job_id"]
files = glob.glob(f"hh_hw_{JOB_ID}.json") + glob.glob(f"/content/drive/MyDrive/qmem_hw/hh_hw_{JOB_ID}.json")
HWr = json.load(open(files[0]))
job = service.job(JOB_ID)
print("status:", job.status(), "— waiting for results…")
res = job.result()

def per_round_(p, r): return 1 - (1 - min(p, 0.4999)) ** (1 / r)
R = HWr["rounds"]; OUT = {}
print(f"\n{HWr['backend']}  d=3, {R} rounds, {HWr['shots']} shots per arm")
print("basis placement   DD  | hardware %/round (±2σ)    | model  | detector rate hw/model")
for k, arm in enumerate(HWr["arms"]):
    bs = res[k].data.c.get_bitstrings()
    arr = (np.frombuffer("".join(bs).encode(), np.uint8).reshape(len(bs), -1) - 48)[:, ::-1].astype(bool)
    det = np.stack([np.logical_xor.reduce(arr[:, d], axis=1) for d in arm["dets"]], axis=1)
    ob = np.logical_xor.reduce(arr[:, arm["obs"]], axis=1)
    dem = stim.Circuit(arm["stim"]).detector_error_model(decompose_errors=True, ignore_decomposition_failures=True,
                                                         approximate_disjoint_errors=True)
    pred = pymatching.Matching.from_detector_error_model(dem).decode_batch(det)[:, 0].astype(bool)
    N = len(ob); p = float(np.mean(pred != ob)); s = (p * (1 - p) / N) ** 0.5
    pr, lo, hi = per_round_(p, R), per_round_(max(p - 2 * s, 0), R), per_round_(p + 2 * s, R)
    OUT[(arm["basis"], arm["layout_name"], arm["dd"])] = (pr, lo, hi)
    print(f"  {arm['basis']}   {arm['layout_name']:10s} {'on ' if arm['dd'] else 'off'} | "
          f"{pr*100:5.2f}%  [{lo*100:5.2f}, {hi*100:5.2f}]  | {arm['pred']*100:5.2f}% | "
          f"{det.mean():.3f} / {arm['sim_det_rate']:.3f}   (undecoded flip {ob.mean()*100:4.1f}%)")

print("\nVERDICTS")
p1 = [(b, dd, OUT[(b, 'tournament', dd)][0] < OUT[(b, 'qiskit', dd)][0]) for b in ("Z", "X") for dd in (False, True)]
p2 = [(b, l, OUT[(b, l, True)][0] < OUT[(b, l, False)][0]) for b in ("Z", "X") for l in ("qiskit", "tournament")]
for b, dd, w in p1:
    r = OUT[(b, 'qiskit', dd)][0] / max(OUT[(b, 'tournament', dd)][0], 1e-9)
    print(f"  P1 {b} DD {'on ' if dd else 'off'}: tournament vs Qiskit ×{r:.2f} {'✅' if w else '❌'}")
for b, l, w in p2:
    r = OUT[(b, l, False)][0] / max(OUT[(b, l, True)][0], 1e-9)
    print(f"  P2 {b} {l:10s}: DD gain ×{r:.2f} {'✅' if w else '❌'}")
bestZ = min(v[0] for k, v in OUT.items() if k[0] == "Z")
print(f"  P1 {'HOLDS' if sum(w for *_, w in p1) >= 3 else 'FAILS'} ({sum(w for *_, w in p1)}/4) | "
      f"P2 {'HOLDS' if sum(w for *_, w in p2) >= 3 else 'FAILS'} ({sum(w for *_, w in p2)}/4) | "
      f"P3 best Z {bestZ*100:.2f}%/round {'✅ ≤6%' if bestZ <= 0.06 else '❌ >6%'}")

qiskit_runtime_service._discover_account:WARNING:2026-10-14 16:15:33,473: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-10-14 16:15:36,439: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.


status: QUEUED — waiting for results…

ibm_marrakesh  d=3, 3 rounds, 10000 shots per arm
basis placement   DD  | hardware %/round (±2σ)    | model  | detector rate hw/model
  Z   qiskit     off | 19.44%  [18.93, 19.95]  | 15.55% | 0.257 / 0.197   (undecoded flip 49.5%)
  Z   qiskit     on  | 15.44%  [14.99, 15.90]  | 15.70% | 0.209 / 0.197   (undecoded flip 49.9%)
  Z   tournament off | 18.07%  [17.58, 18.57]  | 11.67% | 0.249 / 0.189   (undecoded flip 49.4%)
  Z   tournament on  | 14.22%  [13.78, 14.66]  | 13.93% | 0.215 / 0.193   (undecoded flip 50.0%)
  X   qiskit     off | 20.53%  [20.01, 20.62]  | 12.49% | 0.231 / 0.169   (undecoded flip 50.5%)
  X   qiskit     on  | 16.97%  [16.49, 17.45]  | 12.89% | 0.207 / 0.174   (undecoded flip 50.3%)
  X   tournament off | 19.66%  [19.14, 20.18]  | 12.28% | 0.230 / 0.169   (undecoded flip 49.2%)
  X   tournament on  | 15.95%  [15.49, 16.41]  | 12.98% | 0.201 / 0.173   (undecoded flip 50.2%)

VERDICTS
  P1 Z DD off: tournament vs Qiskit ×1.08

In [17]:
import json, os
import rustworkx as rx
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import ALAPScheduleAnalysis
from qiskit.transpiler.passes import PadDelay

from google.colab import userdata  # IBM key comes from Colab Secrets, never from the notebook
API_KEY = userdata.get("IBM_QUANTUM_KEY")
service = QiskitRuntimeService(
    channel="ibm_quantum_platform", token=API_KEY)
hw = service.backend("ibm_kingston")
print(hw.name, hw.status().pending_jobs, "jobs in queue")
noise = ChipNoise(hw)
ROUNDS_LIST, SHOTS7, D7 = (1, 2, 3, 4), 5000, 3

# staggered DD: colour A gets X.X, colour B gets X.X.X.X
g = rx.PyGraph()
g.add_nodes_from(range(hw.num_qubits))
E = {tuple(sorted(e)) for e in hw.coupling_map.get_edges()}
g.add_edges_from_no_data(list(E))
COLOR = rx.two_color(g)
assert COLOR is not None, "chip graph not bipartite"
t_ = hw.target
ALIGN = max(getattr(t_, "granularity", 16) or 16,
            getattr(t_, "pulse_alignment", 16) or 16)
SPACING = {2: [0.25, 0.5, 0.25],
           4: [0.125, 0.25, 0.25, 0.25, 0.125]}
XD = {}
def xdur(q):
    if q not in XD:
        d_ = hw.target["x"][(q,)].duration
        XD[q] = int(round(d_ / hw.dt))
    return XD[q]

def add_dd(sc, mode):
    out = sc.copy_empty_like()
    fresh = set(range(sc.num_qubits))
    npulse = 0
    for inst in sc.data:
        op = inst.operation
        qs = [sc.find_bit(q).index for q in inst.qubits]
        unit = getattr(op, "unit", "dt")
        if (op.name == "delay" and qs[0] not in fresh
                and unit == "dt"):
            q = qs[0]
            T = int(op.params[0])
            n = 2
            if mode == "stag" and COLOR[q] == 1:
                n = 4
            xd = xdur(q)
            F = T - n * xd
            ok = (T % ALIGN == 0 and xd % ALIGN == 0
                  and F >= 2 * n * ALIGN)
            if ok:
                sp = SPACING[n]
                pieces = [int(F * s) // ALIGN * ALIGN
                          for s in sp[:-1]]
                pieces.append(F - sum(pieces))
                qb = inst.qubits[0]
                for k in range(n):
                    if pieces[k] > 0:
                        out.delay(pieces[k], qb, unit="dt")
                    out.x(qb)
                if pieces[-1] > 0:
                    out.delay(pieces[-1], qb, unit="dt")
                npulse += n
                continue
        out.append(inst)
        if op.name == "reset":
            fresh.update(qs)
        elif op.name not in ("delay", "barrier", "measure"):
            fresh.difference_update(qs)
    return out, npulse

def route(qc, lay=None):
    pm = generate_preset_pass_manager(
        optimization_level=1, target=hw.target,
        initial_layout=lay, seed_transpiler=1)
    return pm.run(qc)

def sched(routed):
    pm = PassManager([ALAPScheduleAnalysis(target=hw.target),
                      PadDelay(target=hw.target)])
    return pm.run(routed)

def slope_fit(rs, ps, N=None):
    # F(r) = 1 - 2 pL(r) = A (1 - 2 eps)^r  -> eps per round
    rs = np.array(rs, float)
    ps = np.array(ps, float)
    F = np.clip(1 - 2 * ps, 1e-3, None)
    y = np.log(F)
    if N:
        sp = np.sqrt(np.clip(ps * (1 - ps), 1e-9, None) / N)
        w = (F / (2 * sp)) ** 2
    else:
        w = np.ones_like(y)
    X = np.vstack([np.ones_like(rs), rs]).T
    cov = np.linalg.inv(X.T @ (w[:, None] * X))
    beta = cov @ X.T @ (w * y)
    se = np.sqrt(cov[1, 1]) if N else 0.0
    f = lambda s: (1 - np.exp(s)) / 2
    return f(beta[1]), f(beta[1] + 2 * se), f(beta[1] - 2 * se)

def with_flags(isa, dets, fl, obs):
    fired, obf = check_dets(isa, noise, dets, obs)
    assert not fired.any() and not obf, "noiseless check failed"
    ff, _ = check_dets(isa, noise, fl, obs)
    good = [f for f, bad in zip(fl, ff) if not bad]
    return dets + good

H7 = {"backend": hw.name, "shots": SHOTS7, "arms": []}
pubs = []
for basis in ("Z", "X"):
    t0 = time.time()
    qc3, dets3, fl3, obs3, S = build_hhc_memory(D7, 3, basis)
    n_cx3 = qc3.count_ops()["cx"]
    r_def = route(qc3)
    lay_def = list(
        r_def.layout.initial_index_layout(filter_ancillas=True))
    cg, uses = code_graph(qc3)
    meas, rst = role_counts(qc3)
    pool = candidate_layouts(hw, cg, np.random.default_rng(7))

    def score(L):
        s = model_score(L, uses, meas, rst, noise,
                        S["n"], 2 * 3 * 5e-6)
        for i, p in enumerate(L):
            s += noise.sx(p) * (40 if i < S["n"] else 20)
        return s
    scored = sorted(pool, key=score)

    def pilot(routed):
        isa, _ = add_dd(sched(routed), "stag")
        alld = with_flags(isa, dets3, fl3, obs3)
        circ = isa_to_stim(isa, noise, alld, obs3)
        return logical_error(circ, 20000, seed=11)

    best = None
    for L in scored[:TOP_K]:
        if L == lay_def:
            continue
        try:
            r = route(qc3, L)
        except Exception:
            continue
        if r.count_ops().get("cz", 0) != n_cx3:
            continue
        p = pilot(r)
        if best is None or p < best[0]:
            best = (p, L)
    p_def = pilot(r_def)
    print()
    print(basis, "basis:", len(pool), "placements")
    print("  pilot 3 rounds: Qiskit %.1f%%  tournament %.1f%%"
          % (p_def * 100, best[0] * 100))

    for lname, L in (("qiskit", lay_def),
                     ("tournament", best[1])):
        for mode in ("xx", "stag"):
            preds = []
            for R in ROUNDS_LIST:
                qc, dets, fl, obs, _ = build_hhc_memory(D7, R, basis)
                routed = route(qc, L)
                ncz = routed.count_ops().get("cz", 0)
                assert ncz == qc.count_ops()["cx"], "SWAPs appeared"
                isa, npul = add_dd(sched(routed), mode)
                assert npul > 0, "no DD pulses - nothing submitted"
                alld = with_flags(isa, dets, fl, obs)
                circ = isa_to_stim(isa, noise, alld, obs, noisy=True)
                pm_ = logical_error(circ, 50000, seed=999)
                preds.append(pm_)
                H7["arms"].append({
                    "basis": basis, "layout_name": lname,
                    "dd": mode, "rounds": R,
                    "layout": [int(x) for x in L],
                    "dets": alld, "obs": obs[0],
                    "pred_pL": pm_, "npulse": npul,
                    "stim": str(circ)})
                pubs.append(isa)
            e, _, _ = slope_fit(ROUNDS_LIST, preds)
            txt = " ".join("%.1f%%" % (p * 100) for p in preds)
            print("  %-10s DD %-4s" % (lname, mode),
                  "model pL r=1..4:", txt)
            print("      -> eps %.2f%% per round, %d DD pulses"
                  % (e * 100, npul))

print()
print("PRE-REGISTERED before the job runs:")
print("  P4: staggered DD beats XX in >=3 of 4")
print("  P5: tournament beats Qiskit in >=3 of 4")
print("  P6: best Z eps <= 6% per round (IBM 2022)")
job = SamplerV2(mode=hw).run(pubs, shots=SHOTS7)
H7["job_id"] = job.job_id()
fname = "hh_hw7_" + H7["job_id"] + ".json"
json.dump(H7, open(fname, "w"))
dr = "/content/drive/MyDrive"
if os.path.isdir(dr):
    os.makedirs(dr + "/qmem_hw", exist_ok=True)
    json.dump(H7, open(dr + "/qmem_hw/" + fname, "w"))
print("SUBMITTED job", H7["job_id"], "on", hw.name,
      "-", len(pubs), "circuits")

qiskit_runtime_service._discover_account:WARNING:2026-10-14 16:26:35,771: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-10-14 16:26:38,821: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-10-14 16:26:38,822: Using instance: open-instance, plan: open


ibm_kingston 9 jobs in queue

Z basis: 148 placements
  pilot 3 rounds: Qiskit 14.4%  tournament 12.6%
  qiskit     DD xx   model pL r=1..4: 3.4% 8.7% 14.5% 19.6%
      -> eps 6.67% per round, 216 DD pulses
  qiskit     DD stag model pL r=1..4: 3.5% 9.0% 14.5% 19.5%
      -> eps 6.57% per round, 344 DD pulses
  tournament DD xx   model pL r=1..4: 2.4% 7.2% 12.6% 17.6%
      -> eps 6.06% per round, 216 DD pulses
  tournament DD stag model pL r=1..4: 2.5% 7.1% 12.8% 17.6%
      -> eps 6.04% per round, 344 DD pulses

X basis: 148 placements
  pilot 3 rounds: Qiskit 13.7%  tournament 13.6%
  qiskit     DD xx   model pL r=1..4: 3.4% 8.3% 13.5% 17.7%
      -> eps 5.80% per round, 214 DD pulses
  qiskit     DD stag model pL r=1..4: 3.4% 8.2% 13.4% 18.0%
      -> eps 5.92% per round, 342 DD pulses
  tournament DD xx   model pL r=1..4: 3.9% 8.7% 13.4% 18.0%
      -> eps 5.72% per round, 214 DD pulses
  tournament DD stag model pL r=1..4: 4.0% 8.6% 13.8% 18.4%
      -> eps 5.93% per round, 342 D

/tmp/ipykernel_707/383175766.py:192: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  job = SamplerV2(mode=hw).run(pubs, shots=SHOTS7)


SUBMITTED job db7qqker6oqs73884obg on ibm_kingston - 32 circuits


In [18]:
import json, glob, stim, pymatching, numpy as np
from qiskit_ibm_runtime import QiskitRuntimeService
JOB7 = None   # None = job from H7, or paste the job id here
from google.colab import userdata  # IBM key comes from Colab Secrets, never from the notebook
API_KEY = userdata.get("IBM_QUANTUM_KEY")
service = QiskitRuntimeService(
    channel="ibm_quantum_platform", token=API_KEY)
if JOB7 is None:
    JOB7 = H7["job_id"]
fn = "hh_hw7_" + JOB7 + ".json"
DRV = "/content/drive/MyDrive/qmem_hw/"
hits = glob.glob(fn) + glob.glob(DRV + fn)
A = json.load(open(hits[0]))
job = service.job(JOB7)
print("status:", job.status(), "- waiting for results...")
res = job.result()

def slope_fit(rs, ps, N=None):
    rs = np.array(rs, float)
    ps = np.array(ps, float)
    F = np.clip(1 - 2 * ps, 1e-3, None)
    y = np.log(F)
    if N:
        sp = np.sqrt(np.clip(ps * (1 - ps), 1e-9, None) / N)
        w = (F / (2 * sp)) ** 2
    else:
        w = np.ones_like(y)
    X = np.vstack([np.ones_like(rs), rs]).T
    cov = np.linalg.inv(X.T @ (w[:, None] * X))
    beta = cov @ X.T @ (w * y)
    se = np.sqrt(cov[1, 1]) if N else 0.0
    f = lambda s: (1 - np.exp(s)) / 2
    return f(beta[1]), f(beta[1] + 2 * se), f(beta[1] - 2 * se)

G = {}
for k, arm in enumerate(A["arms"]):
    bs = res[k].data.c.get_bitstrings()
    raw = np.frombuffer("".join(bs).encode(), np.uint8)
    arr = (raw.reshape(len(bs), -1) - 48)[:, ::-1].astype(bool)
    det = np.stack([np.logical_xor.reduce(arr[:, d], axis=1)
                    for d in arm["dets"]], axis=1)
    ob = np.logical_xor.reduce(arr[:, arm["obs"]], axis=1)
    dem = stim.Circuit(arm["stim"]).detector_error_model(
        decompose_errors=True,
        ignore_decomposition_failures=True,
        approximate_disjoint_errors=True)
    m = pymatching.Matching.from_detector_error_model(dem)
    pred = m.decode_batch(det)[:, 0].astype(bool)
    key = (arm["basis"], arm["layout_name"], arm["dd"])
    G.setdefault(key, []).append(
        (arm["rounds"], float(np.mean(pred != ob)),
         arm["pred_pL"], float(det.mean())))

N = A["shots"]
EPS = {}
print()
print(A["backend"], "d=3,", N, "shots per circuit")
for key, rows in G.items():
    rows.sort()
    rs = [r[0] for r in rows]
    e, lo, hi = slope_fit(rs, [r[1] for r in rows], N)
    em, _, _ = slope_fit(rs, [r[2] for r in rows])
    EPS[key] = (e, lo, hi, em)
    txt = " ".join("%.1f%%" % (r[1] * 100) for r in rows)
    print(" ", key[0], key[1], "DD", key[2])
    print("    hw pL r=1..4:", txt)
    print("    eps %.2f%% [%.2f, %.2f]  model %.2f%%"
          % (e * 100, lo * 100, hi * 100, em * 100))
    print("    det rate r=3 hw %.3f" % rows[2][3])

print()
print("VERDICTS")
n4 = 0
for b in ("Z", "X"):
    for l in ("qiskit", "tournament"):
        xx = EPS[(b, l, "xx")][0]
        st = EPS[(b, l, "stag")][0]
        n4 += st < xx
        print("  P4", b, l, "stag vs XX x%.2f" % (xx / max(st, 1e-9)),
              "OK" if st < xx else "NO")
n5 = 0
for b in ("Z", "X"):
    for m in ("xx", "stag"):
        q_ = EPS[(b, "qiskit", m)][0]
        t_ = EPS[(b, "tournament", m)][0]
        n5 += t_ < q_
        print("  P5", b, m, "tournament vs Qiskit x%.2f"
              % (q_ / max(t_, 1e-9)), "OK" if t_ < q_ else "NO")
bz = min((v[0], k) for k, v in EPS.items() if k[0] == "Z")
print("  P4", "HOLDS" if n4 >= 3 else "FAILS", n4, "/4")
print("  P5", "HOLDS" if n5 >= 3 else "FAILS", n5, "/4")
print("  P6 best Z eps %.2f%%" % (bz[0] * 100), bz[1][1], bz[1][2],
      "OK <=6%" if bz[0] <= 0.06 else "NO >6%")

qiskit_runtime_service._discover_account:WARNING:2026-10-14 16:27:14,345: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-10-14 16:27:17,248: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.


status: RUNNING - waiting for results...

ibm_kingston d=3, 5000 shots per circuit
  Z qiskit DD xx
    hw pL r=1..4: 3.5% 10.2% 17.2% 21.4%
    eps 7.63% [7.17, 8.09]  model 6.67%
    det rate r=3 hw 0.113
  Z qiskit DD stag
    hw pL r=1..4: 2.9% 9.3% 16.7% 23.7%
    eps 8.15% [7.69, 8.61]  model 6.57%
    det rate r=3 hw 0.115
  Z tournament DD xx
    hw pL r=1..4: 2.7% 9.2% 16.2% 21.7%
    eps 7.67% [7.23, 8.11]  model 6.06%
    det rate r=3 hw 0.114
  Z tournament DD stag
    hw pL r=1..4: 2.6% 8.8% 15.6% 21.5%
    eps 7.48% [7.04, 7.91]  model 6.04%
    det rate r=3 hw 0.113
  X qiskit DD xx
    hw pL r=1..4: 4.7% 11.2% 17.6% 21.9%
    eps 7.47% [6.97, 7.96]  model 5.80%
    det rate r=3 hw 0.114
  X qiskit DD stag
    hw pL r=1..4: 4.4% 11.3% 17.7% 22.8%
    eps 7.87% [7.38, 8.37]  model 5.92%
    det rate r=3 hw 0.111
  X tournament DD xx
    hw pL r=1..4: 4.5% 9.8% 15.7% 21.0%
    eps 6.73% [6.26, 7.20]  model 5.72%
    det rate r=3 hw 0.108
  X tournament DD stag
    hw pL r=

In [19]:
import json, glob, stim, pymatching, numpy as np
from qiskit_ibm_runtime import QiskitRuntimeService
JOB9 = "db7qqker6oqs73884obg"
from google.colab import userdata  # IBM key comes from Colab Secrets, never from the notebook
API_KEY = userdata.get("IBM_QUANTUM_KEY")
service = QiskitRuntimeService(
    channel="ibm_quantum_platform", token=API_KEY)
fn = "hh_hw7_" + JOB9 + ".json"
DRV = "/content/drive/MyDrive/qmem_hw/"
A = json.load(open((glob.glob(fn) + glob.glob(DRV + fn))[0]))
res9 = service.job(JOB9).result()

def corr_p(D):
    D = D.astype(float)
    x = D.mean(0)
    xx = D.T @ D / len(D)
    num = xx - np.outer(x, x)
    den = 1 - 2 * x[:, None] - 2 * x[None, :] + 4 * xx
    with np.errstate(all="ignore"):
        P = 0.5 - 0.5 * np.sqrt(1 - 4 * num / den)
    return x, P

def ok_p(p, fallback):
    if not np.isfinite(p) or p <= 0:
        p = fallback
    return float(np.clip(p, 1e-5, 0.45))

def edge_list(dem):
    m = pymatching.Matching.from_detector_error_model(dem)
    out = []
    for u, v, a in m.edges():
        p = a.get("error_probability", -1)
        if not (0 < p < 0.5):
            p = 1 / (1 + np.exp(a["weight"]))
        out.append((u, v, sorted(a["fault_ids"]), p))
    return out

def matcher(edges, probs, ndet):
    lines = []
    for (u, v, obs, _), p in zip(edges, probs):
        t = "error(%.8f) D%d" % (p, u)
        if v is not None:
            t += " D%d" % v
        for o in obs:
            t += " L%d" % o
        lines.append(t)
    lines.append("detector D%d" % (ndet - 1))
    dem = stim.DetectorErrorModel("\n".join(lines))
    return pymatching.Matching.from_detector_error_model(dem)

def learned_probs(edges, D):
    x, P = corr_p(D)
    newp, nb = [], {}
    for u, v, obs, p0 in edges:
        if v is None:
            newp.append(None)
            continue
        p = ok_p(P[u, v], p0)
        newp.append(p)
        nb[u] = nb.get(u, 1.0) * (1 - 2 * p)
        nb[v] = nb.get(v, 1.0) * (1 - 2 * p)
    for k, (u, v, obs, p0) in enumerate(edges):
        if v is None:
            q = 0.5 - 0.5 * (1 - 2 * x[u]) / nb.get(u, 1.0)
            newp[k] = ok_p(q, p0)
    return newp

def slope_fit(rs, ps, N=None):
    rs = np.array(rs, float)
    ps = np.array(ps, float)
    F = np.clip(1 - 2 * ps, 1e-3, None)
    y = np.log(F)
    if N:
        sp = np.sqrt(np.clip(ps * (1 - ps), 1e-9, None) / N)
        w = (F / (2 * sp)) ** 2
    else:
        w = np.ones_like(y)
    X = np.vstack([np.ones_like(rs), rs]).T
    cov = np.linalg.inv(X.T @ (w[:, None] * X))
    beta = cov @ X.T @ (w * y)
    se = np.sqrt(cov[1, 1]) if N else 0.0
    f = lambda s: (1 - np.exp(s)) / 2
    return f(beta[1]), f(beta[1] + 2 * se), f(beta[1] - 2 * se)

G9 = {}
for k, arm in enumerate(A["arms"]):
    bs = res9[k].data.c.get_bitstrings()
    raw = np.frombuffer("".join(bs).encode(), np.uint8)
    arr = (raw.reshape(len(bs), -1) - 48)[:, ::-1].astype(bool)
    det = np.stack([np.logical_xor.reduce(arr[:, d], axis=1)
                    for d in arm["dets"]], axis=1)
    ob = np.logical_xor.reduce(arr[:, arm["obs"]], axis=1)
    circ = stim.Circuit(arm["stim"])
    dem = circ.detector_error_model(
        decompose_errors=True,
        ignore_decomposition_failures=True,
        approximate_disjoint_errors=True)
    E = edge_list(dem)
    nd = det.shape[1]
    p_model = [e[3] for e in E]
    sim = circ.compile_detector_sampler(seed=4).sample(20000)
    halves = [np.arange(len(ob)) % 2 == 0,
              np.arange(len(ob)) % 2 == 1]
    err = {"model": 0, "rescaled": 0, "learned": 0}
    for tr, te in ((halves[0], halves[1]), (halves[1], halves[0])):
        s = det[tr].mean() / max(sim.mean(), 1e-9)
        p_res = [float(np.clip(p * s, 1e-5, 0.45)) for p in p_model]
        p_lrn = learned_probs(E, det[tr])
        for name, ps in (("model", p_model), ("rescaled", p_res),
                         ("learned", p_lrn)):
            m = matcher(E, ps, nd)
            pr = m.decode_batch(det[te])[:, 0].astype(bool)
            err[name] += int(np.sum(pr != ob[te]))
    key = (arm["basis"], arm["layout_name"], arm["dd"])
    G9.setdefault(key, []).append(
        (arm["rounds"], {n: v / len(ob) for n, v in err.items()}))
    print("decoded circuit", k + 1, "of", len(A["arms"]))

N = A["shots"]
R9 = {}
print()
print("eps per round (SPAM-free), 2-fold cross-validated")
print("                       model   rescaled  learned")
for key, rows in G9.items():
    rows.sort(key=lambda r: r[0])
    rs = [r[0] for r in rows]
    out = {}
    for n in ("model", "rescaled", "learned"):
        out[n] = slope_fit(rs, [r[1][n] for r in rows], N)
    R9[key] = out
    print("  %s %-10s %-4s  %5.2f%%   %5.2f%%   %5.2f%%  [%.2f, %.2f]"
          % (key[0], key[1], key[2], out["model"][0] * 100,
             out["rescaled"][0] * 100, out["learned"][0] * 100,
             out["learned"][1] * 100, out["learned"][2] * 100))

gain = [R9[k]["model"][0] / R9[k]["learned"][0] for k in R9]
wins = sum(R9[k]["learned"][0] < R9[k]["model"][0] for k in R9)
print()
print("learned beats model weights in", wins, "of", len(R9),
      "| median gain x%.2f" % np.median(gain))
bz = min((v["learned"][0], k) for k, v in R9.items() if k[0] == "Z")
bx = min((v["learned"][0], k) for k, v in R9.items() if k[0] == "X")
print("best Z eps %.2f%%" % (bz[0] * 100), bz[1][1], bz[1][2],
      "| best X eps %.2f%%" % (bx[0] * 100), bx[1][1], bx[1][2])

qiskit_runtime_service._discover_account:WARNING:2026-10-14 16:31:25,861: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-10-14 16:31:28,930: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.


decoded circuit 1 of 32
decoded circuit 2 of 32
decoded circuit 3 of 32
decoded circuit 4 of 32
decoded circuit 5 of 32
decoded circuit 6 of 32
decoded circuit 7 of 32
decoded circuit 8 of 32
decoded circuit 9 of 32
decoded circuit 10 of 32
decoded circuit 11 of 32
decoded circuit 12 of 32
decoded circuit 13 of 32
decoded circuit 14 of 32
decoded circuit 15 of 32
decoded circuit 16 of 32
decoded circuit 17 of 32
decoded circuit 18 of 32
decoded circuit 19 of 32
decoded circuit 20 of 32
decoded circuit 21 of 32
decoded circuit 22 of 32
decoded circuit 23 of 32
decoded circuit 24 of 32
decoded circuit 25 of 32
decoded circuit 26 of 32
decoded circuit 27 of 32
decoded circuit 28 of 32
decoded circuit 29 of 32
decoded circuit 30 of 32
decoded circuit 31 of 32
decoded circuit 32 of 32

eps per round (SPAM-free), 2-fold cross-validated
                       model   rescaled  learned
  Z qiskit     xx     7.63%    7.62%    7.61%  [7.15, 8.07]
  Z qiskit     stag   8.15%    8.15%    7.79%  [7

In [20]:
import stim, numpy as np
from collections import Counter

def mod2(lst):
    return sorted(b for b, k in Counter(lst).items() if k % 2)

def build_hhc_v2(d, rounds, basis, merge=False,
                 skip_safe=False, skip_flags=False):
    S = hhc_structure(d)
    n, xg, zg = S["n"], S["xg"], S["zg"]
    sched_ = ("xz" if basis == "Z" else "zx") * rounds
    nx, nz = len(xg), len(zg)
    ncl = sum(nx if r == "x" else 3 * nz for r in sched_) + n
    qc = QuantumCircuit(QuantumRegister(S["nq"], "q"),
                        ClassicalRegister(ncl, "c"))
    cb = iter(range(ncl))
    last, pend, prev = {}, {}, {}

    def RESET(q, skip):
        if skip:
            if q in last:
                pend[q] = last[q]
        else:
            qc.reset(q)
            pend.pop(q, None)

    def MEAS(q):
        b = next(cb)
        qc.measure(q, b)
        if q in pend:
            prev[b] = pend.pop(q)
        last[q] = b
        return b

    for i in range(n):
        qc.reset(i)
    if basis == "X":
        for i in range(n):
            qc.h(i)
    log = []
    for r in sched_:
        if r == "x":
            for a in S["xanc"]:
                RESET(a, skip_safe)
                qc.h(a)
            for step in range(2):
                for k, g in enumerate(xg):
                    qc.cx(S["xanc"][k], g[step])
            bits = []
            for a in S["xanc"]:
                qc.h(a)
                bits.append(MEAS(a))
            log.append(("x", bits))
            continue
        gb, lf, rf = [None] * nz, [None] * nz, [None] * nz
        for step in range(7):
            for j, z in enumerate(zg):
                l, m, rr = S["zanc"][j]
                bnd, top = len(z) == 2, z[0] < d
                if step == 0:
                    RESET(m, skip_safe)
                    RESET(rr, skip_flags)
                    qc.h(rr)
                elif step == 1:
                    qc.cx(rr, m)
                    RESET(l, skip_flags)
                    qc.h(l)
                elif step == 2:
                    qc.cx(l, m)
                    if not (bnd and top):
                        qc.cx(z[1], rr)
                elif step == 3:
                    if bnd and top:
                        qc.cx(z[0], l)
                        qc.cx(z[1], rr)
                    elif not bnd:
                        qc.cx(z[2], l)
                        qc.cx(z[3], rr)
                elif step == 4:
                    qc.cx(rr, m)
                    if not (bnd and top):
                        qc.cx(z[0], l)
                elif step == 5:
                    qc.cx(l, m)
                    qc.h(rr)
                    if not merge:
                        rf[j] = MEAS(rr)
                elif step == 6:
                    qc.h(l)
                    if merge:
                        rf[j] = MEAS(rr)
                    gb[j] = MEAS(m)
                    lf[j] = MEAS(l)
        log.append(("z", gb, lf, rf))
    if basis == "X":
        for i in range(n):
            qc.h(i)
    data_bits = [MEAS(i) for i in range(n)]

    eff = lambda b: [b] + ([prev[b]] if b in prev else [])
    E = lambda bits: [x for b in bits for x in eff(b)]
    dets, flag_dets = [], []
    zr = [e for e in log if e[0] == "z"]
    xr = [e for e in log if e[0] == "x"]
    zs = lambda e, j: E([e[1][g] for g in range(nz)
                         if S["zcol"][g] == j])
    xs = lambda e, s: E([e[1][g] for g in S["xstab"][s]])
    for j in range(d - 1):
        for t, e in enumerate(zr):
            if t == 0:
                if basis == "Z":
                    dets.append(zs(e, j))
            else:
                dets.append(zs(e, j) + zs(zr[t - 1], j))
        if basis == "Z":
            cols = [data_bits[r_ * d + j] for r_ in range(d)]
            cols += [data_bits[r_ * d + j + 1] for r_ in range(d)]
            dets.append(cols + zs(zr[-1], j))
    for s, gs in enumerate(S["xstab"]):
        for t, e in enumerate(xr):
            if t == 0:
                if basis == "X":
                    dets.append(xs(e, s))
            else:
                dets.append(xs(e, s) + xs(xr[t - 1], s))
        if basis == "X":
            sup = sorted({q_ for g in gs for q_ in xg[g]})
            dets.append([data_bits[q_] for q_ in sup] + xs(xr[-1], s))
    for e in zr:
        flag_dets += [E([b]) for b in e[2] + e[3]]
    obs = list(data_bits)
    Q = sorted(set(prev.values()))
    return qc, dets, flag_dets, obs, S, Q

def clean_samples(qc, shots, seed=0):
    c, order = stim.Circuit(), []
    for inst in qc.data:
        nm = inst.operation.name
        qs = [qc.find_bit(q).index for q in inst.qubits]
        if nm == "h": c.append("H", qs)
        elif nm == "cx": c.append("CX", qs)
        elif nm == "reset": c.append("R", qs)
        elif nm == "measure":
            c.append("M", qs)
            order.append(qc.find_bit(inst.clbits[0]).index)
    raw = c.compile_sampler(seed=seed).sample(shots)
    B = np.zeros((shots, qc.num_clbits), np.uint8)
    B[:, order] = raw
    return B

def gf2_solve(A, y):
    M = np.concatenate([A, y[:, None]], 1).astype(np.uint8) & 1
    nr, nc = A.shape
    piv, r = [], 0
    for c in range(nc):
        rows = np.nonzero(M[r:, c])[0]
        if len(rows) == 0:
            continue
        p = r + rows[0]
        M[[r, p]] = M[[p, r]]
        oth = np.nonzero(M[:, c])[0]
        oth = oth[oth != r]
        M[oth] ^= M[r]
        piv.append(c)
        r += 1
        if r == nr:
            break
    if M[r:, -1].any():
        return None
    x = np.zeros(nc, np.uint8)
    for i, c in enumerate(piv):
        x[c] = M[i, -1]
    return x

def repair(lst, B, Q):
    lst = mod2(lst)
    y = B[:, lst].sum(1) % 2
    if y.min() == y.max():
        return lst
    if not Q:
        return None
    A = np.concatenate([B[:, Q], np.ones((len(B), 1), np.uint8)], 1)
    x = gf2_solve(A, y.astype(np.uint8))
    if x is None:
        return None
    return mod2(lst + [Q[i] for i in range(len(Q)) if x[i]])

def make_variant(d, rounds, basis, **kw):
    qc, dets, fl, obs, S, Q = build_hhc_v2(d, rounds, basis, **kw)
    B = clean_samples(qc, 3 * qc.num_clbits + 200)
    dets2 = [repair(x, B, Q) for x in dets]
    bad = sum(x is None for x in dets2)
    fl2 = [x for x in (repair(f, B, Q) for f in fl) if x is not None]
    ob2 = repair(obs, B, Q)
    assert ob2 is not None, "observable could not be repaired"
    return qc, [x for x in dets2 if x is not None], fl2, \
        {0: ob2}, S, bad

print("compressed builder ready")

compressed builder ready


In [23]:
RZ2 = {0: None, 1: "S", 2: "Z", 3: "S_DAG"}
def isa_to_stim2(isa, noise, dets, obs, noisy=True):
    out, pos, nmeas, active = stim.Circuit(), {}, 0, set()
    for inst in isa.data:
        op, name = inst.operation, inst.operation.name
        qs = [isa.find_bit(q).index for q in inst.qubits]
        if name in ("barrier", "id"):
            continue
        if name == "delay":
            if noisy and qs[0] in active:
                val = float(op.params[0])
                u = getattr(op, "unit", "dt")
                dur = val * noise.dt if u == "dt" else val
                if dur > 0:
                    out.append("PAULI_CHANNEL_1", qs,
                               list(noise.idle(qs[0], dur)))
            continue
        if name == "rz":
            ang = float(op.params[0]) / (math.pi / 2)
            k = round(ang)
            if abs(ang - k) > 1e-6:
                raise ValueError("non-Clifford rz")
            if RZ2[k % 4]:
                out.append(RZ2[k % 4], qs)
            active.update(qs)
        elif name in ("sx", "sxdg", "x"):
            g = {"sx": "SQRT_X", "sxdg": "SQRT_X_DAG", "x": "X"}[name]
            out.append(g, qs)
            if noisy:
                out.append("DEPOLARIZE1", qs, noise.sx(qs[0]))
            active.update(qs)
        elif name == "cz":
            out.append("CZ", qs)
            if noisy:
                out.append("DEPOLARIZE2", qs, noise.cz(*qs))
            active.update(qs)
        elif name == "measure":
            if noisy:
                out.append("X_ERROR", qs, noise.ro(qs[0]))
            out.append("M", qs)
            pos[isa.find_bit(inst.clbits[0]).index] = nmeas
            nmeas += 1
            active.update(qs)
        elif name == "reset":
            out.append("R", qs)
            if noisy:
                out.append("X_ERROR", qs, noise.ro(qs[0]))
            active.difference_update(qs)
        else:
            raise ValueError("unsupported gate " + name)
    for det in dets:
        out.append("DETECTOR",
                   [stim.target_rec(pos[c] - nmeas) for c in det])
    for k, lst in obs.items():
        out.append("OBSERVABLE_INCLUDE",
                   [stim.target_rec(pos[c] - nmeas) for c in lst], k)
    return out

def duration_s(isa, backend):
    try:
        return isa.estimate_duration(backend.target, unit="s")
    except Exception:
        return float("nan")

VARS = [("baseline", {}),
        ("merge readouts", dict(merge=True)),
        ("no safe resets", dict(skip_safe=True)),
        ("safe + merge", dict(merge=True, skip_safe=True)),
        ("no resets", dict(skip_safe=True, skip_flags=True)),
        ("no resets + merge",
         dict(merge=True, skip_safe=True, skip_flags=True))]
COMP = {}
for chip in HH_CHIPS:
    backend = getattr(fp, chip)()
    noise = ChipNoise(backend)
    for basis in ("Z", "X"):
        for d in (3, 5):
            t0 = time.time()
            print()
            print(chip, basis, "d=%d" % d)
            print("   variant        us/rnd resets dist"
                  "  eps/rnd  gain")
            lay = None
            for name, kw in VARS:
                qc, dets, fl, obs, S, bad = make_variant(
                    d, HH_ROUNDS, basis, **kw)
                pm = generate_preset_pass_manager(
                    optimization_level=1, target=backend.target,
                    seed_transpiler=1, scheduling_method="alap",
                    initial_layout=lay)
                isa = pm.run(qc)
                if lay is None:
                    lay = list(isa.layout.initial_index_layout(
                        filter_ancillas=True))
                ncx = qc.count_ops().get("cx", 0)
                assert isa.count_ops().get("cz", 0) == ncx, "SWAPs"
                fired, obf = check_dets(isa, noise, dets, obs)
                assert not fired.any() and not obf, name + " not clean"
                ff, _ = check_dets(isa, noise, fl, obs)
                good = [f for f, b_ in zip(fl, ff) if not b_]
                circ = isa_to_stim2(isa, noise, dets + good, obs)
                dist = len(dem_of(circ).shortest_graphlike_error(
                    ignore_ungraphlike_errors=True))
                shots = 60000 if d == 3 else 20000
                eps = per_round(logical_error(circ, shots, seed=7),
                                HH_ROUNDS)
                us = duration_s(isa, backend) / HH_ROUNDS * 1e6
                nres = qc.count_ops().get("reset", 0)
                COMP[(chip, basis, d, name)] = (eps, us, dist, nres)
                b0 = COMP[(chip, basis, d, "baseline")][0]
                print("   %-18s %6.2f %5d %4d   %6.2f%%  x%.2f%s"
                      % (name, us, nres, dist, eps * 100,
                         b0 / max(eps, 1e-9),
                         "" if bad == 0 else "  (%d dets lost)" % bad))
            print("   (%.0fs)" % (time.time() - t0))


FakeKingston Z d=3
   variant        us/rnd resets dist  eps/rnd  gain
   baseline             9.81    63    3     6.63%  x1.00
   merge readouts       9.81    63    3     6.63%  x1.00
   no safe resets       8.27    33    3     5.56%  x1.19
   safe + merge         8.27    33    3     5.64%  x1.18
   no resets            5.95     9    3     3.45%  x1.92
   no resets + merge    5.95     9    3     3.32%  x2.00
   (2s)

FakeKingston Z d=5
   variant        us/rnd resets dist  eps/rnd  gain
   baseline             9.86   193    5    10.40%  x1.00
   merge readouts       9.86   193    5    10.52%  x0.99
   no safe resets       8.32    97    5     8.89%  x1.17
   safe + merge         8.32    97    5     8.86%  x1.17
   no resets            6.01    25    5     5.59%  x1.86
   no resets + merge    6.01    25    5     5.45%  x1.91
   (5s)

FakeKingston X d=3
   variant        us/rnd resets dist  eps/rnd  gain
   baseline             9.81    63    3     6.08%  x1.00
   merge readouts       9.8

In [22]:
_le_orig = logical_error
def logical_error(circ, shots, seed=1):
    try:
        return _le_orig(circ, shots, seed)
    except ValueError:
        return float("nan")   # decoder cannot handle this graph
print("patched: undecodable variants now show nan")

patched: undecodable variants now show nan


In [24]:
# H10c: physics audit on LIVE Kingston calibration (no job, free)
import statistics as st_
from qiskit_ibm_runtime import QiskitRuntimeService
from google.colab import userdata  # IBM key comes from Colab Secrets, never from the notebook
API_KEY = userdata.get("IBM_QUANTUM_KEY")
service = QiskitRuntimeService(
    channel="ibm_quantum_platform", token=API_KEY)
live = service.backend("ibm_kingston")
noise = ChipNoise(live)
tg = live.target

def med_dur(name):
    v = [p.duration for p in tg[name].values()
         if p is not None and p.duration]
    return st_.median(v) * 1e6 if v else float("nan")

def med_err(name):
    v = [p.error for p in tg[name].values()
         if p is not None and p.error is not None]
    return st_.median(v) if v else float("nan")

nq = live.num_qubits
print("LIVE", live.name, "calibration (medians over the chip)")
print("  T1 %.0f us   T2(echo) %.0f us"
      % (st_.median(noise.t1(q) for q in range(nq)) * 1e6,
         st_.median(noise.t2(q) for q in range(nq)) * 1e6))
print("  readout error %.2f%%   CZ error %.2f%%   SX error %.3f%%"
      % (med_err("measure") * 100, med_err("cz") * 100,
         med_err("sx") * 100))
print("  durations: measure %.2f us  reset %.2f us  CZ %.3f us"
      "  X %.3f us" % (med_dur("measure"), med_dur("reset"),
                       med_dur("cz"), med_dur("x")))

# readout error model: two physical limits
#  "flip":   the qubit itself flips, record follows
#  "record": only the record is wrong, qubit stays
# real IBM readout is a mix -> truth lies between the two
def isa_to_stim3(isa, noise, dets, obs, ro_mode="flip"):
    c = isa_to_stim2(isa, noise, dets, obs, noisy=True)
    if ro_mode == "flip":
        return c
    out = stim.Circuit()
    items = list(c)
    i = 0
    while i < len(items):
        ins = items[i]
        nxt = items[i + 1] if i + 1 < len(items) else None
        if (ins.name == "X_ERROR" and nxt is not None
                and nxt.name == "M"
                and ins.targets_copy() == nxt.targets_copy()):
            p = ins.gate_args_copy()[0]
            out.append("M", nxt.targets_copy(), p)
            i += 2
            continue
        out.append(ins)
        i += 1
    return out

def safe_le(circ, shots):
    try:
        return logical_error(circ, shots, seed=7)
    except ValueError:
        return float("nan")

VARS3 = [("baseline", {}),
         ("safe + merge", dict(merge=True, skip_safe=True)),
         ("no resets + merge",
          dict(merge=True, skip_safe=True, skip_flags=True))]
LIVE_COMP = {}
for basis in ("Z", "X"):
    for d in (3, 5):
        print()
        print("LIVE Kingston", basis, "d=%d" % d)
        print("   variant          us/rnd  flip-model"
              "  record-model")
        lay = None
        for name, kw in VARS3:
            qc, dets, fl, obs, S, bad = make_variant(
                d, HH_ROUNDS, basis, **kw)
            pm = generate_preset_pass_manager(
                optimization_level=1, target=tg, seed_transpiler=1,
                scheduling_method="alap", initial_layout=lay)
            isa = pm.run(qc)
            if lay is None:
                lay = list(isa.layout.initial_index_layout(
                    filter_ancillas=True))
            fired, obf = check_dets(isa, noise, dets, obs)
            assert not fired.any() and not obf, name + " not clean"
            ff, _ = check_dets(isa, noise, fl, obs)
            good = [f for f, b_ in zip(fl, ff) if not b_]
            shots = 60000 if d == 3 else 20000
            row = {}
            for mode in ("flip", "record"):
                circ = isa_to_stim3(isa, noise,
                                    dets + good, obs, mode)
                row[mode] = per_round(safe_le(circ, shots), HH_ROUNDS)
            us = duration_s(isa, live) / HH_ROUNDS * 1e6
            LIVE_COMP[(basis, d, name)] = row
            b0 = LIVE_COMP[(basis, d, "baseline")]
            print("   %-18s %6.2f  %5.2f%% x%.2f  %5.2f%% x%.2f"
                  % (name, us, row["flip"] * 100,
                     b0["flip"] / max(row["flip"], 1e-9),
                     row["record"] * 100,
                     b0["record"] / max(row["record"], 1e-9)))

qiskit_runtime_service._discover_account:WARNING:2026-10-14 16:46:28,436: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-10-14 16:46:31,512: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-10-14 16:46:31,514: Using instance: open-instance, plan: open


LIVE ibm_kingston calibration (medians over the chip)
  T1 243 us   T2(echo) 131 us
  readout error 1.05%   CZ error 0.21%   SX error 0.026%
  durations: measure 2.18 us  reset 2.21 us  CZ 0.068 us  X 0.032 us

LIVE Kingston Z d=3
   variant          us/rnd  flip-model  record-model
   baseline             9.41   4.99% x1.00   5.06% x1.00
   safe + merge         7.93   4.06% x1.23   4.36% x1.16
   no resets + merge    5.72   2.48% x2.01   2.70% x1.87

LIVE Kingston Z d=5
   variant          us/rnd  flip-model  record-model
   baseline             9.46  11.84% x1.00  11.93% x1.00
   safe + merge         7.99  10.30% x1.15  10.85% x1.10
   no resets + merge    5.77   6.90% x1.71   7.21% x1.66

LIVE Kingston X d=3
   variant          us/rnd  flip-model  record-model
   baseline             9.41   4.66% x1.00   4.65% x1.00
   safe + merge         7.19   3.12% x1.50   3.34% x1.40
   no resets + merge    5.69   6.90% x0.68   7.28% x0.64

LIVE Kingston X d=5
   variant          us/rnd  flip-m

In [25]:
import json, os, glob
from qiskit_ibm_runtime import SamplerV2
hw = service.backend("ibm_kingston")
noise = ChipNoise(hw)
print(hw.name, hw.status().pending_jobs, "jobs in queue")
COMPRESS = {"Z": dict(merge=True, skip_safe=True, skip_flags=True),
            "X": dict(merge=True, skip_safe=True)}

# readout = half qubit-flip, half misread (truth is a mix)
def isa_to_stim3(isa, noise, dets, obs, ro_mode="flip"):
    c = isa_to_stim2(isa, noise, dets, obs, noisy=True)
    if ro_mode == "flip":
        return c
    out, items, i = stim.Circuit(), list(c), 0
    while i < len(items):
        ins = items[i]
        nxt = items[i + 1] if i + 1 < len(items) else None
        if (ins.name == "X_ERROR" and nxt is not None
                and nxt.name == "M"
                and ins.targets_copy() == nxt.targets_copy()):
            p = ins.gate_args_copy()[0]
            t = nxt.targets_copy()
            if ro_mode == "mix":
                out.append("X_ERROR", t, p / 2)
                out.append("M", t, p / 2)
            else:
                out.append("M", t, p)
            i += 2
            continue
        out.append(ins)
        i += 1
    return out

def tour_layout(basis):
    if "H7" in globals():
        src = H7
    else:
        src = json.load(open(glob.glob("hh_hw7_*.json")[0]))
    return next(a["layout"] for a in src["arms"]
                if a["basis"] == basis
                and a["layout_name"] == "tournament")

H11 = {"backend": hw.name, "shots": 5000, "arms": []}
pubs = []
for basis in ("Z", "X"):
    L = tour_layout(basis)
    print()
    print(basis, "basis, placement", L[:5], "...")
    for vname in ("baseline", "compressed"):
        kw = COMPRESS[basis] if vname == "compressed" else {}
        pf, pr = [], []
        for R in (1, 2, 3, 4):
            qc, dets, fl, obs, S, bad = make_variant(
                3, R, basis, **kw)
            assert bad == 0, "detectors lost"
            routed = route(qc, L)
            ncz = routed.count_ops().get("cz", 0)
            assert ncz == qc.count_ops()["cx"], "SWAPs appeared"
            isa, npul = add_dd(sched(routed), "xx")
            assert npul > 0, "no DD pulses"
            alld = with_flags(isa, dets, fl, obs)
            B = clean_samples(qc, 8)
            ref_d = [int(B[0, x].sum() % 2) for x in alld]
            ref_o = int(B[0, obs[0]].sum() % 2)
            for x in alld:
                col = B[:, x].sum(1) % 2
                assert col.min() == col.max(), "detector not fixed"
            cf = isa_to_stim3(isa, noise, alld, obs, "flip")
            cr = isa_to_stim3(isa, noise, alld, obs, "record")
            cm = isa_to_stim3(isa, noise, alld, obs, "mix")
            pf.append(logical_error(cf, 30000, seed=9))
            pr.append(logical_error(cr, 30000, seed=9))
            H11["arms"].append({
                "basis": basis, "variant": vname, "rounds": R,
                "dets": alld, "obs": obs[0], "ref_d": ref_d,
                "ref_o": ref_o, "stim": str(cm),
                "us": duration_s(isa, hw) * 1e6})
            pubs.append(isa)
        ef = slope_fit((1, 2, 3, 4), pf)[0]
        er = slope_fit((1, 2, 3, 4), pr)[0]
        H11["pred_" + basis + "_" + vname] = (ef, er)
        print("  %-10s model eps %.2f%% (flip) .. %.2f%% (misread)"
              % (vname, ef * 100, er * 100))

print()
print("PRE-REGISTERED before the job runs:")
print("  P7: compressed eps < baseline eps in BOTH bases")
print("  P8: Z gain >= x1.3 and X gain >= x1.2 on hardware")
print("      (model says Z x1.9-2.0, X x1.4-1.5; model has been")
print("       15-30% optimistic before, so we ask for less)")
job = SamplerV2(mode=hw).run(pubs, shots=H11["shots"])
H11["job_id"] = job.job_id()
fname = "hh_hw11_" + H11["job_id"] + ".json"
json.dump(H11, open(fname, "w"))
dr = "/content/drive/MyDrive"
if os.path.isdir(dr):
    os.makedirs(dr + "/qmem_hw", exist_ok=True)
    json.dump(H11, open(dr + "/qmem_hw/" + fname, "w"))
print("SUBMITTED job", H11["job_id"], "-", len(pubs), "circuits")

qiskit_runtime_service.backends:WARNING:2026-10-14 16:48:25,205: Using instance: open-instance, plan: open


ibm_kingston 10 jobs in queue

Z basis, placement [58, 68, 77, 50, 57] ...
  baseline   model eps 5.92% (flip) .. 5.98% (misread)
  compressed model eps 2.61% (flip) .. 2.94% (misread)

X basis, placement [42, 37, 28, 36, 24] ...
  baseline   model eps 5.85% (flip) .. 5.93% (misread)
  compressed model eps 3.96% (flip) .. 4.10% (misread)

PRE-REGISTERED before the job runs:
  P7: compressed eps < baseline eps in BOTH bases
  P8: Z gain >= x1.3 and X gain >= x1.2 on hardware
      (model says Z x1.9-2.0, X x1.4-1.5; model has been
       15-30% optimistic before, so we ask for less)


/tmp/ipykernel_707/2479375727.py:91: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  job = SamplerV2(mode=hw).run(pubs, shots=H11["shots"])


SUBMITTED job db7r4per6oqs73885d10 - 16 circuits


In [26]:
import json, glob, stim, pymatching, numpy as np
from qiskit_ibm_runtime import QiskitRuntimeService
JOB11 = None   # None = job from H11, or paste the job id
from google.colab import userdata  # IBM key comes from Colab Secrets, never from the notebook
API_KEY = userdata.get("IBM_QUANTUM_KEY")
service = QiskitRuntimeService(
    channel="ibm_quantum_platform", token=API_KEY)
if JOB11 is None:
    JOB11 = H11["job_id"]
fn = "hh_hw11_" + JOB11 + ".json"
DRV = "/content/drive/MyDrive/qmem_hw/"
A = json.load(open((glob.glob(fn) + glob.glob(DRV + fn))[0]))
job = service.job(JOB11)
print("status:", job.status(), "- waiting for results...")
res = job.result()

def slope_fit(rs, ps, N=None):
    rs = np.array(rs, float)
    ps = np.array(ps, float)
    F = np.clip(1 - 2 * ps, 1e-3, None)
    y = np.log(F)
    if N:
        sp = np.sqrt(np.clip(ps * (1 - ps), 1e-9, None) / N)
        w = (F / (2 * sp)) ** 2
    else:
        w = np.ones_like(y)
    X = np.vstack([np.ones_like(rs), rs]).T
    cov = np.linalg.inv(X.T @ (w[:, None] * X))
    beta = cov @ X.T @ (w * y)
    se = np.sqrt(cov[1, 1]) if N else 0.0
    f = lambda s: (1 - np.exp(s)) / 2
    return f(beta[1]), f(beta[1] + 2 * se), f(beta[1] - 2 * se)

G = {}
for k, arm in enumerate(A["arms"]):
    bs = res[k].data.c.get_bitstrings()
    raw = np.frombuffer("".join(bs).encode(), np.uint8)
    arr = (raw.reshape(len(bs), -1) - 48)[:, ::-1].astype(bool)
    det = np.stack([np.logical_xor.reduce(arr[:, d], axis=1)
                    for d in arm["dets"]], axis=1)
    det ^= np.array(arm["ref_d"], bool)[None, :]
    ob = np.logical_xor.reduce(arr[:, arm["obs"]], axis=1)
    ob ^= bool(arm["ref_o"])
    dem = stim.Circuit(arm["stim"]).detector_error_model(
        decompose_errors=True,
        ignore_decomposition_failures=True,
        approximate_disjoint_errors=True)
    m = pymatching.Matching.from_detector_error_model(dem)
    pred = m.decode_batch(det)[:, 0].astype(bool)
    key = (arm["basis"], arm["variant"])
    G.setdefault(key, []).append(
        (arm["rounds"], float(np.mean(pred != ob)),
         float(det.mean()), arm["us"]))

N = A["shots"]
EPS = {}
print()
print(A["backend"], "d=3, DD on, same placement,", N, "shots")
for key, rows in G.items():
    rows.sort()
    e, lo, hi = slope_fit([r[0] for r in rows],
                          [r[1] for r in rows], N)
    EPS[key] = (e, lo, hi)
    pf, pr = A["pred_" + key[0] + "_" + key[1]]
    print(" ", key[0], key[1])
    print("    hw pL r=1..4:",
          " ".join("%.1f%%" % (r[1] * 100) for r in rows))
    print("    hw eps %.2f%% [%.2f, %.2f]   model %.2f..%.2f%%"
          % (e * 100, lo * 100, hi * 100, pf * 100, pr * 100))
    print("    circuit time r=4: %.1f us   det rate r=3: %.3f"
          % (rows[3][3], rows[2][2]))

print()
print("VERDICTS")
ok7, ok8 = True, True
for b, need in (("Z", 1.3), ("X", 1.2)):
    eb, lb, hb = EPS[(b, "baseline")]
    ec, lc, hc = EPS[(b, "compressed")]
    g = eb / max(ec, 1e-9)
    sep = hc < lb
    ok7 &= ec < eb
    ok8 &= g >= need
    print("  %s: baseline %.2f%% -> compressed %.2f%%  gain x%.2f"
          % (b, eb * 100, ec * 100, g),
          "| 2-sigma separated" if sep else "| overlapping")
print("  P7", "HOLDS" if ok7 else "FAILS",
      "| P8", "HOLDS" if ok8 else "FAILS")

qiskit_runtime_service._discover_account:WARNING:2026-10-14 16:48:55,102: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-10-14 16:48:57,604: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.


status: QUEUED - waiting for results...

ibm_kingston d=3, DD on, same placement, 5000 shots
  Z baseline
    hw pL r=1..4: 4.1% 13.5% 22.0% 27.2%
    hw eps 10.56% [10.01, 11.10]   model 5.92..5.98%
    circuit time r=4: 37.6 us   det rate r=3: 0.134
  Z compressed
    hw pL r=1..4: 2.9% 8.6% 13.6% 18.6%
    hw eps 6.20% [5.79, 6.61]   model 2.61..2.94%
    circuit time r=4: 22.1 us   det rate r=3: 0.117
  X baseline
    hw pL r=1..4: 5.4% 11.8% 19.4% 24.9%
    hw eps 8.47% [7.93, 9.01]   model 5.85..5.93%
    circuit time r=4: 37.6 us   det rate r=3: 0.115
  X compressed
    hw pL r=1..4: 4.9% 9.2% 15.0% 19.5%
    hw eps 5.95% [5.48, 6.42]   model 3.96..4.10%
    circuit time r=4: 28.8 us   det rate r=3: 0.104

VERDICTS
  Z: baseline 10.56% -> compressed 6.20%  gain x1.70 | 2-sigma separated
  X: baseline 8.47% -> compressed 5.95%  gain x1.42 | 2-sigma separated
  P7 HOLDS | P8 HOLDS


In [27]:
class Pat:                      # lets make_variant take a pattern
    def __init__(self, s): self.s = s
    def __rmul__(self, other): return self.s

PATTERNS = ["xz", "xxz", "xzz", "xxxz"]
REPS = (2, 4)
PAT = {}
for pat in PATTERNS:
    row = {}
    for basis in ("Z", "X"):
        L = tour_layout(basis)
        F, T = {}, {}
        for k in REPS:
            qc, dets, fl, obs, S, bad = make_variant(
                3, Pat(pat * k), basis, **COMPRESS[basis])
            assert bad == 0, "detectors lost"
            routed = route(qc, L)
            ncz = routed.count_ops().get("cz", 0)
            assert ncz == qc.count_ops()["cx"], "SWAPs"
            isa, _ = add_dd(sched(routed), "xx")
            alld = with_flags(isa, dets, fl, obs)
            circ = isa_to_stim3(isa, noise, alld, obs, "mix")
            p = logical_error(circ, 40000, seed=11)
            F[k] = max(1 - 2 * p, 1e-3)
            T[k] = duration_s(isa, hw) * 1e6
        ratio = F[REPS[1]] / F[REPS[0]]
        dt = T[REPS[1]] - T[REPS[0]]
        e10 = (1 - ratio ** (10 / dt)) / 2
        row[basis] = (e10, dt / (len(pat) * (REPS[1] - REPS[0])))
    PAT[pat] = row
    worst = max(row["Z"][0], row["X"][0])
    print("%-5s Z %.2f%%  X %.2f%%  per 10us | worst %.2f%%"
          " | %.2f us per check" % (pat, row["Z"][0] * 100,
          row["X"][0] * 100, worst * 100, row["Z"][1]))
w0 = max(PAT["xz"]["Z"][0], PAT["xz"]["X"][0])
best = min(PAT, key=lambda s: max(PAT[s]["Z"][0], PAT[s]["X"][0]))
wb = max(PAT[best]["Z"][0], PAT[best]["X"][0])
print()
print("best pattern:", best, "| worst-basis gain vs xz x%.2f"
      % (w0 / max(wb, 1e-9)))

xz    Z 5.81%  X 5.69%  per 10us | worst 5.81% | 2.48 us per check
xxz   Z 7.91%  X 3.78%  per 10us | worst 7.91% | 2.45 us per check
xzz   Z 3.56%  X 9.20%  per 10us | worst 9.20% | 2.54 us per check
xxxz  Z 8.42%  X 3.13%  per 10us | worst 8.42% | 2.43 us per check

best pattern: xz | worst-basis gain vs xz x1.00


In [28]:
import json, glob, stim, pymatching, numpy as np
from qiskit_ibm_runtime import QiskitRuntimeService
JOB = "db7r4per6oqs73885d10"
from google.colab import userdata  # IBM key comes from Colab Secrets, never from the notebook
API_KEY = userdata.get("IBM_QUANTUM_KEY")
service = QiskitRuntimeService(
    channel="ibm_quantum_platform", token=API_KEY)
fn = "hh_hw11_" + JOB + ".json"
DRV = "/content/drive/MyDrive/qmem_hw/"
A14 = json.load(open((glob.glob(fn) + glob.glob(DRV + fn))[0]))
r14 = service.job(JOB).result()
KEEP = (1.0, 0.9, 0.75, 0.5)
H = {}
for k, arm in enumerate(A14["arms"]):
    bs = r14[k].data.c.get_bitstrings()
    raw = np.frombuffer("".join(bs).encode(), np.uint8)
    arr = (raw.reshape(len(bs), -1) - 48)[:, ::-1].astype(bool)
    det = np.stack([np.logical_xor.reduce(arr[:, d], axis=1)
                    for d in arm["dets"]], axis=1)
    det ^= np.array(arm["ref_d"], bool)[None, :]
    ob = np.logical_xor.reduce(arr[:, arm["obs"]], axis=1)
    ob ^= bool(arm["ref_o"])
    dem = stim.Circuit(arm["stim"]).detector_error_model(
        decompose_errors=True,
        ignore_decomposition_failures=True,
        approximate_disjoint_errors=True)
    m = pymatching.Matching.from_detector_error_model(dem)
    pred, w = m.decode_batch(det, return_weights=True)
    wrong = pred[:, 0].astype(bool) != ob
    out = {}
    for keep in KEEP:
        thr = np.quantile(w, keep) if keep < 1 else np.inf
        acc = w <= thr
        out[keep] = (float(wrong[acc].mean()), float(acc.mean()))
    H[(arm["basis"], arm["variant"], arm["rounds"])] = out

print("logical error after 4 rounds (actual % of reads kept)")
heads = ["keep %d%%" % (k * 100) for k in KEEP]
print("               ", " ".join("%12s" % h for h in heads))
for b in ("Z", "X"):
    for v in ("baseline", "compressed"):
        o = H[(b, v, 4)]
        cells = ["%5.2f%% (%2d)" % (o[k][0] * 100, o[k][1] * 100)
                 for k in KEEP]
        print("  %s %-11s" % (b, v), " ".join(cells))
print()
c = H[("Z", "compressed", 4)]
print("Z compressed, 4 rounds: flagging 10%% of reads cuts errors"
      " x%.2f; flagging 25%% cuts x%.2f"
      % (c[1.0][0] / max(c[0.9][0], 1e-9),
         c[1.0][0] / max(c[0.75][0], 1e-9)))

qiskit_runtime_service._discover_account:WARNING:2026-10-14 16:59:08,766: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-10-14 16:59:11,845: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.


logical error after 4 rounds (actual % of reads kept)
                   keep 100%     keep 90%     keep 75%     keep 50%
  Z baseline    27.22% (100) 25.82% (90) 23.73% (75) 19.24% (50)
  Z compressed  18.58% (100) 17.02% (90) 14.40% (75) 10.92% (50)
  X baseline    24.88% (100) 23.29% (90) 20.27% (75) 16.20% (50)
  X compressed  19.52% (100) 17.16% (90) 14.75% (75)  9.24% (50)

Z compressed, 4 rounds: flagging 10% of reads cuts errors x1.09; flagging 25% cuts x1.29


In [30]:
import json, glob, stim, pymatching, numpy as np

def clean_dem(dem):
    out, skipped, total = stim.DetectorErrorModel(), 0, 0
    for ins in dem.flattened():
        if ins.type == "error":
            total += 1
            comps, cur = [], 0
            for t in ins.targets_copy():
                if t.is_separator():
                    comps.append(cur)
                    cur = 0
                elif t.is_relative_detector_id():
                    cur += 1
            comps.append(cur)
            if max(comps) > 2:
                skipped += 1
                continue
        out.append(ins)
    n = dem.num_detectors
    out += stim.DetectorErrorModel("detector D%d" % (n - 1))
    return out, skipped, total

JOB = "db7r4per6oqs73885d10"
fn = "hh_hw11_" + JOB + ".json"
DRV = "/content/drive/MyDrive/qmem_hw/"
A15 = json.load(open((glob.glob(fn) + glob.glob(DRV + fn))[0]))
r15 = service.job(JOB).result()
G15, SK = {}, [0, 0]
for k, arm in enumerate(A15["arms"]):
    bs = r15[k].data.c.get_bitstrings()
    raw = np.frombuffer("".join(bs).encode(), np.uint8)
    arr = (raw.reshape(len(bs), -1) - 48)[:, ::-1].astype(bool)
    det = np.stack([np.logical_xor.reduce(arr[:, d], axis=1)
                    for d in arm["dets"]], axis=1)
    det ^= np.array(arm["ref_d"], bool)[None, :]
    ob = np.logical_xor.reduce(arr[:, arm["obs"]], axis=1)
    ob ^= bool(arm["ref_o"])
    dem = stim.Circuit(arm["stim"]).detector_error_model(
        decompose_errors=True,
        ignore_decomposition_failures=True,
        approximate_disjoint_errors=True)
    m1 = pymatching.Matching.from_detector_error_model(dem)
    p1 = m1.decode_batch(det)[:, 0].astype(bool)
    dem2, sk, tot = clean_dem(dem)
    SK[0] += sk
    SK[1] += tot
    m2 = pymatching.Matching.from_detector_error_model(
        dem2, enable_correlations=True)
    p2 = m2.decode_batch(det, enable_correlations=True)
    p2 = p2[:, 0].astype(bool)
    key = (arm["basis"], arm["variant"])
    G15.setdefault(key, []).append(
        (arm["rounds"], float(np.mean(p1 != ob)),
         float(np.mean(p2 != ob))))

print("error terms dropped for correlated decoder: %d of %d"
      % (SK[0], SK[1]))
N = A15["shots"]
print()
print("eps per round, Kingston compression data")
print("                     plain matching   correlated")
for key, rows in G15.items():
    rows.sort()
    rs = [r[0] for r in rows]
    a = slope_fit(rs, [r[1] for r in rows], N)
    b = slope_fit(rs, [r[2] for r in rows], N)
    gain = a[0] / max(b[0], 1e-9)
    print("  %s %-11s     %5.2f%%          %5.2f%% [%.2f, %.2f]"
          "  x%.2f" % (key[0], key[1], a[0] * 100, b[0] * 100,
                       b[1] * 100, b[2] * 100, gain))

error terms dropped for correlated decoder: 2 of 6110

eps per round, Kingston compression data
                     plain matching   correlated
  Z baseline        10.56%          10.17% [9.64, 10.71]  x1.04
  Z compressed       6.20%           5.84% [5.44, 6.24]  x1.06
  X baseline         8.47%           8.55% [8.01, 9.09]  x0.99
  X compressed       5.95%           5.87% [5.41, 6.34]  x1.01


In [31]:
import json, os, glob
from qiskit_ibm_runtime import SamplerV2
hw = service.backend("ibm_marrakesh")
noise = ChipNoise(hw)
print(hw.name, hw.status().pending_jobs, "jobs in queue")
g = rx.PyGraph()
g.add_nodes_from(range(hw.num_qubits))
E = {tuple(sorted(e)) for e in hw.coupling_map.get_edges()}
g.add_edges_from_no_data(list(E))
COLOR = rx.two_color(g)
XD.clear()
t_ = hw.target
ALIGN = max(getattr(t_, "granularity", 16) or 16,
            getattr(t_, "pulse_alignment", 16) or 16)

def mk_layout(basis):      # placement fixed in the earlier run
    src = HW if "HW" in globals() else json.load(
        open(glob.glob("hh_hw_db7qkveil99s73foma1g.json")[0]))
    return next(a["layout"] for a in src["arms"]
                if a["basis"] == basis
                and a["layout_name"] == "tournament")

R16 = {"backend": hw.name, "shots": 5000, "arms": []}
pubs, preds = [], {}
for basis in ("X", "Z"):
    L = mk_layout(basis)
    for R in (1, 2, 3, 4):
        for vname in ("compressed", "baseline"):
            kw = COMPRESS[basis] if vname == "compressed" else {}
            qc, dets, fl, obs, S, bad = make_variant(
                3, R, basis, **kw)
            assert bad == 0, "detectors lost"
            routed = route(qc, L)
            ncz = routed.count_ops().get("cz", 0)
            assert ncz == qc.count_ops()["cx"], "SWAPs appeared"
            isa, npul = add_dd(sched(routed), "xx")
            assert npul > 0, "no DD pulses"
            alld = with_flags(isa, dets, fl, obs)
            B = clean_samples(qc, 8)
            ref_d = [int(B[0, x].sum() % 2) for x in alld]
            ref_o = int(B[0, obs[0]].sum() % 2)
            cf = isa_to_stim3(isa, noise, alld, obs, "flip")
            cr = isa_to_stim3(isa, noise, alld, obs, "record")
            cm = isa_to_stim3(isa, noise, alld, obs, "mix")
            kk = (basis, vname)
            preds.setdefault(kk, ([], []))
            preds[kk][0].append(logical_error(cf, 30000, seed=9))
            preds[kk][1].append(logical_error(cr, 30000, seed=9))
            R16["arms"].append({
                "basis": basis, "variant": vname, "rounds": R,
                "dets": alld, "obs": obs[0], "ref_d": ref_d,
                "ref_o": ref_o, "stim": str(cm),
                "us": duration_s(isa, hw) * 1e6})
            pubs.append(isa)
for (basis, vname), (pf, pr) in preds.items():
    ef = slope_fit((1, 2, 3, 4), pf)[0]
    er = slope_fit((1, 2, 3, 4), pr)[0]
    R16["pred_" + basis + "_" + vname] = (ef, er)
    print("  %s %-10s model eps %.2f%% .. %.2f%%"
          % (basis, vname, ef * 100, er * 100))
print()
print("PRE-REGISTERED: compressed beats baseline in both bases")
print("on a second chip, with order reversed and interleaved.")
job = SamplerV2(mode=hw).run(pubs, shots=R16["shots"])
R16["job_id"] = job.job_id()
fname = "hh_hw11_" + R16["job_id"] + ".json"
json.dump(R16, open(fname, "w"))
dr = "/content/drive/MyDrive"
if os.path.isdir(dr):
    os.makedirs(dr + "/qmem_hw", exist_ok=True)
    json.dump(R16, open(dr + "/qmem_hw/" + fname, "w"))
print("SUBMITTED job", R16["job_id"], "-", len(pubs), "circuits")
print("decode it with cell H12: set JOB11 = '%s'" % R16["job_id"])

qiskit_runtime_service.backends:WARNING:2026-10-14 17:03:10,660: Using instance: open-instance, plan: open


ibm_marrakesh 4 jobs in queue
  X compressed model eps 13.72% .. 14.42%
  X baseline   model eps 18.22% .. 18.25%
  Z compressed model eps 13.89% .. 14.78%
  Z baseline   model eps 19.31% .. 19.83%

PRE-REGISTERED: compressed beats baseline in both bases
on a second chip, with order reversed and interleaved.


/tmp/ipykernel_707/3052759517.py:64: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  job = SamplerV2(mode=hw).run(pubs, shots=R16["shots"])


SUBMITTED job db7rbner6oqs73885t8g - 16 circuits
decode it with cell H12: set JOB11 = 'db7rbner6oqs73885t8g'


In [32]:
# H17: decode the Marrakesh replication (self-contained)
import json, glob, stim, pymatching, numpy as np
from qiskit_ibm_runtime import QiskitRuntimeService
JOB17 = "db7rbner6oqs73885t8g"
from google.colab import userdata  # IBM key comes from Colab Secrets, never from the notebook
API_KEY = userdata.get("IBM_QUANTUM_KEY")
service = QiskitRuntimeService(
    channel="ibm_quantum_platform", token=API_KEY)
fn = "hh_hw11_" + JOB17 + ".json"
DRV = "/content/drive/MyDrive/qmem_hw/"
A17 = json.load(open((glob.glob(fn) + glob.glob(DRV + fn))[0]))
job = service.job(JOB17)
print("status:", job.status(), "- waiting for results...")
res17 = job.result()

def slope_fit(rs, ps, N=None):
    rs = np.array(rs, float)
    ps = np.array(ps, float)
    F = np.clip(1 - 2 * ps, 1e-3, None)
    y = np.log(F)
    if N:
        sp = np.sqrt(np.clip(ps * (1 - ps), 1e-9, None) / N)
        w = (F / (2 * sp)) ** 2
    else:
        w = np.ones_like(y)
    X = np.vstack([np.ones_like(rs), rs]).T
    cov = np.linalg.inv(X.T @ (w[:, None] * X))
    beta = cov @ X.T @ (w * y)
    se = np.sqrt(cov[1, 1]) if N else 0.0
    f = lambda s: (1 - np.exp(s)) / 2
    return f(beta[1]), f(beta[1] + 2 * se), f(beta[1] - 2 * se)

def clean_dem(dem):
    out = stim.DetectorErrorModel()
    for ins in dem.flattened():
        if ins.type == "error":
            comps, cur = [], 0
            for t in ins.targets_copy():
                if t.is_separator():
                    comps.append(cur)
                    cur = 0
                elif t.is_relative_detector_id():
                    cur += 1
            comps.append(cur)
            if max(comps) > 2:
                continue
        out.append(ins)
    n = dem.num_detectors
    out += stim.DetectorErrorModel("detector D%d" % (n - 1))
    return out

G17 = {}
for k, arm in enumerate(A17["arms"]):
    bs = res17[k].data.c.get_bitstrings()
    raw = np.frombuffer("".join(bs).encode(), np.uint8)
    arr = (raw.reshape(len(bs), -1) - 48)[:, ::-1].astype(bool)
    det = np.stack([np.logical_xor.reduce(arr[:, d], axis=1)
                    for d in arm["dets"]], axis=1)
    det ^= np.array(arm["ref_d"], bool)[None, :]
    ob = np.logical_xor.reduce(arr[:, arm["obs"]], axis=1)
    ob ^= bool(arm["ref_o"])
    dem = stim.Circuit(arm["stim"]).detector_error_model(
        decompose_errors=True,
        ignore_decomposition_failures=True,
        approximate_disjoint_errors=True)
    m1 = pymatching.Matching.from_detector_error_model(dem)
    p1 = m1.decode_batch(det)[:, 0].astype(bool)
    m2 = pymatching.Matching.from_detector_error_model(
        clean_dem(dem), enable_correlations=True)
    p2 = m2.decode_batch(det, enable_correlations=True)
    p2 = p2[:, 0].astype(bool)
    key = (arm["basis"], arm["variant"])
    G17.setdefault(key, []).append(
        (arm["rounds"], float(np.mean(p1 != ob)),
         float(np.mean(p2 != ob)), float(det.mean()), arm["us"]))

N = A17["shots"]
E17 = {}
print()
print(A17["backend"], "d=3, DD on,", N, "shots, order reversed")
for key in [("X", "compressed"), ("X", "baseline"),
            ("Z", "compressed"), ("Z", "baseline")]:
    rows = sorted(G17[key])
    rs = [r[0] for r in rows]
    a = slope_fit(rs, [r[1] for r in rows], N)
    b = slope_fit(rs, [r[2] for r in rows], N)
    E17[key] = (a, b)
    pf, pr = A17["pred_" + key[0] + "_" + key[1]]
    print(" ", key[0], key[1])
    print("    hw pL r=1..4:",
          " ".join("%.1f%%" % (r[1] * 100) for r in rows))
    print("    eps plain %.2f%% [%.2f, %.2f]  correlated %.2f%%"
          % (a[0] * 100, a[1] * 100, a[2] * 100, b[0] * 100))
    print("    model %.2f..%.2f%%   4-round time %.1f us"
          % (pf * 100, pr * 100, rows[3][4]))

print()
print("VERDICT (plain matching, as pre-registered)")
wins = 0
KING = {"Z": 1.70, "X": 1.42}
for b in ("X", "Z"):
    eb, lb, hb = E17[(b, "baseline")][0]
    ec, lc, hc = E17[(b, "compressed")][0]
    g = eb / max(ec, 1e-9)
    wins += ec < eb
    print("  %s: baseline %.2f%% -> compressed %.2f%%  x%.2f"
          % (b, eb * 100, ec * 100, g),
          "| 2-sigma separated" if hc < lb else "| overlapping",
          "| Kingston was x%.2f" % KING[b])
print("  replication", "HOLDS" if wins == 2 else "FAILS",
      "(%d of 2 bases)" % wins)


qiskit_runtime_service._discover_account:WARNING:2026-10-14 17:06:41,851: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-10-14 17:06:44,988: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.


status: DONE - waiting for results...

ibm_marrakesh d=3, DD on, 5000 shots, order reversed
  X compressed
    hw pL r=1..4: 19.4% 34.7% 42.6% 45.7%
    eps plain 24.87% [23.30, 26.34]  correlated 25.20%
    model 13.72..14.42%   4-round time 35.0 us
  X baseline
    hw pL r=1..4: 27.2% 39.6% 44.9% 46.9%
    eps plain 26.05% [23.85, 28.07]  correlated 26.46%
    model 18.22..18.25%   4-round time 45.9 us
  Z compressed
    hw pL r=1..4: 15.8% 27.1% 32.9% 36.6%
    eps plain 14.16% [13.17, 15.13]  correlated 13.25%
    model 13.89..14.78%   4-round time 26.8 us
  Z baseline
    hw pL r=1..4: 20.2% 29.2% 37.6% 38.8%
    eps plain 15.21% [13.98, 16.39]  correlated 15.43%
    model 19.31..19.83%   4-round time 45.9 us

VERDICT (plain matching, as pre-registered)
  X: baseline 26.05% -> compressed 24.87%  x1.05 | overlapping | Kingston was x1.42
  Z: baseline 15.21% -> compressed 14.16%  x1.07 | overlapping | Kingston was x1.70
  replication HOLDS (2 of 2 bases)


In [33]:
import json, os, glob
from qiskit_ibm_runtime import SamplerV2
hw = service.backend("ibm_kingston")
noise = ChipNoise(hw)
print(hw.name, hw.status().pending_jobs, "jobs in queue")
g = rx.PyGraph()
g.add_nodes_from(range(hw.num_qubits))
E = {tuple(sorted(e)) for e in hw.coupling_map.get_edges()}
g.add_edges_from_no_data(list(E))
COLOR = rx.two_color(g)
XD.clear()
t_ = hw.target
ALIGN = max(getattr(t_, "granularity", 16) or 16,
            getattr(t_, "pulse_alignment", 16) or 16)
ROUNDS18 = (1, 2, 3)

def build_isa(basis, R, kw, L):
    qc, dets, fl, obs, S, bad = make_variant(3, R, basis, **kw)
    assert bad == 0, "detectors lost"
    routed = route(qc, L)
    ncz = routed.count_ops().get("cz", 0)
    assert ncz == qc.count_ops()["cx"], "SWAPs appeared"
    isa, npul = add_dd(sched(routed), "xx")
    assert npul > 0, "no DD pulses"
    alld = with_flags(isa, dets, fl, obs)
    B = clean_samples(qc, 8)
    ref_d = [int(B[0, x].sum() % 2) for x in alld]
    ref_o = int(B[0, obs[0]].sum() % 2)
    circ = isa_to_stim3(isa, noise, alld, obs, "mix")
    return isa, circ, alld, obs, ref_d, ref_o

def new_placement(basis, old):
    kw = COMPRESS[basis]
    qc, dets, fl, obs, S, bad = make_variant(3, 3, basis, **kw)
    cg, uses = code_graph(qc)
    meas, rst = role_counts(qc)
    pool = candidate_layouts(hw, cg, np.random.default_rng(7))

    def score(L):
        s = model_score(L, uses, meas, rst, noise,
                        S["n"], 3 * 4e-6)
        for i, p in enumerate(L):
            s += noise.sx(p) * (40 if i < S["n"] else 20)
        return s
    best = None
    for L in sorted(pool, key=score)[:TOP_K]:
        if L == old:
            continue
        try:
            _, circ, *_ = build_isa(basis, 3, kw, L)
        except Exception:
            continue
        p = logical_error(circ, 20000, seed=11)
        if best is None or p < best[0]:
            best = (p, L)
    _, c_old, *_ = build_isa(basis, 3, kw, old)
    p_old = logical_error(c_old, 20000, seed=11)
    print("  %s: %d placements | pilot 3 rounds: earlier %.1f%%"
          "  new %.1f%%" % (basis, len(pool), p_old * 100,
                            best[0] * 100))
    return best[1]

LAY = {}
for basis in ("X", "Z"):
    old = tour_layout(basis)
    LAY[basis] = {"baseline": ([], old), "compressed":
                  (COMPRESS[basis], old), "compressed+new":
                  (COMPRESS[basis], new_placement(basis, old))}

ORDERS = [("compressed", "baseline", "compressed+new"),
          ("baseline", "compressed+new", "compressed"),
          ("compressed+new", "compressed", "baseline")]
H18 = {"backend": hw.name, "shots": 5000, "arms": []}
pubs, preds = [], {}
for basis in ("X", "Z"):
    for i, R in enumerate(ROUNDS18):
        for v in ORDERS[i % 3]:
            kw, L = LAY[basis][v]
            kw = kw if kw else {}
            isa, circ, alld, obs, ref_d, ref_o = build_isa(
                basis, R, kw, L)
            preds.setdefault((basis, v), {})[R] = \
                logical_error(circ, 30000, seed=9)
            H18["arms"].append({
                "basis": basis, "variant": v, "rounds": R,
                "dets": alld, "obs": obs[0], "ref_d": ref_d,
                "ref_o": ref_o, "stim": str(circ),
                "layout": [int(x) for x in L],
                "us": duration_s(isa, hw) * 1e6})
            pubs.append(isa)
print()
for (basis, v), d_ in sorted(preds.items()):
    e = slope_fit(ROUNDS18, [d_[r] for r in ROUNDS18])[0]
    H18["pred_" + basis + "_" + v] = e
    print("  %s %-15s model eps %.2f%%" % (basis, v, e * 100))
print()
print("PRE-REGISTERED before the job runs:")
print("  P9:  compressed beats baseline in both bases,")
print("       2-sigma separated (Kingston replication)")
print("  P10: compressed+new beats compressed in both bases")
job = SamplerV2(mode=hw).run(pubs, shots=H18["shots"])
H18["job_id"] = job.job_id()
fname = "hh_hw18_" + H18["job_id"] + ".json"
json.dump(H18, open(fname, "w"))
dr = "/content/drive/MyDrive"
if os.path.isdir(dr):
    os.makedirs(dr + "/qmem_hw", exist_ok=True)
    json.dump(H18, open(dr + "/qmem_hw/" + fname, "w"))
print("SUBMITTED job", H18["job_id"], "-", len(pubs), "circuits")

qiskit_runtime_service.backends:WARNING:2026-10-14 17:10:16,911: Using instance: open-instance, plan: open


ibm_kingston 0 jobs in queue
  X: 148 placements | pilot 3 rounds: earlier 9.8%  new 9.4%
  Z: 148 placements | pilot 3 rounds: earlier 7.1%  new 6.9%

  X baseline        model eps 5.75%
  X compressed      model eps 4.07%
  X compressed+new  model eps 3.77%
  Z baseline        model eps 5.86%
  Z compressed      model eps 2.66%
  Z compressed+new  model eps 2.67%

PRE-REGISTERED before the job runs:
  P9:  compressed beats baseline in both bases,
       2-sigma separated (Kingston replication)
  P10: compressed+new beats compressed in both bases


/tmp/ipykernel_707/1618647113.py:101: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  job = SamplerV2(mode=hw).run(pubs, shots=H18["shots"])


SUBMITTED job db7rf1gr1c5c738l5ig0 - 18 circuits


In [34]:
import json, glob, stim, pymatching, numpy as np
from qiskit_ibm_runtime import QiskitRuntimeService
JOB18 = None
from google.colab import userdata  # IBM key comes from Colab Secrets, never from the notebook
API_KEY = userdata.get("IBM_QUANTUM_KEY")
service = QiskitRuntimeService(
    channel="ibm_quantum_platform", token=API_KEY)
if JOB18 is None:
    JOB18 = H18["job_id"]
fn = "hh_hw18_" + JOB18 + ".json"
DRV = "/content/drive/MyDrive/qmem_hw/"
A18 = json.load(open((glob.glob(fn) + glob.glob(DRV + fn))[0]))
job = service.job(JOB18)
print("status:", job.status(), "- waiting for results...")
res18 = job.result()

def slope_fit(rs, ps, N=None):
    rs = np.array(rs, float)
    ps = np.array(ps, float)
    F = np.clip(1 - 2 * ps, 1e-3, None)
    y = np.log(F)
    if N:
        sp = np.sqrt(np.clip(ps * (1 - ps), 1e-9, None) / N)
        w = (F / (2 * sp)) ** 2
    else:
        w = np.ones_like(y)
    X = np.vstack([np.ones_like(rs), rs]).T
    cov = np.linalg.inv(X.T @ (w[:, None] * X))
    beta = cov @ X.T @ (w * y)
    se = np.sqrt(cov[1, 1]) if N else 0.0
    f = lambda s: (1 - np.exp(s)) / 2
    return f(beta[1]), f(beta[1] + 2 * se), f(beta[1] - 2 * se)

def clean_dem(dem):
    out = stim.DetectorErrorModel()
    for ins in dem.flattened():
        if ins.type == "error":
            comps, cur = [], 0
            for t in ins.targets_copy():
                if t.is_separator():
                    comps.append(cur)
                    cur = 0
                elif t.is_relative_detector_id():
                    cur += 1
            comps.append(cur)
            if max(comps) > 2:
                continue
        out.append(ins)
    n = dem.num_detectors
    out += stim.DetectorErrorModel("detector D%d" % (n - 1))
    return out

G = {}
for k, arm in enumerate(A18["arms"]):
    bs = res18[k].data.c.get_bitstrings()
    raw = np.frombuffer("".join(bs).encode(), np.uint8)
    arr = (raw.reshape(len(bs), -1) - 48)[:, ::-1].astype(bool)
    det = np.stack([np.logical_xor.reduce(arr[:, d], axis=1)
                    for d in arm["dets"]], axis=1)
    det ^= np.array(arm["ref_d"], bool)[None, :]
    ob = np.logical_xor.reduce(arr[:, arm["obs"]], axis=1)
    ob ^= bool(arm["ref_o"])
    dem = stim.Circuit(arm["stim"]).detector_error_model(
        decompose_errors=True,
        ignore_decomposition_failures=True,
        approximate_disjoint_errors=True)
    m1 = pymatching.Matching.from_detector_error_model(dem)
    p1 = m1.decode_batch(det)[:, 0].astype(bool)
    m2 = pymatching.Matching.from_detector_error_model(
        clean_dem(dem), enable_correlations=True)
    p2 = m2.decode_batch(det, enable_correlations=True)
    p2 = p2[:, 0].astype(bool)
    G.setdefault((arm["basis"], arm["variant"]), []).append(
        (arm["rounds"], float(np.mean(p1 != ob)),
         float(np.mean(p2 != ob)), arm["us"]))

N = A18["shots"]
E = {}
VARS = ("baseline", "compressed", "compressed+new")
print()
print(A18["backend"], "d=3, DD on,", N, "shots, rounds 1-3")
for b in ("X", "Z"):
    for v in VARS:
        rows = sorted(G[(b, v)])
        rs = [r[0] for r in rows]
        a = slope_fit(rs, [r[1] for r in rows], N)
        c = slope_fit(rs, [r[2] for r in rows], N)
        E[(b, v)] = a
        print("  %s %-15s pL %s" % (b, v, " ".join(
            "%.1f%%" % (r[1] * 100) for r in rows)))
        print("      eps %.2f%% [%.2f, %.2f]  correlated %.2f%%"
              "  model %.2f%%" % (a[0] * 100, a[1] * 100,
              a[2] * 100, c[0] * 100,
              A18["pred_" + b + "_" + v] * 100))

def versus(b, new, old):
    en, ln, hn = E[(b, new)]
    eo, lo, ho = E[(b, old)]
    sep = hn < lo
    return en < eo, sep, eo / max(en, 1e-9)

print()
print("VERDICTS (plain matching)")
ok9, ok10 = True, True
for b in ("X", "Z"):
    w, s, g_ = versus(b, "compressed", "baseline")
    ok9 &= w and s
    print("  P9  %s compressed vs baseline x%.2f %s" % (
        b, g_, "separated" if s else "overlapping"))
for b in ("X", "Z"):
    w, s, g_ = versus(b, "compressed+new", "compressed")
    ok10 &= w
    print("  P10 %s new placement vs earlier x%.2f %s" % (
        b, g_, "separated" if s else "overlapping"))
print("  P9", "HOLDS" if ok9 else "FAILS",
      "| P10", "HOLDS" if ok10 else "FAILS")

qiskit_runtime_service._discover_account:WARNING:2026-10-14 17:10:51,821: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-10-14 17:10:55,807: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.


status: RUNNING - waiting for results...

ibm_kingston d=3, DD on, 5000 shots, rounds 1-3
  X baseline        pL 4.8% 11.4% 18.8%
      eps 8.10% [7.40, 8.79]  correlated 7.99%  model 5.75%
  X compressed      pL 3.4% 9.4% 14.6%
      eps 6.45% [5.85, 7.03]  correlated 5.97%  model 4.07%
  X compressed+new  pL 4.5% 9.3% 14.0%
      eps 5.48% [4.85, 6.09]  correlated 5.40%  model 3.77%
  Z baseline        pL 4.4% 12.4% 22.0%
      eps 10.11% [9.38, 10.84]  correlated 9.69%  model 5.86%
  Z compressed      pL 2.4% 7.8% 13.7%
      eps 6.11% [5.56, 6.64]  correlated 5.43%  model 2.66%
  Z compressed+new  pL 2.1% 8.5% 14.0%
      eps 6.67% [6.13, 7.20]  correlated 5.97%  model 2.67%

VERDICTS (plain matching)
  P9  X compressed vs baseline x1.26 separated
  P9  Z compressed vs baseline x1.66 separated
  P10 X new placement vs earlier x1.18 overlapping
  P10 Z new placement vs earlier x0.92 overlapping
  P9 HOLDS | P10 FAILS


In [35]:
# H20: MEASUREMENT CLEANLINESS on the qubits our code uses
import numpy as np
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import SamplerV2
hw = service.backend("ibm_kingston")
noise = ChipNoise(hw)
ROLE = {}
for basis in ("X", "Z"):
    for i, p in enumerate(tour_layout(basis)):
        r = "data" if i < 9 else "anc"
        ROLE[p] = r if ROLE.get(p, r) == r else "both"
Q = sorted(ROLE)
gap = int(2.2e-6 / hw.dt) // 16 * 16       # one measure time
circs, tags = [], []
for prep in (0, 1):
    for wait in (0, gap):
        qc = QuantumCircuit(hw.num_qubits, 2 * len(Q))
        for j, q in enumerate(Q):
            if prep:
                qc.x(q)
            qc.measure(q, 2 * j)
            if wait:
                qc.delay(wait, q, unit="dt")
            qc.measure(q, 2 * j + 1)
        circs.append(qc)
        tags.append((prep, wait))
print(len(Q), "qubits,", len(circs), "circuits - submitting")
job = SamplerV2(mode=hw).run(circs, shots=4000)
print("job", job.job_id(), "- waiting...")
res = job.result()

M = {}
for k, (prep, wait) in enumerate(tags):
    bs = res[k].data.c.get_bitstrings()
    raw = np.frombuffer("".join(bs).encode(), np.uint8)
    arr = (raw.reshape(len(bs), -1) - 48)[:, ::-1].astype(bool)
    for j, q in enumerate(Q):
        m1, m2 = arr[:, 2 * j], arr[:, 2 * j + 1]
        wrong1 = float(np.mean(m1 != bool(prep)))
        flip = float(np.mean(m1 != m2))
        M[(q, prep, wait)] = (wrong1, flip)

t1_exp = {q: 1 - np.exp(-gap * hw.dt / noise.t1(q)) for q in Q}
print()
print(" qubit role  cal_ro  wrong|0 wrong|1  2nd!=1st|0  "
      "2nd!=1st|1  excess")
EX = {}
for q in Q:
    w0, f0 = M[(q, 0, 0)]
    w1, f1 = M[(q, 1, 0)]
    # clean QND: 2nd!=1st ~ two readout errors (one each)
    ex = 0.5 * ((f0 - 2 * w0) + (f1 - 2 * w1))
    EX[q] = ex
    print(" %4d  %-4s  %5.2f%%  %5.2f%%  %5.2f%%   %6.2f%%"
          "     %6.2f%%  %6.2f%%" % (q, ROLE[q],
          noise.ro(q) * 100, w0 * 100, w1 * 100, f0 * 100,
          f1 * 100, ex * 100))

anc = [q for q in Q if ROLE[q] != "data"]
exs = np.array([EX[q] for q in anc])
model_q = np.median([noise.ro(q) / 2 for q in anc])
print()
print("ancillas: median excess %.2f%%, worst %.2f%% (q%d)"
      % (np.median(exs) * 100, exs.max() * 100,
         anc[int(np.argmax(exs))]))
print("model assumed a state flip of ~%.2f%% per measurement"
      % (model_q * 100))
dirty = [q for q in anc if EX[q] > max(2 * model_q, 0.01)]
print("dirty ancillas (excess > 2x model and > 1%):", dirty)
print()
print("with a 2.2 us wait (state |1>, T1 decay expected):")
for q in anc:
    w1, f1 = M[(q, 1, gap)]
    w1b, f1b = M[(q, 1, 0)]
    extra = f1 - f1b
    if extra > 2 * t1_exp[q] + 0.01:
        print("  q%d: wait adds %.2f%% flips, T1 predicts %.2f%%"
              % (q, extra * 100, t1_exp[q] * 100))
print("(no lines above = waiting behaves as T1 predicts)")
CLEAN = {"job": job.job_id(), "excess": EX, "dirty": dirty}

qiskit_runtime_service.backends:WARNING:2026-10-14 17:13:43,591: Using instance: open-instance, plan: open


43 qubits, 4 circuits - submitting


/tmp/ipykernel_707/3721468823.py:28: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  job = SamplerV2(mode=hw).run(circs, shots=4000)


job db7rgigr1c5c738l5kq0 - waiting...

 qubit role  cal_ro  wrong|0 wrong|1  2nd!=1st|0  2nd!=1st|1  excess
    3  anc    0.78%   0.07%   0.97%     0.07%       2.60%    0.29%
    4  anc    0.83%   0.68%   3.02%     0.60%       5.80%   -0.50%
    5  anc    0.42%   0.35%   1.15%     0.22%       3.28%    0.25%
    6  data   2.00%   0.57%   1.12%     0.57%       3.95%    0.56%
    7  anc    1.09%   1.03%   1.38%     0.90%       2.12%   -0.89%
   16  data   1.17%   2.10%   1.07%     1.30%       2.02%   -1.51%
   17  data   0.49%   0.10%   0.83%     0.15%       2.20%    0.25%
   20  data   0.74%   0.12%   1.10%     0.10%       2.02%   -0.16%
   21  anc    0.39%   2.02%   1.93%     1.52%      13.28%    3.45%
   22  anc    1.94%   0.85%   1.55%     0.73%       3.25%   -0.41%
   23  anc    0.38%   0.07%   0.68%     0.10%       1.70%    0.15%
   24  data   1.68%   1.38%   2.45%     1.40%       3.57%   -1.34%
   25  anc    0.48%   0.20%   0.73%     0.25%       1.62%    0.01%
   26  anc    0.77%  

In [36]:
import json, os, numpy as np
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import SamplerV2
hw = service.backend("ibm_kingston")
cal = ChipNoise(hw)
NQ = hw.num_qubits
W = int(20e-6 / hw.dt) // 32 * 32            # 20 us wait
pm0 = generate_preset_pass_manager(
    optimization_level=1, target=hw.target,
    initial_layout=list(range(NQ)))

# ---- 1) probe: 3 circuits, every qubit in parallel ----
a = QuantumCircuit(NQ, NQ)
a.measure(range(NQ), range(NQ))                      # |0> readout
b = QuantumCircuit(NQ, 2 * NQ)
b.x(range(NQ))
b.measure(range(NQ), range(NQ))                      # |1> readout
b.delay(W, range(NQ), unit="dt")
b.measure(range(NQ), range(NQ, 2 * NQ))              # T1 decay
c = QuantumCircuit(NQ, NQ)
c.h(range(NQ))
c.delay(W // 2, range(NQ), unit="dt")
c.x(range(NQ))
c.delay(W // 2, range(NQ), unit="dt")
c.h(range(NQ))
c.measure(range(NQ), range(NQ))                      # echo T2
job = SamplerV2(mode=hw).run([pm0.run(x) for x in (a, b, c)],
                             shots=3000)
print("probe job", job.job_id(), "- waiting...")
pr = job.result()

def bits(k):
    bs = pr[k].data.c.get_bitstrings()
    raw = np.frombuffer("".join(bs).encode(), np.uint8)
    return (raw.reshape(len(bs), -1) - 48)[:, ::-1].astype(bool)
A_, B_, C_ = bits(0), bits(1), bits(2)
ro0 = A_.mean(0)
ro1 = 1 - B_[:, :NQ].mean(0)
dec = np.array([np.mean(~B_[B_[:, q], NQ + q]) if B_[:, q].any()
                else 0.5 for q in range(NQ)])
ech = C_.mean(0)
LIVE = {}
tw = W * hw.dt
for q in range(NQ):
    ro = 0.5 * (ro0[q] + ro1[q])
    d = min(max(dec[q] - ro1[q], 1e-4), 0.99)
    t1 = -tw / np.log(1 - d)
    e = min(max(ech[q] - ro, 1e-4), 0.49)
    t2 = min(-tw / np.log(1 - 2 * e), 2 * t1)
    LIVE[q] = (t1, t2, ro)

worse = [q for q in range(NQ) if LIVE[q][0] < cal.t1(q) / 2]
print("live median T1 %.0f us (calibration %.0f us)"
      % (np.median([LIVE[q][0] for q in range(NQ)]) * 1e6,
         np.median([cal.t1(q) for q in range(NQ)]) * 1e6))
print("qubits with live T1 < half of calibration:", len(worse))
print("  e.g.", worse[:15])

# ---- 2) live noise model ----
class LiveNoise(ChipNoise):
    def t1(self, q): return LIVE[q][0]
    def t2(self, q): return LIVE[q][1]
    def ro(self, q): return float(min(max(LIVE[q][2], 1e-4), .45))
live = LiveNoise(hw)

# ---- 3) same tournament, fed with calibration OR live data ----
def place(basis, nz):
    kw = COMPRESS[basis]
    qc, dets, fl, obs, S, bad = make_variant(3, 3, basis, **kw)
    cg, uses = code_graph(qc)
    meas, rst = role_counts(qc)
    pool = candidate_layouts(hw, cg, np.random.default_rng(7))

    def score(L):
        s = model_score(L, uses, meas, rst, nz, S["n"], 3 * 4e-6)
        for i, p in enumerate(L):
            s += nz.sx(p) * (40 if i < S["n"] else 20)
        return s
    best = None
    for L in sorted(pool, key=score)[:TOP_K]:
        try:
            routed = route(qc, L)
        except Exception:
            continue
        if routed.count_ops().get("cz", 0) != qc.count_ops()["cx"]:
            continue
        isa, _ = add_dd(sched(routed), "xx")
        fired, obf = check_dets(isa, nz, dets, obs)
        ff, _ = check_dets(isa, nz, fl, obs)
        alld = dets + [f for f, b_ in zip(fl, ff) if not b_]
        circ = isa_to_stim3(isa, nz, alld, obs, "mix")
        p = logical_error(circ, 20000, seed=11)
        if best is None or p < best[0]:
            best = (p, L)
    return best[1]

noise = live                  # everything below judged on live data
H21 = {"backend": hw.name, "shots": 5000, "arms": [],
       "probe_job": job.job_id()}
LAY21 = {}
for basis in ("X", "Z"):
    Lc = place(basis, cal)
    Lp = place(basis, live)
    LAY21[basis] = {"calib-placed": Lc, "probe-placed": Lp}
    print(basis, "same placement" if Lc == Lp else
          "different placements",
          "| bad qubits in calib-placed:",
          sorted(set(Lc) & set(worse)),
          "| in probe-placed:", sorted(set(Lp) & set(worse)))

# ---- 4) build the hardware job ----
ORD = [("calib-placed", "probe-placed"),
       ("probe-placed", "calib-placed")]
pubs, preds = [], {}
for basis in ("X", "Z"):
    for i, R in enumerate((1, 2, 3)):
        for v in ORD[i % 2]:
            isa, circ, alld, obs, ref_d, ref_o = build_isa(
                basis, R, COMPRESS[basis], LAY21[basis][v])
            preds.setdefault((basis, v), {})[R] = \
                logical_error(circ, 30000, seed=9)
            H21["arms"].append({
                "basis": basis, "variant": v, "rounds": R,
                "dets": alld, "obs": obs[0], "ref_d": ref_d,
                "ref_o": ref_o, "stim": str(circ),
                "layout": [int(x) for x in LAY21[basis][v]],
                "us": duration_s(isa, hw) * 1e6})
            pubs.append(isa)
print()
for (basis, v), d_ in sorted(preds.items()):
    e = slope_fit((1, 2, 3), [d_[r] for r in (1, 2, 3)])[0]
    H21["pred_" + basis + "_" + v] = e
    print("  %s %-13s live-model eps %.2f%%" % (basis, v, e * 100))
print()
print("PRE-REGISTERED: P11 probe-placed beats calib-placed")
print("in both bases (only the data source differs)")
job2 = SamplerV2(mode=hw).run(pubs, shots=H21["shots"])
H21["job_id"] = job2.job_id()
fname = "hh_hw18_" + H21["job_id"] + ".json"
json.dump(H21, open(fname, "w"))
dr = "/content/drive/MyDrive"
if os.path.isdir(dr):
    os.makedirs(dr + "/qmem_hw", exist_ok=True)
    json.dump(H21, open(dr + "/qmem_hw/" + fname, "w"))
print("SUBMITTED job", H21["job_id"], "-", len(pubs), "circuits")

qiskit_runtime_service.backends:WARNING:2026-10-14 17:17:03,779: Using instance: open-instance, plan: open
/tmp/ipykernel_707/4056523310.py:27: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  job = SamplerV2(mode=hw).run([pm0.run(x) for x in (a, b, c)],


probe job db7ri4mil99s73foo0f0 - waiting...
live median T1 197 us (calibration 243 us)
qubits with live T1 < half of calibration: 24
  e.g. [4, 33, 36, 40, 43, 47, 48, 50, 58, 64, 65, 68, 76, 82, 89]
X different placements | bad qubits in calib-placed: [36, 43, 47, 48] | in probe-placed: [50, 58, 89]
Z different placements | bad qubits in calib-placed: [43, 47, 48, 50, 58, 64, 65, 68] | in probe-placed: [33, 50, 58]

  X calib-placed  live-model eps 6.67%
  X probe-placed  live-model eps 4.44%
  Z calib-placed  live-model eps 5.88%
  Z probe-placed  live-model eps 3.58%

PRE-REGISTERED: P11 probe-placed beats calib-placed
in both bases (only the data source differs)


/tmp/ipykernel_707/4056523310.py:137: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  job2 = SamplerV2(mode=hw).run(pubs, shots=H21["shots"])


SUBMITTED job db7rid8r1c5c738l5mtg - 12 circuits


In [37]:
import json, glob, stim, pymatching, numpy as np
JOB22 = None
if JOB22 is None:
    JOB22 = H21["job_id"]
fn = "hh_hw18_" + JOB22 + ".json"
DRV = "/content/drive/MyDrive/qmem_hw/"
A22 = json.load(open((glob.glob(fn) + glob.glob(DRV + fn))[0]))
job = service.job(JOB22)
print("status:", job.status(), "- waiting for results...")
res22 = job.result()
G = {}
for k, arm in enumerate(A22["arms"]):
    bs = res22[k].data.c.get_bitstrings()
    raw = np.frombuffer("".join(bs).encode(), np.uint8)
    arr = (raw.reshape(len(bs), -1) - 48)[:, ::-1].astype(bool)
    det = np.stack([np.logical_xor.reduce(arr[:, d], axis=1)
                    for d in arm["dets"]], axis=1)
    det ^= np.array(arm["ref_d"], bool)[None, :]
    ob = np.logical_xor.reduce(arr[:, arm["obs"]], axis=1)
    ob ^= bool(arm["ref_o"])
    dem = stim.Circuit(arm["stim"]).detector_error_model(
        decompose_errors=True,
        ignore_decomposition_failures=True,
        approximate_disjoint_errors=True)
    m1 = pymatching.Matching.from_detector_error_model(dem)
    p1 = m1.decode_batch(det)[:, 0].astype(bool)
    m2 = pymatching.Matching.from_detector_error_model(
        clean_dem(dem), enable_correlations=True)
    p2 = m2.decode_batch(det, enable_correlations=True)
    p2 = p2[:, 0].astype(bool)
    G.setdefault((arm["basis"], arm["variant"]), []).append(
        (arm["rounds"], float(np.mean(p1 != ob)),
         float(np.mean(p2 != ob))))
N = A22["shots"]
E = {}
print()
for b in ("X", "Z"):
    for v in ("calib-placed", "probe-placed"):
        rows = sorted(G[(b, v)])
        rs = [r[0] for r in rows]
        a = slope_fit(rs, [r[1] for r in rows], N)
        c = slope_fit(rs, [r[2] for r in rows], N)
        E[(b, v)] = a
        print("  %s %-13s pL %s" % (b, v, " ".join(
            "%.1f%%" % (r[1] * 100) for r in rows)))
        print("      eps %.2f%% [%.2f, %.2f]  correlated %.2f%%"
              "  live model %.2f%%" % (a[0] * 100, a[1] * 100,
              a[2] * 100, c[0] * 100,
              A22["pred_" + b + "_" + v] * 100))
print()
ok = True
for b in ("X", "Z"):
    en, ln, hn = E[(b, "probe-placed")]
    eo, lo, ho = E[(b, "calib-placed")]
    ok &= en < eo
    print("  P11 %s probe vs calib x%.2f %s" % (
        b, eo / max(en, 1e-9),
        "separated" if hn < lo else "overlapping"))
print("  P11", "HOLDS" if ok else "FAILS")

status: DONE - waiting for results...

  X calib-placed  pL 4.3% 9.7% 14.2%
      eps 5.80% [5.18, 6.41]  correlated 5.75%  live model 6.67%
  X probe-placed  pL 4.7% 9.7% 13.6%
      eps 5.25% [4.62, 5.87]  correlated 5.39%  live model 4.44%
  Z calib-placed  pL 2.8% 8.0% 14.6%
      eps 6.33% [5.76, 6.89]  correlated 5.65%  live model 5.88%
  Z probe-placed  pL 5.0% 8.3% 10.7%
      eps 3.33% [2.74, 3.92]  correlated 3.01%  live model 3.58%

  P11 X probe vs calib x1.11 overlapping
  P11 Z probe vs calib x1.90 separated
  P11 HOLDS


In [38]:
import json, os, numpy as np
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import SamplerV2
hw = service.backend("ibm_kingston")
cal = ChipNoise(hw)
NQ = hw.num_qubits
W = int(20e-6 / hw.dt) // 32 * 32
pm0 = generate_preset_pass_manager(
    optimization_level=1, target=hw.target,
    initial_layout=list(range(NQ)))

def run_probe():
    a = QuantumCircuit(NQ, NQ)
    a.measure(range(NQ), range(NQ))
    b = QuantumCircuit(NQ, 2 * NQ)
    b.x(range(NQ))
    b.measure(range(NQ), range(NQ))
    b.delay(W, range(NQ), unit="dt")
    b.measure(range(NQ), range(NQ, 2 * NQ))
    c = QuantumCircuit(NQ, NQ)
    c.h(range(NQ))
    c.delay(W // 2, range(NQ), unit="dt")
    c.x(range(NQ))
    c.delay(W // 2, range(NQ), unit="dt")
    c.h(range(NQ))
    c.measure(range(NQ), range(NQ))
    j = SamplerV2(mode=hw).run([pm0.run(x) for x in (a, b, c)],
                               shots=3000)
    print("probe job", j.job_id(), "- waiting...")
    r = j.result()

    def bits(k):
        bs = r[k].data.c.get_bitstrings()
        raw = np.frombuffer("".join(bs).encode(), np.uint8)
        return (raw.reshape(len(bs), -1) - 48)[:, ::-1].astype(bool)
    A_, B_, C_ = bits(0), bits(1), bits(2)
    ro0, ro1 = A_.mean(0), 1 - B_[:, :NQ].mean(0)
    ech = C_.mean(0)
    tw, out = W * hw.dt, {}
    for q in range(NQ):
        sel = B_[:, q]
        dec = np.mean(~B_[sel, NQ + q]) if sel.any() else 0.5
        ro = 0.5 * (ro0[q] + ro1[q])
        d = min(max(dec - ro1[q], 1e-4), 0.99)
        t1 = -tw / np.log(1 - d)
        e = min(max(ech[q] - ro, 1e-4), 0.49)
        out[q] = (t1, min(-tw / np.log(1 - 2 * e), 2 * t1), ro)
    return out, j.job_id()

LIVE, probe_id = run_probe()
worse = [q for q in range(NQ) if LIVE[q][0] < cal.t1(q) / 2]
print("qubits with live T1 < half of calibration:", len(worse))
live = LiveNoise(hw)            # reads the new LIVE dict

def place2(basis, nz, kw):
    qc, dets, fl, obs, S, bad = make_variant(3, 3, basis, **kw)
    cg, uses = code_graph(qc)
    meas, rst = role_counts(qc)
    pool = candidate_layouts(hw, cg, np.random.default_rng(7))

    def score(L):
        s = model_score(L, uses, meas, rst, nz, S["n"], 3 * 5e-6)
        for i, p in enumerate(L):
            s += nz.sx(p) * (40 if i < S["n"] else 20)
        return s
    best = None
    for L in sorted(pool, key=score)[:TOP_K]:
        try:
            routed = route(qc, L)
        except Exception:
            continue
        if routed.count_ops().get("cz", 0) != qc.count_ops()["cx"]:
            continue
        isa, _ = add_dd(sched(routed), "xx")
        ff, _ = check_dets(isa, nz, fl, obs)
        alld = dets + [f for f, b_ in zip(fl, ff) if not b_]
        circ = isa_to_stim3(isa, nz, alld, obs, "mix")
        p = logical_error(circ, 20000, seed=11)
        if best is None or p < best[0]:
            best = (p, L)
    return best[1]

noise = live
ARMS = {}
for basis in ("X", "Z"):
    Lc = place2(basis, cal, {})
    Lp = place2(basis, live, COMPRESS[basis])
    ARMS[basis] = {"baseline": ({}, Lc),
                   "compressed": (COMPRESS[basis], Lc),
                   "compressed+probe": (COMPRESS[basis], Lp)}
    print(basis, "| drifted qubits used: calib-placed",
          len(set(Lc) & set(worse)), "| probe-placed",
          len(set(Lp) & set(worse)))

ORDERS = [("baseline", "compressed", "compressed+probe"),
          ("compressed+probe", "baseline", "compressed"),
          ("compressed", "compressed+probe", "baseline"),
          ("compressed+probe", "compressed", "baseline")]
H23 = {"backend": hw.name, "shots": 5000, "arms": [],
       "probe_job": probe_id}
pubs, preds = [], {}
for basis in ("X", "Z"):
    for i, R in enumerate((1, 2, 3, 4)):
        for v in ORDERS[i]:
            kw, L = ARMS[basis][v]
            isa, circ, alld, obs, ref_d, ref_o = build_isa(
                basis, R, kw, L)
            preds.setdefault((basis, v), {})[R] = \
                logical_error(circ, 30000, seed=9)
            H23["arms"].append({
                "basis": basis, "variant": v, "rounds": R,
                "dets": alld, "obs": obs[0], "ref_d": ref_d,
                "ref_o": ref_o, "stim": str(circ),
                "layout": [int(x) for x in L],
                "us": duration_s(isa, hw) * 1e6})
            pubs.append(isa)
print()
for (basis, v), d_ in sorted(preds.items()):
    e = slope_fit((1, 2, 3, 4), [d_[r] for r in (1, 2, 3, 4)])[0]
    H23["pred_" + basis + "_" + v] = e
    print("  %s %-17s live-model eps %.2f%%" % (basis, v, e * 100))
print()
print("PRE-REGISTERED before the job runs:")
print("  P12 compressed beats baseline in both bases")
print("  P13 compressed+probe beats compressed in Z by x1.3+")
print("      and is not worse in X")
print("  P14 best Z eps <= 4.5% per round")
job2 = SamplerV2(mode=hw).run(pubs, shots=H23["shots"])
H23["job_id"] = job2.job_id()
fname = "hh_hw18_" + H23["job_id"] + ".json"
json.dump(H23, open(fname, "w"))
dr = "/content/drive/MyDrive"
if os.path.isdir(dr):
    os.makedirs(dr + "/qmem_hw", exist_ok=True)
    json.dump(H23, open(dr + "/qmem_hw/" + fname, "w"))
print("SUBMITTED job", H23["job_id"], "-", len(pubs), "circuits")

qiskit_runtime_service.backends:WARNING:2026-10-14 17:46:25,138: Using instance: open-instance, plan: open
/tmp/ipykernel_707/1968443167.py:27: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  j = SamplerV2(mode=hw).run([pm0.run(x) for x in (a, b, c)],


probe job db7rvsuil99s73fooj00 - waiting...
qubits with live T1 < half of calibration: 16
X | drifted qubits used: calib-placed 3 | probe-placed 4
Z | drifted qubits used: calib-placed 3 | probe-placed 4

  X baseline          live-model eps 8.91%
  X compressed        live-model eps 5.95%
  X compressed+probe  live-model eps 5.28%
  Z baseline          live-model eps 9.25%
  Z compressed        live-model eps 4.14%
  Z compressed+probe  live-model eps 4.88%

PRE-REGISTERED before the job runs:
  P12 compressed beats baseline in both bases
  P13 compressed+probe beats compressed in Z by x1.3+
      and is not worse in X
  P14 best Z eps <= 4.5% per round


/tmp/ipykernel_707/1968443167.py:128: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  job2 = SamplerV2(mode=hw).run(pubs, shots=H23["shots"])


SUBMITTED job db7s2daraffc73fb6320 - 24 circuits


In [39]:
import json, glob, stim, pymatching, numpy as np
JOB24 = None
if JOB24 is None:
    JOB24 = H23["job_id"]
fn = "hh_hw18_" + JOB24 + ".json"
DRV = "/content/drive/MyDrive/qmem_hw/"
A24 = json.load(open((glob.glob(fn) + glob.glob(DRV + fn))[0]))
job = service.job(JOB24)
print("status:", job.status(), "- waiting for results...")
res24 = job.result()
G = {}
for k, arm in enumerate(A24["arms"]):
    bs = res24[k].data.c.get_bitstrings()
    raw = np.frombuffer("".join(bs).encode(), np.uint8)
    arr = (raw.reshape(len(bs), -1) - 48)[:, ::-1].astype(bool)
    det = np.stack([np.logical_xor.reduce(arr[:, d], axis=1)
                    for d in arm["dets"]], axis=1)
    det ^= np.array(arm["ref_d"], bool)[None, :]
    ob = np.logical_xor.reduce(arr[:, arm["obs"]], axis=1)
    ob ^= bool(arm["ref_o"])
    dem = stim.Circuit(arm["stim"]).detector_error_model(
        decompose_errors=True,
        ignore_decomposition_failures=True,
        approximate_disjoint_errors=True)
    m1 = pymatching.Matching.from_detector_error_model(dem)
    p1 = m1.decode_batch(det)[:, 0].astype(bool)
    m2 = pymatching.Matching.from_detector_error_model(
        clean_dem(dem), enable_correlations=True)
    p2 = m2.decode_batch(det, enable_correlations=True)
    p2 = p2[:, 0].astype(bool)
    G.setdefault((arm["basis"], arm["variant"]), []).append(
        (arm["rounds"], float(np.mean(p1 != ob)),
         float(np.mean(p2 != ob))))
N = A24["shots"]
E, EC = {}, {}
VARS = ("baseline", "compressed", "compressed+probe")
print()
for b in ("X", "Z"):
    for v in VARS:
        rows = sorted(G[(b, v)])
        rs = [r[0] for r in rows]
        a = slope_fit(rs, [r[1] for r in rows], N)
        c = slope_fit(rs, [r[2] for r in rows], N)
        E[(b, v)], EC[(b, v)] = a, c
        print("  %s %-17s pL %s" % (b, v, " ".join(
            "%.1f%%" % (r[1] * 100) for r in rows)))
        print("      eps %.2f%% [%.2f, %.2f]  correlated %.2f%%"
              "  live model %.2f%%" % (a[0] * 100, a[1] * 100,
              a[2] * 100, c[0] * 100,
              A24["pred_" + b + "_" + v] * 100))

def cmp(b, new, old):
    en, ln, hn = E[(b, new)]
    eo, lo, ho = E[(b, old)]
    return eo / max(en, 1e-9), hn < lo

print()
print("VERDICTS (plain matching)")
p12 = p13 = True
for b in ("X", "Z"):
    g, s = cmp(b, "compressed", "baseline")
    p12 &= g > 1
    print("  %s compression        x%.2f %s" % (
        b, g, "separated" if s else "overlapping"))
    g2, s2 = cmp(b, "compressed+probe", "compressed")
    p13 &= (g2 >= 1.3) if b == "Z" else (g2 >= 1.0)
    print("  %s probe placement    x%.2f %s" % (
        b, g2, "separated" if s2 else "overlapping"))
    g3, s3 = cmp(b, "compressed+probe", "baseline")
    print("  %s full stack         x%.2f %s" % (
        b, g3, "separated" if s3 else "overlapping"))
bz = min(E[("Z", "compressed+probe")][0],
         E[("Z", "compressed")][0])
print("  P12", "HOLDS" if p12 else "FAILS",
      "| P13", "HOLDS" if p13 else "FAILS",
      "| P14 best Z %.2f%%" % (bz * 100),
      "HOLDS" if bz <= 0.045 else "FAILS")

status: DONE - waiting for results...

  X baseline          pL 4.3% 9.9% 16.7% 21.8%
      eps 7.23% [6.74, 7.70]  correlated 7.23%  live model 8.91%
  X compressed        pL 3.2% 8.3% 14.0% 18.1%
      eps 5.99% [5.57, 6.40]  correlated 5.96%  live model 5.95%
  X compressed+probe  pL 4.7% 9.1% 12.5% 17.6%
      eps 4.98% [4.54, 5.42]  correlated 4.83%  live model 5.28%
  Z baseline          pL 3.2% 9.0% 16.7% 22.0%
      eps 7.61% [7.15, 8.06]  correlated 7.03%  live model 9.25%
  Z compressed        pL 2.2% 4.6% 9.1% 12.4%
      eps 3.67% [3.34, 4.00]  correlated 3.28%  live model 4.14%
  Z compressed+probe  pL 2.3% 5.7% 10.2% 13.6%
      eps 4.23% [3.88, 4.57]  correlated 3.61%  live model 4.88%

VERDICTS (plain matching)
  X compression        x1.21 separated
  X probe placement    x1.20 separated
  X full stack         x1.45 separated
  Z compression        x2.07 separated
  Z probe placement    x0.87 overlapping
  Z full stack         x1.80 separated
  P12 HOLDS | P13 FAILS | P

In [40]:
import json, os, numpy as np
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import SamplerV2
hw = service.backend("ibm_kingston")
cal = ChipNoise(hw)
LIVE, probe_id = run_probe()
live = LiveNoise(hw)
noise = live

def candidates(basis, nz, kw, k=TOP_K):
    qc, dets, fl, obs, S, bad = make_variant(3, 3, basis, **kw)
    cg, uses = code_graph(qc)
    meas, rst = role_counts(qc)
    pool = candidate_layouts(hw, cg, np.random.default_rng(7))

    def score(L):
        s = model_score(L, uses, meas, rst, nz, S["n"], 3 * 5e-6)
        for i, p in enumerate(L):
            s += nz.sx(p) * (40 if i < S["n"] else 20)
        return s
    return sorted(pool, key=score)[:k]

def live_sim(basis, L):
    try:
        isa, circ, *_ = build_isa(basis, 3, COMPRESS[basis], L)
    except Exception:
        return 1.0
    return logical_error(circ, 20000, seed=11)

LAY = {}
for basis in ("X", "Z"):
    Lc = place2(basis, cal, COMPRESS[basis])        # calibration
    pool = {tuple(L) for L in
            candidates(basis, cal, COMPRESS[basis])
            + candidates(basis, live, COMPRESS[basis])
            + [Lc]}
    judged = sorted(pool, key=lambda L: live_sim(basis, list(L)))
    Lj = list(judged[0])
    LAY[basis] = {"calib-placed": Lc, "live-judged": Lj}
    print(basis, len(pool), "candidates judged |",
          "same as calibration" if Lj == Lc else "new placement")

# logical arms, rounds 1-4, rotating order
H25 = {"backend": hw.name, "shots": 5000, "arms": [],
       "phys": [], "probe_job": probe_id}
pubs, preds = [], {}
ORD = [("calib-placed", "live-judged"),
       ("live-judged", "calib-placed")]
for basis in ("X", "Z"):
    for i, R in enumerate((1, 2, 3, 4)):
        for v in ORD[i % 2]:
            L = LAY[basis][v]
            isa, circ, alld, obs, ref_d, ref_o = build_isa(
                basis, R, COMPRESS[basis], L)
            preds.setdefault((basis, v), {})[R] = \
                logical_error(circ, 30000, seed=9)
            H25["arms"].append({
                "basis": basis, "variant": v, "rounds": R,
                "dets": alld, "obs": obs[0], "ref_d": ref_d,
                "ref_o": ref_o, "stim": str(circ),
                "layout": [int(x) for x in L],
                "us": duration_s(isa, hw) * 1e6})
            pubs.append(isa)

# physical reference: best 3 qubits on the chip (live probe),
# idle with the same DD for the same durations
NQ = hw.num_qubits
bestZ = sorted(range(NQ), key=lambda q: -LIVE[q][0])[:3]
bestX = sorted(range(NQ), key=lambda q: -LIVE[q][1])[:3]
for basis, best in (("Z", bestZ), ("X", bestX)):
    for R in (1, 2, 3, 4):
        us = next(a["us"] for a in H25["arms"]
                  if a["basis"] == basis and a["rounds"] == R
                  and a["variant"] == "live-judged")
        T = int(us * 1e-6 / hw.dt) // 32 * 32
        qc = QuantumCircuit(NQ, len(best))
        for j, q in enumerate(best):
            if basis == "X":
                qc.h(q)
            else:
                qc.x(q)
            qc.delay(T, q, unit="dt")
            if basis == "X":
                qc.h(q)
            qc.measure(q, j)
        isa = pm0.run(qc)
        isa, _ = add_dd(sched(isa), "xx")
        H25["phys"].append({"basis": basis, "rounds": R,
                            "qubits": best, "us": us,
                            "k": len(pubs)})
        pubs.append(isa)

print()
for (basis, v), d_ in sorted(preds.items()):
    e = slope_fit((1, 2, 3, 4), [d_[r] for r in (1, 2, 3, 4)])[0]
    H25["pred_" + basis + "_" + v] = e
    print("  %s %-13s live-model eps %.2f%%" % (basis, v, e * 100))
print("  best physical qubits: Z (T1)", bestZ,
      [round(LIVE[q][0] * 1e6) for q in bestZ], "us | X (T2)",
      bestX, [round(LIVE[q][1] * 1e6) for q in bestX], "us")
print()
print("PRE-REGISTERED before the job runs:")
print("  P15 live-judged >= calib-placed in both bases")
print("  P16 break-even NOT reached: logical lifetime is")
print("      0.15-0.6x the best physical qubit's, both bases")
job2 = SamplerV2(mode=hw).run(pubs, shots=H25["shots"])
H25["job_id"] = job2.job_id()
fname = "hh_hw25_" + H25["job_id"] + ".json"
json.dump(H25, open(fname, "w"))
dr = "/content/drive/MyDrive"
if os.path.isdir(dr):
    os.makedirs(dr + "/qmem_hw", exist_ok=True)
    json.dump(H25, open(dr + "/qmem_hw/" + fname, "w"))
print("SUBMITTED job", H25["job_id"], "-", len(pubs), "circuits")

qiskit_runtime_service.backends:WARNING:2026-10-14 17:56:32,146: Using instance: open-instance, plan: open
/tmp/ipykernel_707/1968443167.py:27: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  j = SamplerV2(mode=hw).run([pm0.run(x) for x in (a, b, c)],


probe job db7s4kqraffc73fb68b0 - waiting...
X 12 candidates judged | new placement
Z 16 candidates judged | new placement

  X calib-placed  live-model eps 5.41%
  X live-judged   live-model eps 5.75%
  Z calib-placed  live-model eps 4.48%
  Z live-judged   live-model eps 3.99%
  best physical qubits: Z (T1) [146, 153, 127] [2696, 483, 483] us | X (T2) [146, 32, 140] [5392, 219, 208] us

PRE-REGISTERED before the job runs:
  P15 live-judged >= calib-placed in both bases
  P16 break-even NOT reached: logical lifetime is
      0.15-0.6x the best physical qubit's, both bases


/tmp/ipykernel_707/1153231768.py:106: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  job2 = SamplerV2(mode=hw).run(pubs, shots=H25["shots"])


SUBMITTED job db7s4t6r6oqs738874lg - 24 circuits


In [41]:
import json, glob, stim, pymatching, numpy as np
JOB26 = None
if JOB26 is None:
    JOB26 = H25["job_id"]
fn = "hh_hw25_" + JOB26 + ".json"
DRV = "/content/drive/MyDrive/qmem_hw/"
A26 = json.load(open((glob.glob(fn) + glob.glob(DRV + fn))[0]))
job = service.job(JOB26)
print("status:", job.status(), "- waiting for results...")
res26 = job.result()

def arr_of(k):
    bs = res26[k].data.c.get_bitstrings()
    raw = np.frombuffer("".join(bs).encode(), np.uint8)
    return (raw.reshape(len(bs), -1) - 48)[:, ::-1].astype(bool)

G = {}
for k, arm in enumerate(A26["arms"]):
    arr = arr_of(k)
    det = np.stack([np.logical_xor.reduce(arr[:, d], axis=1)
                    for d in arm["dets"]], axis=1)
    det ^= np.array(arm["ref_d"], bool)[None, :]
    ob = np.logical_xor.reduce(arr[:, arm["obs"]], axis=1)
    ob ^= bool(arm["ref_o"])
    dem = stim.Circuit(arm["stim"]).detector_error_model(
        decompose_errors=True,
        ignore_decomposition_failures=True,
        approximate_disjoint_errors=True)
    m2 = pymatching.Matching.from_detector_error_model(
        clean_dem(dem), enable_correlations=True)
    m1 = pymatching.Matching.from_detector_error_model(dem)
    p1 = m1.decode_batch(det)[:, 0].astype(bool)
    p2 = m2.decode_batch(det, enable_correlations=True)
    p2 = p2[:, 0].astype(bool)
    G.setdefault((arm["basis"], arm["variant"]), []).append(
        (arm["rounds"], float(np.mean(p1 != ob)),
         float(np.mean(p2 != ob)), arm["us"]))
N = A26["shots"]

def life(ts, ps):     # lifetime in us from F = A exp(-t/tau)
    e, lo, hi = slope_fit(ts, ps, N)
    f = lambda x: -1 / np.log(max(1 - 2 * x, 1e-9))
    return f(e), f(hi), f(lo)

print()
E, TAU = {}, {}
for b in ("X", "Z"):
    for v in ("calib-placed", "live-judged"):
        rows = sorted(G[(b, v)])
        rs = [r[0] for r in rows]
        E[(b, v)] = slope_fit(rs, [r[1] for r in rows], N)
        TAU[(b, v)] = life([r[3] for r in rows],
                           [r[2] for r in rows])
        e = E[(b, v)]
        print("  %s %-13s eps %.2f%% [%.2f, %.2f]  lifetime %.0f us"
              " [%.0f, %.0f] (correlated)" % (b, v, e[0] * 100,
              e[1] * 100, e[2] * 100, *TAU[(b, v)]))

print()
print("PHYSICAL QUBITS idle with DD, same durations:")
PH = {}
for b in ("X", "Z"):
    ph = [p for p in A26["phys"] if p["basis"] == b]
    for j, q in enumerate(ph[0]["qubits"]):
        ts, ps = [], []
        for p in sorted(ph, key=lambda p: p["rounds"]):
            bit = arr_of(p["k"])[:, j]
            err = bit.mean() if b == "X" else 1 - bit.mean()
            ts.append(p["us"])
            ps.append(float(err))
        PH[(b, q)] = life(ts, ps)
        txt = " ".join("%.1f%%" % (x * 100) for x in ps)
        print("  %s qubit %3d  error %s" % (b, q, txt))
        print("      lifetime %.0f us [%.0f, %.0f]" % PH[(b, q)])

print()
print("VERDICTS")
ok15 = all(E[(b, "live-judged")][0] <= E[(b, "calib-placed")][0]
           for b in ("X", "Z"))
print("  P15", "HOLDS" if ok15 else "FAILS")
ok16 = True
for b in ("X", "Z"):
    tl = max(TAU[(b, v)][0] for v in ("calib-placed",
                                      "live-judged"))
    tp = max(v[0] for k, v in PH.items() if k[0] == b)
    r = tl / tp
    ok16 &= 0.15 <= r <= 0.6
    print("  %s best logical %.0f us vs best physical %.0f us"
          " -> %.2fx of break-even" % (b, tl, tp, r))
print("  P16", "HOLDS" if ok16 else "FAILS",
      "(break-even = 1.00x)")

status: DONE - waiting for results...

  X calib-placed  eps 4.53% [4.12, 4.94]  lifetime 78 us [71, 86] (correlated)
  X live-judged   eps 5.70% [5.28, 6.11]  lifetime 60 us [55, 65] (correlated)
  Z calib-placed  eps 4.55% [4.18, 4.92]  lifetime 56 us [52, 61] (correlated)
  Z live-judged   eps 3.73% [3.42, 4.04]  lifetime 73 us [66, 80] (correlated)

PHYSICAL QUBITS idle with DD, same durations:
  X qubit 146  error 47.4% 47.6% 48.1% 47.9%
      lifetime 78 us [20, -39]
  X qubit  32  error 3.6% 4.0% 5.9% 6.1%
      lifetime 344 us [267, 483]
  X qubit 140  error 1.5% 1.9% 2.9% 3.9%
      lifetime 444 us [353, 598]
  Z qubit 146  error 53.2% 52.1% 52.7% 52.7%
      lifetime 1501199875790165 us [0, -0]
  Z qubit 153  error 4.1% 4.7% 4.9% 5.6%
      lifetime 479 us [304, 1134]
  Z qubit 127  error 6.3% 7.8% 7.4% 8.9%
      lifetime 284 us [196, 518]

VERDICTS
  P15 FAILS
  X best logical 78 us vs best physical 444 us -> 0.18x of break-even
  Z best logical 73 us vs best physical 15011

In [ ]:
# DAILY: multi-day replication of compression (one cell)
import sys, glob, shutil, os, importlib
from google.colab import drive, files
drive.mount("/content/drive")
FOLDER = "/content/drive/MyDrive/qmem_hw"
os.makedirs(FOLDER, exist_ok=True)
for fn in glob.glob("/content/hh_hw*.json"):
    shutil.copy(fn, FOLDER)                  # keep today's runs
if not os.path.exists(FOLDER + "/hhmem.py"):
    up = files.upload()                      # pick hhmem.py
    for name in up:
        shutil.move(name, FOLDER + "/hhmem.py")
sys.path.insert(0, FOLDER)
import hhmem
importlib.reload(hhmem)
from qiskit_ibm_runtime import QiskitRuntimeService
from google.colab import userdata  # IBM key comes from Colab Secrets, never from the notebook
API_KEY = userdata.get("IBM_QUANTUM_KEY")
service = QiskitRuntimeService(
    channel="ibm_quantum_platform", token=API_KEY)

SUBMIT = False            # today False; from tomorrow True
hhmem.update_log(service, FOLDER)
if SUBMIT:
    hhmem.submit_daily(service, "ibm_kingston", FOLDER)

Mounted at /content/drive


In [ ]:
"""hhmem: reset-free heavy-hex memory experiments on IBM Heron.

Daily use (Colab):
    import hhmem
    hhmem.update_log(service, FOLDER)            # decode finished runs
    hhmem.submit_daily(service, "ibm_kingston", FOLDER)

Everything here was validated in the 14 Oct 2026 notebook (H1-H24).
"""
import glob, json, math, os, time
from collections import Counter
import numpy as np
import pymatching
import rustworkx as rx
import stim
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import ALAPScheduleAnalysis, PadDelay
from qiskit.transpiler.preset_passmanagers import \
    generate_preset_pass_manager

COMPRESS = {"Z": dict(merge=True, skip_safe=True, skip_flags=True),
            "X": dict(merge=True, skip_safe=True)}


# ------------------------------------------------------------------
# noise model from calibration (T1, echo T2, gate/readout errors)
# ------------------------------------------------------------------
class ChipNoise:
    def __init__(self, backend):
        self.b, t = backend, backend.target
        self.dt = backend.dt
        self.qp = t.qubit_properties or [None] * t.num_qubits
        self.t_ = t

    def _err(self, name, qargs):
        if name not in self.t_.operation_names:
            return None
        p = self.t_[name].get(qargs)
        return p.error if p is not None else None

    def t1(self, q):
        p = self.qp[q]
        return p.t1 if p is not None and p.t1 else 100e-6

    def t2(self, q):
        p = self.qp[q]
        t2 = p.t2 if p is not None and p.t2 else 100e-6
        return min(t2, 2 * self.t1(q))

    def ro(self, q):
        e = self._err("measure", (q,))
        return min(e, 0.45) if e is not None else 0.03

    def sx(self, q):
        e = self._err("sx", (q,))
        return min(e, 0.1) if e is not None else 3e-4

    def cz(self, a, b):
        e = self._err("cz", (a, b))
        if e is None:
            e = self._err("cz", (b, a))
        return min(e, 0.5) if e is not None else 5e-3

    def idle(self, q, dur):
        a = 1 - math.exp(-dur / self.t1(q))
        b = 1 - math.exp(-dur / self.t2(q))
        return a / 4, a / 4, max(b / 2 - a / 4, 0.0)


RZ = {0: None, 1: "S", 2: "Z", 3: "S_DAG"}


def isa_to_stim(isa, noise, dets, obs, noisy=True, ro_mode="mix"):
    """Compiled circuit -> Stim. Readout error: 'flip' (qubit flips),
    'record' (only the record), 'mix' (half each)."""
    out, pos, nmeas, active = stim.Circuit(), {}, 0, set()
    for inst in isa.data:
        op, name = inst.operation, inst.operation.name
        qs = [isa.find_bit(q).index for q in inst.qubits]
        if name in ("barrier", "id"):
            continue
        if name == "delay":
            if noisy and qs[0] in active:
                val = float(op.params[0])
                u = getattr(op, "unit", "dt")
                dur = val * noise.dt if u == "dt" else val
                if dur > 0:
                    out.append("PAULI_CHANNEL_1", qs,
                               list(noise.idle(qs[0], dur)))
            continue
        if name == "rz":
            ang = float(op.params[0]) / (math.pi / 2)
            k = round(ang)
            if abs(ang - k) > 1e-6:
                raise ValueError("non-Clifford rz")
            if RZ[k % 4]:
                out.append(RZ[k % 4], qs)
            active.update(qs)
        elif name in ("sx", "sxdg", "x"):
            g = {"sx": "SQRT_X", "sxdg": "SQRT_X_DAG", "x": "X"}[name]
            out.append(g, qs)
            if noisy:
                out.append("DEPOLARIZE1", qs, noise.sx(qs[0]))
            active.update(qs)
        elif name == "cz":
            out.append("CZ", qs)
            if noisy:
                out.append("DEPOLARIZE2", qs, noise.cz(*qs))
            active.update(qs)
        elif name == "measure":
            p = noise.ro(qs[0]) if noisy else 0.0
            if ro_mode == "flip":
                if p:
                    out.append("X_ERROR", qs, p)
                out.append("M", qs)
            elif ro_mode == "record":
                out.append("M", qs, p)
            else:
                if p:
                    out.append("X_ERROR", qs, p / 2)
                out.append("M", qs, p / 2)
            pos[isa.find_bit(inst.clbits[0]).index] = nmeas
            nmeas += 1
            active.update(qs)
        elif name == "reset":
            out.append("R", qs)
            if noisy:
                out.append("X_ERROR", qs, noise.ro(qs[0]))
            active.difference_update(qs)
        else:
            raise ValueError("unsupported gate " + name)
    for det in dets:
        out.append("DETECTOR",
                   [stim.target_rec(pos[c] - nmeas) for c in det])
    for k, lst in obs.items():
        out.append("OBSERVABLE_INCLUDE",
                   [stim.target_rec(pos[c] - nmeas) for c in lst], k)
    return out


def dem_of(circ):
    return circ.detector_error_model(
        decompose_errors=True, ignore_decomposition_failures=True,
        approximate_disjoint_errors=True)


def clean_dem(dem):
    """Drop the few undecomposable terms (for correlated matching)."""
    out = stim.DetectorErrorModel()
    for ins in dem.flattened():
        if ins.type == "error":
            comps, cur = [], 0
            for t in ins.targets_copy():
                if t.is_separator():
                    comps.append(cur)
                    cur = 0
                elif t.is_relative_detector_id():
                    cur += 1
            comps.append(cur)
            if max(comps) > 2:
                continue
        out.append(ins)
    out += stim.DetectorErrorModel(
        "detector D%d" % (dem.num_detectors - 1))
    return out


def logical_error(circ, shots, seed=1):
    try:
        m = pymatching.Matching.from_detector_error_model(dem_of(circ))
        det, ob = circ.compile_detector_sampler(seed=seed).sample(
            shots, separate_observables=True)
        return float(np.mean(np.any(m.decode_batch(det) != ob, axis=1)))
    except ValueError:
        return float("nan")


def slope_fit(rs, ps, N=None):
    """F(r) = 1 - 2 pL(r) = A (1 - 2 eps)^r -> (eps, lo, hi)."""
    rs = np.array(rs, float)
    ps = np.array(ps, float)
    F = np.clip(1 - 2 * ps, 1e-3, None)
    y = np.log(F)
    if N:
        sp = np.sqrt(np.clip(ps * (1 - ps), 1e-9, None) / N)
        w = (F / (2 * sp)) ** 2
    else:
        w = np.ones_like(y)
    X = np.vstack([np.ones_like(rs), rs]).T
    cov = np.linalg.inv(X.T @ (w[:, None] * X))
    beta = cov @ X.T @ (w * y)
    se = np.sqrt(cov[1, 1]) if N else 0.0
    f = lambda s: (1 - np.exp(s)) / 2
    return f(beta[1]), f(beta[1] + 2 * se), f(beta[1] - 2 * se)


# ------------------------------------------------------------------
# IBM heavy-hex subsystem code (qiskit-qec construction)
# ------------------------------------------------------------------
def hhc_structure(d):
    idx = lambda r, c: r * d + c
    xg = [[idx(i, j), idx(i + 1, j)]
          for i in range(d - 1) for j in range(d)]
    zg, zcol = [], []
    for i in range(d + 1):
        for j in range(d - 1):
            if i % 2 == j % 2:
                if i == 0:
                    zg.append([idx(0, j), idx(0, j + 1)])
                elif i == d:
                    zg.append([idx(d - 1, j), idx(d - 1, j + 1)])
                else:
                    zg.append([idx(i - 1, j), idx(i - 1, j + 1),
                               idx(i, j), idx(i, j + 1)])
                zcol.append(j)
    xstab = []
    for i in range(d - 1):
        for j in range(d):
            g = i * d + j
            if i % 2 == j % 2:
                xstab.append([g] if j == d - 1 else [g, g + 1])
            elif j == 0:
                xstab.append([g])
    n = d * d
    xanc = [n + k for k in range(len(xg))]
    total, zanc = len(xg), []
    for z in zg:
        if len(z) == 2:
            zanc.append([n + total, n + total + 1, n + total + 2])
            total += 3
        else:
            left = next(n + k for k, x in enumerate(xg)
                        if z[0] in x and z[2] in x)
            right = next(n + k for k, x in enumerate(xg)
                         if z[1] in x and z[3] in x)
            zanc.append([left, n + total, right])
            total += 1
    return {"d": d, "n": n, "xg": xg, "zg": zg, "zcol": zcol,
            "xstab": xstab, "xanc": xanc, "zanc": zanc,
            "nq": n + total}


def mod2(lst):
    return sorted(b for b, k in Counter(lst).items() if k % 2)


def build_hhc(d, rounds, basis, merge=False, skip_safe=False,
              skip_flags=False):
    S = hhc_structure(d)
    n, xg, zg = S["n"], S["xg"], S["zg"]
    sched_ = ("xz" if basis == "Z" else "zx") * rounds
    nx, nz = len(xg), len(zg)
    ncl = sum(nx if r == "x" else 3 * nz for r in sched_) + n
    qc = QuantumCircuit(QuantumRegister(S["nq"], "q"),
                        ClassicalRegister(ncl, "c"))
    cb = iter(range(ncl))
    last, pend, prev = {}, {}, {}

    def RESET(q, skip):
        if skip:
            if q in last:
                pend[q] = last[q]
        else:
            qc.reset(q)
            pend.pop(q, None)

    def MEAS(q):
        b = next(cb)
        qc.measure(q, b)
        if q in pend:
            prev[b] = pend.pop(q)
        last[q] = b
        return b

    for i in range(n):
        qc.reset(i)
    if basis == "X":
        for i in range(n):
            qc.h(i)
    log = []
    for r in sched_:
        if r == "x":
            for a in S["xanc"]:
                RESET(a, skip_safe)
                qc.h(a)
            for step in range(2):
                for k, g in enumerate(xg):
                    qc.cx(S["xanc"][k], g[step])
            bits = []
            for a in S["xanc"]:
                qc.h(a)
                bits.append(MEAS(a))
            log.append(("x", bits))
            continue
        gb, lf, rf = [None] * nz, [None] * nz, [None] * nz
        for step in range(7):
            for j, z in enumerate(zg):
                l, m, rr = S["zanc"][j]
                bnd, top = len(z) == 2, z[0] < d
                if step == 0:
                    RESET(m, skip_safe)
                    RESET(rr, skip_flags)
                    qc.h(rr)
                elif step == 1:
                    qc.cx(rr, m)
                    RESET(l, skip_flags)
                    qc.h(l)
                elif step == 2:
                    qc.cx(l, m)
                    if not (bnd and top):
                        qc.cx(z[1], rr)
                elif step == 3:
                    if bnd and top:
                        qc.cx(z[0], l)
                        qc.cx(z[1], rr)
                    elif not bnd:
                        qc.cx(z[2], l)
                        qc.cx(z[3], rr)
                elif step == 4:
                    qc.cx(rr, m)
                    if not (bnd and top):
                        qc.cx(z[0], l)
                elif step == 5:
                    qc.cx(l, m)
                    qc.h(rr)
                    if not merge:
                        rf[j] = MEAS(rr)
                elif step == 6:
                    qc.h(l)
                    if merge:
                        rf[j] = MEAS(rr)
                    gb[j] = MEAS(m)
                    lf[j] = MEAS(l)
        log.append(("z", gb, lf, rf))
    if basis == "X":
        for i in range(n):
            qc.h(i)
    data_bits = [MEAS(i) for i in range(n)]

    eff = lambda b: [b] + ([prev[b]] if b in prev else [])
    E = lambda bits: [x for b in bits for x in eff(b)]
    dets, flag_dets = [], []
    zr = [e for e in log if e[0] == "z"]
    xr = [e for e in log if e[0] == "x"]
    zs = lambda e, j: E([e[1][g] for g in range(nz)
                         if S["zcol"][g] == j])
    xs = lambda e, s: E([e[1][g] for g in S["xstab"][s]])
    for j in range(d - 1):
        for t, e in enumerate(zr):
            if t == 0:
                if basis == "Z":
                    dets.append(zs(e, j))
            else:
                dets.append(zs(e, j) + zs(zr[t - 1], j))
        if basis == "Z":
            cols = [data_bits[r_ * d + j] for r_ in range(d)]
            cols += [data_bits[r_ * d + j + 1] for r_ in range(d)]
            dets.append(cols + zs(zr[-1], j))
    for s, gs in enumerate(S["xstab"]):
        for t, e in enumerate(xr):
            if t == 0:
                if basis == "X":
                    dets.append(xs(e, s))
            else:
                dets.append(xs(e, s) + xs(xr[t - 1], s))
        if basis == "X":
            sup = sorted({q_ for g in gs for q_ in xg[g]})
            dets.append([data_bits[q_] for q_ in sup] + xs(xr[-1], s))
    for e in zr:
        flag_dets += [E([b]) for b in e[2] + e[3]]
    Q = sorted(set(prev.values()))
    return qc, dets, flag_dets, list(data_bits), S, Q


def clean_samples(qc, shots, seed=0):
    c, order = stim.Circuit(), []
    for inst in qc.data:
        nm = inst.operation.name
        qs = [qc.find_bit(q).index for q in inst.qubits]
        if nm == "h":
            c.append("H", qs)
        elif nm == "cx":
            c.append("CX", qs)
        elif nm == "reset":
            c.append("R", qs)
        elif nm == "measure":
            c.append("M", qs)
            order.append(qc.find_bit(inst.clbits[0]).index)
    raw = c.compile_sampler(seed=seed).sample(shots)
    B = np.zeros((shots, qc.num_clbits), np.uint8)
    B[:, order] = raw
    return B


def gf2_solve(A, y):
    M = np.concatenate([A, y[:, None]], 1).astype(np.uint8) & 1
    nr, nc = A.shape
    piv, r = [], 0
    for c in range(nc):
        rows = np.nonzero(M[r:, c])[0]
        if len(rows) == 0:
            continue
        p = r + rows[0]
        M[[r, p]] = M[[p, r]]
        oth = np.nonzero(M[:, c])[0]
        oth = oth[oth != r]
        M[oth] ^= M[r]
        piv.append(c)
        r += 1
        if r == nr:
            break
    if M[r:, -1].any():
        return None
    x = np.zeros(nc, np.uint8)
    for i, c in enumerate(piv):
        x[c] = M[i, -1]
    return x


def repair(lst, B, Q):
    """Make a detector deterministic after resets were removed."""
    lst = mod2(lst)
    y = B[:, lst].sum(1) % 2
    if y.min() == y.max():
        return lst
    if not Q:
        return None
    A = np.concatenate([B[:, Q], np.ones((len(B), 1), np.uint8)], 1)
    x = gf2_solve(A, y.astype(np.uint8))
    if x is None:
        return None
    return mod2(lst + [Q[i] for i in range(len(Q)) if x[i]])


def make_variant(d, rounds, basis, **kw):
    qc, dets, fl, obs, S, Q = build_hhc(d, rounds, basis, **kw)
    B = clean_samples(qc, 3 * qc.num_clbits + 200)
    dets2 = [repair(x, B, Q) for x in dets]
    bad = sum(x is None for x in dets2)
    fl2 = [x for x in (repair(f, B, Q) for f in fl) if x is not None]
    ob2 = repair(obs, B, Q)
    assert ob2 is not None, "observable could not be repaired"
    return qc, [x for x in dets2 if x is not None], fl2, \
        {0: ob2}, S, bad


# ------------------------------------------------------------------
# placement candidates (zero-SWAP VF2 matches) and scoring
# ------------------------------------------------------------------
def code_graph(qc):
    uses = {}
    for inst in qc.data:
        if inst.operation.name == "cx":
            a, b = sorted(qc.find_bit(q).index for q in inst.qubits)
            uses[(a, b)] = uses.get((a, b), 0) + 1
    g = rx.PyGraph()
    g.add_nodes_from(range(qc.num_qubits))
    g.add_edges_from_no_data(list(uses))
    return g, uses


def role_counts(qc):
    meas, rst = np.zeros(qc.num_qubits), np.zeros(qc.num_qubits)
    for inst in qc.data:
        i = qc.find_bit(inst.qubits[0]).index
        if inst.operation.name == "measure":
            meas[i] += 1
        elif inst.operation.name == "reset":
            rst[i] += 1
    return meas, rst


def candidate_layouts(backend, cg, rng, n_shuffle=30, per=40):
    nq = backend.num_qubits
    edges = {tuple(sorted(e)) for e in backend.coupling_map.get_edges()}
    seen, out = set(), []
    for _ in range(n_shuffle):
        perm = rng.permutation(nq)
        inv = {int(p): i for i, p in enumerate(perm)}
        chipg = rx.PyGraph()
        chipg.add_nodes_from(range(nq))
        chipg.add_edges_from_no_data([(inv[a], inv[b])
                                      for a, b in edges])
        it = rx.vf2_mapping(chipg, cg, subgraph=True, induced=False,
                            id_order=False, call_limit=5_000_000)
        for k, m in enumerate(it):
            if k >= per:
                break
            lay = [0] * cg.num_nodes()
            for cn, vn in m.items():
                lay[vn] = int(perm[cn])
            t = tuple(lay)
            if t not in seen:
                seen.add(t)
                out.append(list(t))
    return out


def model_score(L, uses, meas, rst, nz, n_data, T):
    s = sum(k * nz.cz(L[a], L[b]) for (a, b), k in uses.items())
    for i, p in enumerate(L):
        s += (meas[i] + rst[i]) * nz.ro(p)
        s += sum(nz.idle(p, T)) * (1.0 if i < n_data else 0.5)
        s += nz.sx(p) * (40 if i < n_data else 20)
    return s


# ------------------------------------------------------------------
# one chip: routing, scheduling, dynamical decoupling, building
# ------------------------------------------------------------------
class Chip:
    SPACING = {2: [0.25, 0.5, 0.25]}

    def __init__(self, backend, noise=None):
        self.hw = backend
        self.noise = noise or ChipNoise(backend)
        t = backend.target
        self.align = max(getattr(t, "granularity", 16) or 16,
                         getattr(t, "pulse_alignment", 16) or 16)
        self.xd = {}

    def xdur(self, q):
        if q not in self.xd:
            d_ = self.hw.target["x"][(q,)].duration
            self.xd[q] = int(round(d_ / self.hw.dt))
        return self.xd[q]

    def route(self, qc, lay=None):
        pm = generate_preset_pass_manager(
            optimization_level=1, target=self.hw.target,
            initial_layout=lay, seed_transpiler=1)
        return pm.run(qc)

    def sched(self, routed):
        pm = PassManager([ALAPScheduleAnalysis(target=self.hw.target),
                          PadDelay(target=self.hw.target)])
        return pm.run(routed)

    def add_dd(self, sc):
        """X-X dynamical decoupling in every idle gap of an active qubit."""
        A = self.align
        out = sc.copy_empty_like()
        fresh = set(range(sc.num_qubits))
        npulse = 0
        for inst in sc.data:
            op = inst.operation
            qs = [sc.find_bit(q).index for q in inst.qubits]
            unit = getattr(op, "unit", "dt")
            if (op.name == "delay" and qs[0] not in fresh
                    and unit == "dt"):
                q, T = qs[0], int(op.params[0])
                xd = self.xdur(q)
                F = T - 2 * xd
                if T % A == 0 and xd % A == 0 and F >= 4 * A:
                    p1 = int(F * 0.25) // A * A
                    p2 = int(F * 0.5) // A * A
                    p3 = F - p1 - p2
                    qb = inst.qubits[0]
                    if p1:
                        out.delay(p1, qb, unit="dt")
                    out.x(qb)
                    if p2:
                        out.delay(p2, qb, unit="dt")
                    out.x(qb)
                    if p3:
                        out.delay(p3, qb, unit="dt")
                    npulse += 2
                    continue
            out.append(inst)
            if op.name == "reset":
                fresh.update(qs)
            elif op.name not in ("delay", "barrier", "measure"):
                fresh.difference_update(qs)
        return out, npulse

    def check(self, isa, dets, obs):
        c = isa_to_stim(isa, self.noise, dets, obs, noisy=False)
        det, ob = c.compile_detector_sampler(seed=3).sample(
            400, separate_observables=True)
        return det.any(axis=0), ob.any()

    def duration_us(self, isa):
        try:
            return isa.estimate_duration(self.hw.target, unit="s") * 1e6
        except Exception:
            return float("nan")

    def build(self, basis, R, kw, L):
        qc, dets, fl, obs, S, bad = make_variant(3, R, basis, **kw)
        assert bad == 0, "detectors lost"
        routed = self.route(qc, L)
        ncz = routed.count_ops().get("cz", 0)
        assert ncz == qc.count_ops()["cx"], "SWAPs appeared"
        isa, npul = self.add_dd(self.sched(routed))
        assert npul > 0, "no DD pulses"
        fired, obf = self.check(isa, dets, obs)
        assert not fired.any() and not obf, "noiseless check failed"
        ff, _ = self.check(isa, fl, obs)
        alld = dets + [f for f, b_ in zip(fl, ff) if not b_]
        B = clean_samples(qc, 8)
        circ = isa_to_stim(isa, self.noise, alld, obs)
        arm = {"basis": basis, "rounds": R, "dets": alld,
               "obs": obs[0], "stim": str(circ),
               "ref_d": [int(B[0, x].sum() % 2) for x in alld],
               "ref_o": int(B[0, obs[0]].sum() % 2),
               "layout": [int(x) for x in L],
               "us": self.duration_us(isa)}
        return isa, circ, arm

    def place(self, basis, kw, top=8, seed=7):
        """Calibration tournament: score nominates, simulation picks."""
        qc, dets, fl, obs, S, bad = make_variant(3, 3, basis, **kw)
        cg, uses = code_graph(qc)
        meas, rst = role_counts(qc)
        pool = candidate_layouts(self.hw, cg,
                                 np.random.default_rng(seed))
        sc = lambda L: model_score(L, uses, meas, rst, self.noise,
                                   S["n"], 3 * 5e-6)
        best = None
        for L in sorted(pool, key=sc)[:top]:
            try:
                _, circ, _ = self.build(basis, 3, kw, L)
            except Exception:
                continue
            p = logical_error(circ, 20000, seed=11)
            if best is None or p < best[0]:
                best = (p, L)
        return best[1]


# ------------------------------------------------------------------
# daily replication: baseline vs compressed, same placement
# ------------------------------------------------------------------
def submit_daily(service, backend_name, folder, shots=5000,
                 rounds=(1, 2, 3)):
    from qiskit_ibm_runtime import SamplerV2
    hw = service.backend(backend_name)
    chip = Chip(hw)
    print(hw.name, hw.status().pending_jobs, "jobs in queue")
    rec = {"backend": hw.name, "shots": shots, "arms": [],
           "date": time.strftime("%Y-%m-%d"), "kind": "daily"}
    pubs = []
    day = int(time.strftime("%j"))
    for basis in ("X", "Z") if day % 2 else ("Z", "X"):
        L = chip.place(basis, {})
        for i, R in enumerate(rounds):
            order = ("baseline", "compressed")
            if (i + day) % 2:
                order = order[::-1]
            for v in order:
                kw = COMPRESS[basis] if v == "compressed" else {}
                isa, circ, arm = chip.build(basis, R, kw, L)
                arm["variant"] = v
                arm["pred_pL"] = logical_error(circ, 20000, seed=9)
                rec["arms"].append(arm)
                pubs.append(isa)
    job = SamplerV2(mode=hw).run(pubs, shots=shots)
    rec["job_id"] = job.job_id()
    os.makedirs(folder, exist_ok=True)
    fn = os.path.join(folder, "daily_%s_%s.json"
                      % (rec["date"], rec["job_id"]))
    json.dump(rec, open(fn, "w"))
    print("SUBMITTED", rec["job_id"], "-", len(pubs), "circuits ->", fn)
    return rec["job_id"]


def decode_record(service, rec):
    res = service.job(rec["job_id"]).result()
    G = {}
    for k, arm in enumerate(rec["arms"]):
        bs = res[k].data.c.get_bitstrings()
        raw = np.frombuffer("".join(bs).encode(), np.uint8)
        arr = (raw.reshape(len(bs), -1) - 48)[:, ::-1].astype(bool)
        det = np.stack([np.logical_xor.reduce(arr[:, d], axis=1)
                        for d in arm["dets"]], axis=1)
        det ^= np.array(arm["ref_d"], bool)[None, :]
        ob = np.logical_xor.reduce(arr[:, arm["obs"]], axis=1)
        ob ^= bool(arm["ref_o"])
        m = pymatching.Matching.from_detector_error_model(
            dem_of(stim.Circuit(arm["stim"])))
        pred = m.decode_batch(det)[:, 0].astype(bool)
        G.setdefault((arm["basis"], arm["variant"]), []).append(
            (arm["rounds"], float(np.mean(pred != ob))))
    out = {}
    for (b, v), rows in G.items():
        rows.sort()
        out[b + "_" + v] = slope_fit([r[0] for r in rows],
                                     [r[1] for r in rows],
                                     rec["shots"])
    return out


def update_log(service, folder):
    """Decode every finished run (daily files and the original
    hh_hw11_/hh_hw18_ runs) and print the multi-day table."""
    files, seen = [], set()
    for d_ in (folder, os.getcwd()):
        for pat in ("daily_*.json", "hh_hw11_*.json", "hh_hw18_*.json"):
            for fn in sorted(glob.glob(os.path.join(d_, pat))):
                if os.path.basename(fn) not in seen:
                    seen.add(os.path.basename(fn))
                    files.append(fn)
    rows = []
    for fn in files:
        rec = json.load(open(fn))
        vs = {a.get("variant") for a in rec.get("arms", [])}
        if not {"baseline", "compressed"} <= vs:
            continue
        if "result" not in rec:
            try:
                st = str(service.job(rec["job_id"]).status())
            except Exception as e:
                print("  skip", rec.get("job_id"), e)
                continue
            if "DONE" not in st.upper():
                print("  pending:", rec["job_id"], st)
                continue
            rec["result"] = {k: list(v) for k, v in
                             decode_record(service, rec).items()}
            json.dump(rec, open(fn, "w"))
        r = rec["result"]
        date = rec.get("date") or time.strftime(
            "%Y-%m-%d", time.localtime(os.path.getmtime(fn)))
        for b in ("Z", "X"):
            eb, lb, hb = r[b + "_baseline"]
            ec, lc, hc = r[b + "_compressed"]
            rows.append((date, rec["backend"], b, eb, ec, eb / ec,
                         hc < lb, rec["job_id"]))
    rows.sort()
    print()
    print("date        chip            basis  baseline  compressed"
          "  gain   2-sigma")
    for d_, bk, b, eb, ec, g, sep, j in rows:
        print("%s  %-14s  %s     %5.2f%%    %5.2f%%     x%.2f  %s"
              % (d_, bk, b, eb * 100, ec * 100, g,
                 "yes" if sep else "no"))
    for b in ("Z", "X"):
        gs = [r[5] for r in rows if r[2] == b]
        if gs:
            print("%s: %d runs, compressed wins %d, median gain x%.2f,"
                  " separated %d" % (b, len(gs),
                  sum(g > 1 for g in gs), float(np.median(gs)),
                  sum(r[6] for r in rows if r[2] == b)))
    return rows